# Markets Dashboard — Python / Jupyter Edition

Live market data via **yfinance** + **FRED**, wrapped in an interactive **ipywidgets** UI: horizon toggle (Short/Medium/Long), 8 category tabs, a sector drill-down that ranks a real candidate pool by *actual* return per horizon, and a **Chart** tab to plot real price/yield history (1 month through 10 years) for any tracked ticker, including a **Compare** mode (added 2026-09-16) to overlay 2-5 tickers at once, each normalized to % change from the start of the selected period so a stock, an index, and a bond yield can all sit on one axis.

Because this notebook runs on **your machine**, not a sandboxed environment, `yfinance`/FRED reach the internet normally — no connector, no per-symbol gating, no daily rate cap, no manual "please refresh" requests.

## One-time setup
```bash
pip install yfinance pandas matplotlib plotly ipywidgets voila pandas-datareader requests
```
(If you're on classic Jupyter Notebook rather than JupyterLab/VS Code, also run `jupyter nbextension enable --py widgetsnbextension`.)

## Run it
- **As a notebook**: open in Jupyter/JupyterLab/VS Code, Run All, then click "🔄 Refresh live data" if numbers still show "n/a" (first load fetches ~80 tickers/series and can take 20–60s — the status label under the toolbar tells you when it's done).
- **As a clean app** (recommended — hides the code, just shows the dashboard): `voila dashboard.ipynb`

## Opportunity signals
Every card across Metals, Energy, Agriculture, Currencies, Indices, Top Performers, and Industries now shows a small line under the price/return with extra signals (skipped automatically if there isn't enough data yet):
- **52wk X%** -- where the current price sits in its own 52-week range (0% = at the year low, 100% = at the year high). Computed from yfinance's live quote snapshot where available, otherwise from the same historical bars already being fetched.
- **z ±Y&sigma;** -- a mean-reversion signal: how many standard deviations the current price is from its own trailing-year average. Highlighted in amber when |z| &ge; 1.5 as a simple "notable, worth a second look" flag -- not a buy/sell signal on its own, just a stretch indicator (an extreme z-score can mean "cheap" or "in a real trend," not automatically "buy" or "sell").
- **vs S&P 500** (Top Performers and Industries only) -- excess return over the same horizon, in percentage points, vs `^GSPC`. Separates genuine outperformance from a stock or sector just riding a broad market rally.
- **&beta; / R&sup2; / &alpha;** (beta &amp; alpha new 2026-09-07; R&sup2; added the same day as a follow-up): beta, R², and annualized alpha vs the S&P 500, from the same trailing 1-year daily returns (fixed window, independent of the horizon toggle, same as z-score above). Beta says how much this instrument tends to move for each 1% move in the market (>1 more volatile, <1 less, negative = tends to move opposite it -- VIX is the clean example) -- it's a *slope*, though, not a measure of fit. **R²** is the piece beta alone can't tell you: the squared correlation between this instrument's daily returns and the market's over the same window, i.e. what share of this instrument's own return variance the market actually explains, versus idiosyncratic/company-specific noise. A high beta paired with a low R² (flagged amber below 10%) means that amplification is real on average but not a reliable day-to-day predictor -- most of what moves this instrument isn't the market at all. Alpha is a simplified Jensen's alpha (skips the risk-free-rate subtraction the textbook version uses, disclosed as a simplification -- see `_beta_alpha_r2()`'s docstring): the return this instrument generated beyond what its own beta and the market's move alone would predict, annualized. All three computed together, in one regression pass, for every single-instrument card that goes through the shared return-stats engine -- stocks, sector ETFs, indices, metals, energy, agriculture, currencies (including the inverted USD/EUR and USD/GBP pairs), crypto, and VIX/DXY on the Signals tab. **Deliberately not computed** for Bonds (yield *levels*, not a return series -- a beta on a yield doesn't carry the same meaning), Cash & Liquidity gauges and COT positioning (also levels/%s, not prices), and the derived ratios/spreads (Gold/Silver Ratio, Copper/Gold Ratio, WTI-Brent Spread, Diesel Crack Spread) -- those are synthetic constructs built from two other tracked instruments, not a single priced thing with a standard market beta of its own.
- **Vol X &middot; 30d avg Y** (new 2026-09-08): the latest daily trading volume and its trailing-30-day average, shown as a small line under the price (e.g. "Vol 42.1M &middot; 30d avg 38.6M"), formatted in K/M/B for readability. Sourced directly from the same historical bars already being fetched for every card -- no extra network calls. Shown for stocks, sector ETFs, indices, metals, energy, agriculture futures, and crypto. **Not shown** for Currencies (FX pairs don't carry a meaningful, comparable volume figure from this data source) or Bonds/yields (a yield is a rate, not a traded quantity) -- the line is simply omitted rather than showing a misleading zero.
- **Trend: Reinforcing / Neutral / Reverting (&plusmn;X.XX)** (new 2026-09-13): the lag-1 autocorrelation of daily returns over a trailing ~90 days -- a deliberately narrow, disclosed proxy for one piece of Soros's **reflexivity theory** (the idea that markets can enter a self-reinforcing feedback loop between price and perception, rather than moving independently at random each day). There's no agreed formula for "reflexivity" itself -- this is one measurable piece of it, not a measurement of the theory. **Reinforcing** (autocorr &ge; +0.15) means a day's move has recently tended to be followed by more of the same direction, consistent with (not proof of) a self-reinforcing regime currently running. **Reverting** (autocorr &le; -0.15) means moves have recently tended to flip direction day to day -- closer to a mean-reverting pattern, or a sign a prior trend just broke down. Both extremes are flagged amber purely as "notable, worth a closer look" -- the &plusmn;0.15 threshold sits close to the statistical noise floor for a correlation at this sample size (~1/&radic;40 &asymp; 0.16), not an arbitrary round number, and neither reading is a buy/sell signal. Says **nothing about the size or timing** of any future move -- and if anything, a strong Reinforcing reading describes the exact kind of regime reflexivity theory says is most prone to an eventual sharp reversal once the underlying gap becomes unsustainable, so it's a reason for more caution about a reversal, not more confidence a move continues. Computed on the instrument's own price series only (no benchmark needed, unlike beta/alpha/R²); same scope otherwise -- stocks, sector ETFs, indices, metals, energy, agriculture, currencies (including inverted pairs), crypto, and VIX/DXY, not Bonds, Cash & Liquidity, COT, or derived ratios/spreads.

Not yet built: a yield-curve slope/inversion indicator on the Bonds tab, and a personal watchlist/portfolio tracker (enter your own holdings and see live P&L) -- both discussed but not requested yet.

## Cash & Liquidity tab
A full "cash holdings across all fund managers, public companies, traders, and investors" doesn't exist as one dataset -- nobody discloses that comprehensively. What's actually free and real, split into two sections:
- **Market-Wide Cash Gauge**: FRED's retail money market fund assets (weekly) and total industry assets (quarterly, retail+institutional combined) as proxies for cash-on-the-sidelines, plus the Fed's daily overnight reverse repo usage. All verified live before shipping, like everything else in this notebook.
- **Normalized cash gauge (share of M2)**: the raw dollar figures above are nominal, so "+12% YoY" on a cash aggregate could just mean the money supply grew, not that investors got more cautious. Dividing Retail MMF Assets and Total MMF Industry Assets by FRED's `M2SL` (confirmed live) nets most of that out -- if MMF assets grow faster than M2 itself, that's real evidence of a relative shift toward cash. Retail MMF balances are technically already part of M2's own definition, so that ratio is a clean compositional share; the total-industry version (which includes institutional money, not part of M2) is a broader cross-check rather than an exact share. Doesn't isolate yield-chasing flows (MMF money moving in/out purely because MMF yields vs. bank deposit yields changed) from genuine risk-sentiment shifts -- both look the same in this ratio.
- **Trading Liquidity Stress (Amihud)** (new 2026-09-22): a cross-sectional Amihud (2002) illiquidity ratio -- |daily return| &divide; dollar volume, &times;10<sup>6</sup>, aggregated as the median across the whole tracked stock universe -- added after discussing whether the academic Pastor-Stambaugh liquidity factor could power a signal here. It can't, for free: PS's own measure needs intraday buyer/seller-signed trade data, and the user's own research found even the real factor is trailing/coincident with a recession or boom's liquidity conditions, not leading. Amihud's ratio is PS's well-known, much simpler cousin -- same underlying idea (price impact of trading), but needs only daily close/volume already cached for every stock, so this adds no new network calls. Rising = liquidity thinning (more price impact per dollar traded); falling = liquidity healthy. **Deliberately trailing/coincident, not predictive** -- same honest limitation as the real PS factor -- meant to confirm/size a liquidity-stress regime already suspected from faster gauges (VIX, HY OAS on the Signals tab), not to get ahead of one.
- **Smart Money Cash Position**: aggregate cash-as-%-of-assets across SEC-registered N-PORT filers (mutual funds & ETFs, money market funds excluded) -- a real, regulator-sourced version of the classic "mutual fund cash ratio" sentiment gauge (low = funds fully invested, little dry powder; high = more cash cushion sitting out). Reports two figures: **Loose Cash** (cash not itemized as a holding -- e.g. tested live, this came out to just 0.27% asset-weighted, because large funds mostly don't hold liquidity this way) and **Near-Cash**, which adds holdings tagged as short-term investment vehicles (money market funds, liquidity pools) or repurchase agreements -- the categories Form N-PORT's own instructions define for exactly this. Near-Cash needs the largest table in the whole dataset, so it's noticeably heavier than Loose Cash alone, and since SEC doesn't publish the literal category codes used in the bulk files, this detects them from the real data each run and discloses exactly what matched -- if it can't confidently identify them, Near-Cash is left unavailable rather than guessed at. Built from SEC's free bulk N-PORT Data Sets, confirmed live -- but each quarter's file is ~400-480MB, so this is **heavy and opt-in only**, loaded by a dedicated button rather than the normal Refresh cycle. Covers all N-PORT filers (not just equity-only funds) and only the latest quarter (no automatic historical trend). A real but broader/rougher proxy than the textbook version -- see `fetch_smart_money_cash()`'s docstring for the full methodology.
- **Berkshire Hathaway Cash**: cash & equivalents from Berkshire's own SEC filings via SEC EDGAR's free XBRL API (the only company card kept here -- Apple/Microsoft/Alphabet/Amazon/Meta were dropped since generic corporate cash for operating companies mostly reflects buyback/tax/M&A timing, not a market signal; Berkshire's is different because capital allocation *is* its business). **This one piece could not be live-tested** from the build environment (SEC EDGAR requests wouldn't complete there) -- unlike every other source in this notebook. It follows SEC's documented, stable API format and degrades cleanly if something's off, but treat your first real run as the actual test of this feature, and let me know if a number looks wrong. Also edit the `SEC_USER_AGENT` constant near the top with your own name/email before relying on it -- SEC's access policy asks for a real contact, not a placeholder.
- Not free anywhere: the BofA Global Fund Manager Survey cash % (proprietary/paywalled), aggregate hedge fund cash allocation (13F filings only disclose long equity positions, not cash or shorts), and individual trader/retail cash balances (private brokerage data).

## Notes / known gaps -- verified against live sources on 2026-08-13
Every ticker/series below was individually checked against a live Yahoo Finance or FRED page before shipping (not assumed from memory). Specifics:
- **Freshness**: every displayed price/yield (stocks, ETFs, metals, energy, agriculture, currencies, indices, and US Treasury yields) is read live -- not from the last row of the historical daily-bar chart, which can lag or misalign with the headline number. **Fixed 2026-09-01**: the live-price source used to be yfinance's `fast_info` snapshot alone; the user reported Brent (BZ=F) showing a materially lower price than the real market, and confirming it live (a direct query against Yahoo's own public chart endpoint) showed Yahoo's real-time data was correct all along -- `fast_info` itself is a known-fragile summary wrapper that can lag/mis-map for some instruments, futures contracts especially, without ever raising an error. Rather than patch Brent alone, every symbol anywhere in this notebook now reads its live price from that same directly-verified Yahoo chart endpoint first (`_yahoo_live_quote()`), falling back to `fast_info` only if that direct call fails. Historical bars are still used (correctly) for the 1mo/1yr/5yr % change math. In practice this means every number should match Yahoo's own quote page at the moment you click Refresh, at minimum reflecting the most recently completed trading session. The Bonds tab's non-US countries get their freshness the same way, just via a different live source -- see below. **Fixed 2026-09-07**: that fast_info fallback was always silent -- fine for graceful degradation, but a user reported Energy tab prices looking stale with no error shown, and there was genuinely no way to tell from the dashboard whether that refresh actually used the verified direct feed or had quietly slipped back to the older fast_info path (e.g. if a specific network blocks/throttles just that one endpoint). Every card now shows a visible <span style="color:#B8862B">&#9888; fallback quote</span> flag when that happens (amber if it fell back to fast_info, red if even that failed and it's showing the last historical close) -- if you ever see this flag, that specific number may be stale and is worth a second look or a retry.
- **US Treasury yields** (`^IRX`/`^FVX`/`^TNX`/`^TYX`): confirmed quoted directly as % (e.g. `^TNX` = 4.68 means 4.68%) -- no `/10` scaling, contrary to an older convention. Note `^IRX` uses a discount-yield convention, not bond-equivalent yield -- a small persistent gap vs. some other public sources for the same maturity is a real convention difference, not an error.
- **International bond yields** (Germany, France, UK, Japan, Switzerland, Euro Area, and China): **every** maturity bucket (short/~5Y, medium/~10Y, long/~30Y) on each Bonds tab card now first tries a real, live, daily figure read directly from that maturity's own TradingEconomics page (`requests` + a regex against their meta tag -- no JS/browser needed, confirmed against 5-country/5-and-30-year pages plus all seven 10-year pages live on 2026-08-31). **Fixed 2026-08-31**: short/long used to be *estimated* (never scraped) by shifting the medium point using the real US curve's own 5Y-10Y/30Y-10Y spread that day -- a fine proxy only when a country's curve shape resembles the US's, and confirmed badly wrong for some right now (Japan's real 30Y was 4.15% on 2026-08-31, the old estimate would have shown ~3.15% -- a full point off; France/Germany diverged by 0.5-0.8pp too). This is real, current data either way, but **not an official API** -- TradingEconomics' terms are written for manual/paid-API access, not automated scraping, so treat these cards as more fragile/gray-area than everything else in the notebook (page-wording or blocking changes can silently break any one scrape). If the **medium (10Y)** scrape fails on a given refresh, Germany/France/UK/Japan/Switzerland/Euro Area fall back automatically to FRED's OECD long-term rate series (also real, but only monthly with a real-world lag of roughly 6-8 weeks -- confirmed directly against FRED while building this: as of 2026-08-24, Germany's newest available FRED point was dated 2026-06-01, not August). If a **short/long** scrape specifically fails, only that bucket falls back to the old US-curve-spread estimate, marked `~`. **China has no FRED equivalent at all** (it isn't an OECD member, and `IRLTLT01CNM156N` does not exist), so a failed China medium-bucket scrape shows "unavailable this refresh" with no fallback. Each card's own note names its actual source and exact as-of date, per bucket -- check that before comparing against a live source elsewhere. Because TradingEconomics' free page gives no downloadable history, only FRED's six countries (not China) have a Chart-tab entry, using FRED's monthly 10-year series either way (see `fetch_bond_yield_te()` and `fetch_fred_yield()`).
- **Copper** (`HG=F`) confirmed quoted directly in $/lb (no cents scaling, unlike grains) at ~$6.61/lb. **Aluminum** (`ALI=F`) confirmed at ~$3,461/tonne.
- **Cross-vendor commodity price differences are normal, not a bug**: investigated 2026-09-01 after Brent (`BZ=F`) appeared to disagree with oilprice.com. Confirmed live, side by side: Yahoo's own `BZ=F` quote matched this notebook's displayed price to the cent, while oilprice.com's real-time headline Brent figure (not the stale cached snapshot a naive fetch of that page returns -- worth knowing on its own) ran about 1-3% higher at the very same moment. Every commodity in this notebook is Yahoo Finance's own quote for that specific futures contract, confirmed to match Yahoo's own site -- a different site's headline number for "the same" commodity can legitimately differ by a real, persistent dollar or two, since these sites aggregate from their own mix of brokers/trading desks rather than the same exchange feed. If a price here ever looks off, compare against Yahoo Finance's own quote page for that exact ticker, not a different vendor's figure.
- **Brent "difference of 3-5" investigated 2026-09-22 -- a second, more specific cause than the general cross-vendor basis above**: a fresh user report of Brent looking off by $3-5 turned out to be a genuine contract-rollover mismatch, not a repeat of the vendor-basis point above. Confirmed live: Yahoo's continuous `BZ=F` had already rolled forward to the December '26 contract (~$97.5) days before Investing.com/oilprice.com/Barchart's own "Brent" headline figures rolled, which were all still quoting the soon-to-expire November '26 contract (~$101.5-101.8) -- cross-checked across all three sites plus Yahoo's own specific-month ticker for November (`BZX26.NYM`) live, same moment. Both prices were real and correct for their own contract, just ~$4 apart on two different delivery months in a backwardated market. The Energy tab's new **Brent Crude (Nearest Month)** card (see below) now shows that same nearest-expiry contract other sites default to, so the two Brent numbers on this dashboard (continuous vs. nearest-month) are directly comparable to whichever one a user is checking against elsewhere.
- **Currency pairs**: `CHF=X`/`JPY=X`/`CNH=X` confirmed to mean USD/CHF, USD/JPY, USD/CNH respectively (not the inverse). **Fixed 2026-09-02**: every row on the Currencies tab now reads USD-first, per request. `EURUSD=X`/`GBPUSD=X` are Yahoo's own USD-per-1-EUR/GBP pairs, so they're displayed inverted as USD/EUR and USD/GBP — price, % change, 52wk range, z-score, and MA trend are all recomputed on the actual inverted series (`fetch_returns_inverted()`), not just relabeled or sign-flipped. `EUR/CHF` has no USD leg, so it's kept as its own real cross.
- **Indices**: all 11 tickers (`^GSPC`, `^IXIC`, `^DJI`, `^FTSE`, `^GDAXI`, `^FCHI`, `^N225`, `^HSI`, `^KS11`, `^SSMI`, `EEM`) confirmed live and trading.
- **Currency of quotes, and FX handling**: commodities (metals/energy/agriculture) are standard USD-denominated futures contracts -- real dollars, nothing to convert. Crypto is quoted natively in USD on Yahoo Finance (e.g. `BTC-USD`) -- also no conversion involved. The Currencies tab itself displays FX *rates* (e.g. EUR/USD), so there's no separate "convert to USD" step -- the number shown already **is** the exchange rate. **Indices are the one place local currency matters and is NOT converted**: `^FTSE` is in GBP, `^GDAXI`/`^FCHI` in EUR, `^N225` in JPY, `^HSI` in HKD, `^KS11` in KRW, `^SSMI` in CHF -- each shown as Yahoo Finance's own raw index level in that market's native currency, exactly as every financial site quotes them (an index level is a calculated point value, not literally "money," so this is standard practice, not an oversight). The practical implication: don't compare raw levels across indices (Nikkei's ~40,000 vs S&P 500's ~6,000 means nothing on its own), and know that the displayed % change for a non-USD index is that index's *local-currency* return -- if you needed the USD-hedged return instead (what a US-based investor actually earned, including currency movement), that would require multiplying by the relevant FX pair's change over the same period, which this dashboard does not do.
- **Crypto**: a new tab tracking the top 5 non-stablecoin cryptocurrencies by market cap (Bitcoin, Ethereum, BNB, XRP, Solana), checked against slickcharts.com on 2026-08-20. This is a fixed snapshot list, not dynamically re-ranked every refresh (there's no free live market-cap-ranking source wired in) -- revisit occasionally since crypto rankings shift. Tether (USDT, #3) and USD Coin (USDC, #6) were deliberately excluded despite ranking in the real top 10 -- a $1-pegged stablecoin has no meaningful 52wk-range or z-score signal, just noise around the peg.
- **Diesel Crack Spread** (Energy tab): a derived refining-margin signal, not a new data source -- (Heating Oil futures price &times; 42 gal/bbl) minus WTI crude price, both tickers already tracked on this tab. Uses `HO=F` as diesel's proxy (CME/NYMEX's Heating Oil contract has tracked ultra-low-sulfur diesel since a 2013 spec change). % change and z-score are computed on the spread's own historical series, not derived from either leg's separate return -- also chartable from the Chart tab.
- **Signals tab** (new): cross-asset risk/stress gauges, all confirmed live 2026-08-20/21 -- **VIX** (`^VIX`, equity fear gauge), **US Dollar Index** (`DX-Y.NYB`), **10Y-2Y Yield Curve** (FRED `T10Y2Y`, negative = inverted), **HY Credit Spread** (FRED `BAMLH0A0HYM2`, ICE BofA option-adjusted spread), and **10Y Breakeven Inflation** (FRED `T10YIE`). All five respect the horizon toggle and get the same 52wk-range/z-score signal line as everything else.
- **Gold/Silver Ratio and Copper/Gold Ratio** (Metals tab): derived from prices already tracked -- no new data source. Copper/Gold is scaled &times;1000 for readability, the conventional way that ratio is quoted (matches typical reported ranges of roughly 1.5-3.5).
- **WTI-Brent Spread** (Energy tab, substituted for oil term structure): a clean, free version of contango/backwardation (specific futures-contract-month term structure) isn't buildable from yfinance -- continuous contracts don't cleanly expose individual expiry months, and hardcoding specific contract-month tickers is fragile since codes roll every quarter. The WTI-Brent spread is a real, commonly-watched substitute instead, reflecting regional supply/logistics dynamics rather than calendar term structure -- disclosed as a substitution, not silently swapped in.
- **50/200-Day Moving Average Crossover**: added to the signal line across every tab that already computes a z-score (stocks, indices, commodities, crypto, sector ETFs) -- "50/200: Bullish/Bearish" for the standing trend, or "Golden Cross"/"Death Cross" when the 50-day crossed the 200-day within the last 10 trading days. Computed from the same history bars already fetched -- no extra network calls. Needs ~210+ days of history, so very new listings won't show it.
- **Market Breadth** (Top Performers tab): % of the ~80 tracked stocks trading above their own 200-day moving average -- an index-independent market-health gauge (broad participation supports a rally; a shrinking % while the index rises can flag a narrow, top-heavy market).
- **Insider Buying vs Selling** (Cash & Liquidity tab, opt-in button, ~10-15MB): aggregate open-market buy/sell activity (transaction codes P/S only -- option exercises, grants, and gifts excluded) from SEC's free bulk Insider Transactions (Form 3/4/5) data set, confirmed live 2026-08-21 (2026 Q2 file is 10.97MB). Reports buying's share of total $ volume and how many companies show net insider buying vs. net selling. Much lighter than Smart Money Cash but still opt-in to keep the normal Refresh cycle fast. Doesn't exclude 10b5-1 pre-scheduled sale plans (not reliably flagged in the raw data).
- **CFTC Commitment of Traders** (Metals + Energy tabs): weekly speculative net positioning (as % of open interest) for Gold, Silver, Copper, WTI Crude Oil, and Natural Gas, via CFTC's free public Socrata API, confirmed live 2026-08-21. Uses the Legacy report's broader "Non-Commercial" bucket (mixes hedge funds with other large speculators) rather than the narrower "Managed Money" category from CFTC's newer Disaggregated report -- a real but rougher proxy, disclosed as such. The specific futures contract for each commodity is detected dynamically each run (largest open interest under that commodity name) rather than a hardcoded contract code, since thin secondary contracts share the same name.
- **Scheduled signal check**: a recurring automated check (see chat) that reviews the same stretch/threshold signals (VIX level, yield-curve status, z-scores, fresh MA crossovers) and messages a short summary on its own schedule -- separate from this notebook, doesn't require the notebook to be open.
- Corn/Wheat/Soybean (CBOT) and Cotton/Sugar/Coffee (ICE) settle in **cents** per unit on the exchange -- divided by 100 to show actual dollars. Rice (CME Rough Rice) already settles in $/cwt directly, no scaling.
- **If a tab shows "n/a" everywhere**: that tab's data hasn't loaded yet — click "🔄 Refresh live data" and wait for the status label to say "Last refreshed …". If something is still "n/a" after a completed refresh and it isn't China's bond card (expected -- see above), tell me which one and I'll dig into that specific ticker.
- **Bonds** doesn't use the horizon toggle at all (it's hidden on that tab) — it shows current yield *levels* by maturity bucket (short/medium/long), not a return over time. US is real/live on all three buckets. Every other country (including China) tries a real scrape on all three buckets now; a bucket only falls back to an estimate (marked with `~`) if that specific maturity's own TradingEconomics scrape fails on a given refresh — check each card's own note for what actually happened this refresh. **Switzerland fixed 2026-09-02**: TradingEconomics has no 5Y or 30Y page for Switzerland at all (only 2Y and 10Y) — its SHORT bucket now reads the real 2Y page instead (labeled `~2Y`, not `~5Y`), and its LONG bucket shows "unavailable" instead of the usual US-curve-spread estimate, which was checked against a real published Swiss 30Y figure and found off by ~0.4 points.
- **Chart** tab also ignores the horizon toggle (hidden there too) — it has its own period selector (1mo/6mo/1yr/5yr) instead. **Fixed 2026-09-01**: it now renders as an interactive Plotly figure instead of a static image, so moving the cursor along the curve shows the exact date and price/yield at that point (a static PNG couldn't do this). Every other tab's cards are unaffected and stay static PNG via matplotlib. Needs the `plotly` package (added to the one-time setup command above) — if it's not installed, this tab alone will error on Plot; everything else in the notebook still works.
- Wool, Lithium, Cobalt, Tin, and Thermal Coal have no reliable free futures/ETF ticker on Yahoo Finance, so they're left out rather than faked.
- Sector "Industries" drill-down ranks a pool of 8–12 real stocks per sector by actual 1mo/1yr/5yr return — the top-6 **set**, not just the order, can change when you switch horizons.
- **Color-coded card frames** (new): every card's left+bottom border is now colored by how stretched it is. Individual stocks and sector ETFs are colored by excess return vs the S&P 500 this horizon (deep green ≥+5pp, light green +1 to +5pp, gray in between, light red -1 to -5pp, deep red ≤-5pp). Everything else with a z-score (metals, energy, agriculture, currencies, indices, signals, cash gauges, COT, ratios/spreads) is colored the same way but using |z|≥2 / |z|≥1 thresholds instead. Reuses numbers already computed elsewhere on each card (the z-score badge, the "vs S&P 500" line) — no new data or network calls. Cards without either (Bonds, insider/Smart Money one-off snapshots) keep the plain neutral border.
- **Refresh spinner** (new): the status line under Refresh now shows a small spinning CSS indicator while `refresh_all()` is running, not just static text -- purely visual, no behavior change.
- **VIX implied monthly move + regime label** (Signals tab): the VIX card now also shows VIX/√12 as a rough implied monthly move, plus a commonly-cited regime label (Very Low/Normal/Elevated/High Stress/Crisis) -- disclosed as finance-media convention, not an official scale.
- **HY Credit Spread stress-territory label** (Signals tab): same idea -- Tight/Normal/Elevated/Distressed bands based on commonly-cited OAS levels.
- **Metals/Agriculture fund alternatives** (new): Gold and Silver ETF cards (Swisscanto/ex-ZKB, SIX Swiss Exchange, CHF, physically-backed) on the Metals tab, and a Wheat ETC card (WisdomTree, London Stock Exchange, not Swiss/iShares -- no such product could be confirmed) on the Agriculture tab. Unlike everything else in this notebook, these three tickers could not be live-tested from the build environment (Yahoo Finance access was blocked there) -- confirmed to exist via live web search instead, so treat your first real run as the actual verification.
- **Corn/Soybean ETFs** (Agriculture tab): Teucrium's CORN and SOYB, long-established US-listed, USD-denominated funds (trading since 2010/2011) -- confirmed real via live web search, same "could not be live-tested from this build environment" caveat as the other fund tickers above.
- **EM Currency Basket** (Currencies tab): WisdomTree's CEW, a USD-denominated fund long a basket of ~15 emerging-market currencies -- the mirror image of what DXY does for G10 currencies, but for EM. Confirmed real via live web search, same live-test caveat. No EUR-denominated equivalent could be confirmed to exist (EM currency baskets are conventionally quoted vs USD, not EUR) -- disclosed as a gap rather than approximated with a synthetic cross calculation.
- **Gold fund pick, 3rd iteration** (Metals tab): now Invesco Physical Gold ETC (SGLD, London Stock Exchange, USD, 0.12% TER) -- an ETC (collateralized debt obligation, Ireland-domiciled, JPMorgan custodian), not a fund like the Swiss listing, but still fully allocated physical gold and materially cheaper on TER. Replaces both earlier gold picks (Swisscanto/ZKB Gold at 0.40% TER, and iShares Gold Trust Micro/IAUM which was cheap per-share at ~$45-50 -- bring IAUM back if per-share affordability matters again, since SGLD trades around $420/share). Silver keeps its original Swiss Swisscanto/ZKB listing. Same live-test caveat as the other fund tickers.
- **Corn/Wheat/Sugar/Rice COT positioning** (Agriculture tab, new): same CFTC Commitment of Traders methodology as the Metals/Energy tabs' COT cards, confirmed against CFTC's live public API for these commodity names (CORN, WHEAT, SUGAR, RICE) before shipping.
- **1 Day horizon** (new): the horizon toggle now has a 1-day option alongside 1mo/1yr/5yr, for checking the most recent single-session move. It reads "n/a" for anything that isn't updated daily -- COT positioning, international bond yields (monthly OECD series), insider filings (quarterly), and a couple of the cash-gauge FRED series (weekly/quarterly) -- since there's no genuine daily figure to show between those series' own release dates. Everything priced via Yahoo Finance (stocks, indices, sectors, commodities, currencies, crypto, US Treasury yields, and the derived ratios/spreads built from them) gets a real 1-day reading.


In [19]:
import yfinance as yf
import pandas as pd
import requests
import re
import zipfile
import tempfile
import os
import time
from pandas_datareader import data as pdr
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

import logging
# yfinance logs (doesn't raise) when a ticker's own valid-range list from
# Yahoo doesn't include the period requested -- e.g. CNH=X (USD/CNH) has
# been confirmed (2026-08-31, direct query against Yahoo's own chart API)
# to only support period="1d"/"5d" now, with no deeper history at all
# (firstTradeDate is null, volume is 0 -- Yahoo just doesn't carry more for
# this specific thin FX cross). _get_closes() already retries with a
# shorter period and degrades to "n/a" for that symbol's longer horizons
# cleanly -- this only silences yfinance's own noisy log line about it so
# it doesn't clutter cell output on every refresh; it doesn't change what
# data is fetched or hide a real fetch failure elsewhere.
logging.getLogger("yfinance").setLevel(logging.CRITICAL)

import matplotlib
matplotlib.use("Agg")  # headless-safe; static PNG cards use this, not the Chart tab
import matplotlib.pyplot as plt
import io

import plotly.graph_objects as go  # Chart tab only -- gives a real hover tooltip along the curve

import ipywidgets as widgets
from IPython.display import display, clear_output


In [20]:
# ---------------------------------------------------------------------------
# Ticker registries. Everything here is a real, publicly tradable
# ticker/futures/FX symbol on Yahoo Finance -- no synthetic or estimated data.
# Every symbol/series in this file was individually checked against a live
# Yahoo Finance or FRED page (2026-08-13) -- see intro cell for specifics.
# ---------------------------------------------------------------------------

METALS = {
    "Gold":        ("GC=F", 1, "/oz"),
    "Silver":      ("SI=F", 1, "/oz"),
    "Platinum":    ("PL=F", 1, "/oz"),
    "Copper":      ("HG=F", 1, "/lb"),
    "Palladium":   ("PA=F", 1, "/oz"),
    "Aluminum":    ("ALI=F", 1, "/tonne"),
}

# Fund/ETP alternatives to the raw futures above -- physically-backed,
# exchange-traded shares rather than futures contracts. Researched and
# confirmed as real, currently-listed Yahoo Finance tickers via live web
# search (this build environment's network access to Yahoo Finance itself
# is blocked, so unlike every other source in this notebook these specific
# tickers could NOT be live-tested end-to-end here -- treat your first real
# run as the actual verification, flag back if a price looks obviously
# wrong). Silver keeps its Swiss listing (Swisscanto/formerly-ZKB, SIX
# Swiss Exchange, CHF, full physical bar replication, 0.40% TER) -- no
# cheaper silver alternative has been requested/researched yet. Gold has
# gone through two earlier iterations before landing here: (1) Swisscanto/
# ZKB Gold (ZGLD.SW, same Swiss structure as silver, but a legally distinct
# wrapper -- a genuine fund under Swiss CISA, with a relatively high 0.40%
# TER and a correspondingly high per-share price); (2) iShares Gold Trust
# Micro (IAUM, NYSE Arca) picked for a low per-share price (~$45-50,
# tracking ~1/100oz) and the lowest TER of any major US gold ETF (0.09%).
# Both were replaced with Invesco Physical Gold ETC (SGLD, London Stock
# Exchange, USD, 0.12% TER) per explicit request, after comparing it
# against ZGLD: SGLD is an ETC -- legally a collateralized debt obligation
# (a secured note), not a fund -- Ireland-domiciled, custodied by JPMorgan
# Chase Bank, benchmarked to the LBMA Gold Price. Functionally similar
# protection to ZGLD (fully allocated, ring-fenced physical gold shields
# holders from the issuer's own credit risk either way) but a different
# legal wrapper, materially cheaper (0.12% vs 0.40%), and NOT cheap on a
# per-share basis (~$420/share, similar magnitude to GLD) -- if per-share
# affordability matters again later, IAUM is the one to bring back. These
# are fund/ETC SHARE prices, not the underlying spot commodity price --
# they won't move 1:1 with GC=F/SI=F due to currency conversion (where
# applicable) and fees/tracking on top of the commodity move.
METALS_ETF = {
    "Silver ETF (Swisscanto/ZKB, SIX, CHF, 0.40% TER)":  ("ZSIL.SW", "CHF "),
    "Gold ETC (Invesco Physical Gold, LSE, USD, 0.12% TER)": ("SGLD.L", "$"),
}
# (symbol, price_prefix) -- Corn/Soybean are long-established, US-listed,
# USD-denominated Teucrium funds (CORN since 2010, SOYB since 2011) tracking
# futures baskets rather than physical delivery; confirmed real via live web
# search. Wheat has no confirmed US/USD equivalent -- see METALS_ETF note.
AGRI_ETF = {
    "Corn Fund (Teucrium, NYSE Arca, USD)":    ("CORN", "$"),
    "Soybean Fund (Teucrium, NYSE Arca, USD)": ("SOYB", "$"),
    "Wheat ETC (WisdomTree, LSE, GBP)":        ("WEAT.L", ""),
}

ENERGY = {
    "WTI Crude":                  ("CL=F", 1, "/bbl"),
    "Brent Crude":                ("BZ=F", 1, "/bbl"),
    "Natural Gas":                ("NG=F", 1, "/MMBtu"),
    "Heating Oil / Diesel proxy": ("HO=F", 1, "/gal"),
}

# Corn/Wheat/Soybean (CBOT) and Cotton/Sugar/Coffee (ICE) settle in CENTS per unit
# on the exchange -- yfinance returns that raw cents figure, so we divide by 100
# to show actual dollars. Rice (CME Rough Rice) already settles in $/cwt directly.
AGRI = {
    "Corn":     ("ZC=F", 100, "/bu"),
    "Wheat":    ("ZW=F", 100, "/bu"),
    "Soybean":  ("ZS=F", 100, "/bu"),
    "Cotton":   ("CT=F", 100, "/lb"),
    "Rice":     ("ZR=F", 1,   "/cwt"),
    "Sugar":    ("SB=F", 100, "/lb"),
    "Coffee":   ("KC=F", 100, "/lb"),
}

CURRENCIES = {
    "USD/EUR":  "EURUSD=X",
    "USD/GBP":  "GBPUSD=X",
    "USD/JPY":  "JPY=X",
    "USD/CHF":  "CHF=X",
    "USD/CNH":  "CNH=X",
    "EUR/CHF":  "EURCHF=X",
}

# Yahoo's own EURUSD=X/GBPUSD=X convention quotes USD per 1 EUR/GBP (the
# standard FX-market "EUR/USD"/"GBP/USD" reading) -- at the user's request
# that every row on this tab read USD-first, these two are displayed
# inverted (EUR/USD per 1 USD) instead. USD/JPY, USD/CHF, USD/CNH already
# read USD-first on Yahoo natively, so they're untouched. EUR/CHF has no
# USD leg at all, so it's left as its own real cross rather than forced
# into a "USD/x" shape that wouldn't mean anything. Every derived signal
# (52wk range, z-score, MA crossover) for these two is recomputed directly
# on the actual inverted price series (see fetch_returns_inverted()), not
# algebraically guessed at from the original pair's numbers -- inversion
# doesn't simply flip a z-score's sign or a crossover's timing.
CURRENCIES_INVERTED = {"USD/EUR", "USD/GBP"}

# Real, confirmed-live basket-currency proxy: WisdomTree's CEW is a USD-
# denominated ETF that's long a basket of ~15 EM currencies (equal-weighted,
# rebalanced quarterly) funded via short USD forwards + a US money-market
# portfolio -- i.e. it rises when EM currencies strengthen vs the dollar and
# falls when they weaken, the mirror image of what DXY does for G10
# currencies. No EUR-denominated equivalent (a basket of EM currencies
# quoted against EUR rather than USD) could be confirmed via live search --
# EM currency baskets are conventionally quoted vs USD, not EUR, so this is
# disclosed as a genuine gap rather than approximated with a synthetic cross
# calculation that could mislead. It's an actively-managed fund SHARE price
# (subject to fees/forward-roll yield), not a clean index level like DXY --
# watch the % change, not the raw price level, for the actual FX signal.
CURRENCY_BASKET_ETF = {
    "EM Currency Basket vs USD (WisdomTree CEW)": "CEW",
}

INDICES = {
    "S&P 500":              "^GSPC",
    "NASDAQ Composite":     "^IXIC",
    "Dow Jones":            "^DJI",
    "FTSE 100":             "^FTSE",
    "DAX":                  "^GDAXI",
    "CAC 40":               "^FCHI",
    "Nikkei 225":           "^N225",
    "Hang Seng":            "^HSI",
    "KOSPI":                "^KS11",
    "Swiss Market Index":   "^SSMI",
    "MSCI EM (EEM proxy)":  "EEM",
}

# Top 5 cryptocurrencies by market cap, checked live against slickcharts.com
# on 2026-08-20 -- a STATIC snapshot list, not dynamically re-ranked on every
# refresh (that would need a separate live market-cap-ranking API call per
# coin, which yfinance doesn't provide reliably/for free). Rankings shift
# over time -- revisit this list occasionally.
# Stablecoins deliberately EXCLUDED even though they rank in the real top 10
# by market cap (Tether USDT was #3, USDC was #6 on the date checked): a
# coin pegged to $1 has no real price signal -- its 52-week range and
# z-score would just be noise around a fixed peg, not a meaningful stretch
# indicator. So this is "top 5 non-stablecoin cryptocurrencies by market
# cap": Bitcoin, Ethereum, BNB, XRP, and Solana (the next-largest after
# skipping USDT/USDC).
CRYPTO = {
    "Bitcoin":  "BTC-USD",
    "Ethereum": "ETH-USD",
    "BNB":      "BNB-USD",
    "XRP":      "XRP-USD",
    "Solana":   "SOL-USD",
}

# Macro/risk gauges -- VIX and the Dollar Index are live yfinance tickers
# (both confirmed live 2026-08-20/21: ^VIX=14.89, DX-Y.NYB=98.81); the other
# three are FRED daily series (all confirmed live the same day: T10Y2Y=0.50,
# BAMLH0A0HYM2=2.73, T10YIE=2.34).
SIGNALS_YF = {
    "VIX (Fear Gauge)": "^VIX",
    "US Dollar Index": "DX-Y.NYB",
}
SIGNALS_FRED = {
    "10Y-2Y Yield Curve": "T10Y2Y",
    "HY Credit Spread (OAS)": "BAMLH0A0HYM2",
    "10Y Breakeven Inflation": "T10YIE",
}

# CFTC Commitment of Traders (Legacy Futures Only report) -- free, weekly,
# no key, via CFTC's public Socrata API (confirmed live 2026-08-21). Maps
# each commodity to the CFTC "commodity_name" field; at runtime the code
# picks whichever contract under that name has the largest open interest on
# each report date (e.g. NYMEX Henry Hub for Natural Gas, COMEX for Gold) --
# detected dynamically rather than a hardcoded contract code, since several
# thin secondary contracts (ICE basis swaps, micro/mini contracts, etc.)
# share the same commodity_name.
COT_COMMODITIES = {
    "Gold":          "GOLD",
    "Silver":        "SILVER",
    "Copper":        "COPPER",
    "WTI Crude Oil": "CRUDE OIL",
    "Natural Gas":   "NATURAL GAS",
    "Corn":          "CORN",
    "Wheat":         "WHEAT",
    "Sugar":         "SUGAR",
    "Rice":          "RICE",
}
_COT_BASE_URL = "https://publicreporting.cftc.gov/resource/6dca-aqww.json"

# SEC bulk Insider Transactions (Form 3/4/5) data sets -- free, quarterly, no
# key, ~8-15MB per quarter (confirmed live 2026-08-21 -- 2026 Q2 is 10.97MB).
# Two URL path prefixes have been used historically (SEC reorganized their
# structured-data hosting at some point) -- the most recent quarter uses
# "datastandardsinnovation", everything before that uses "structureddata";
# both are tried since a future quarter could use either.
_INSIDER_URL_TEMPLATES = [
    "https://www.sec.gov/files/datastandardsinnovation/data/insider-transactions-data-sets/{year}q{quarter}_form345.zip",
    "https://www.sec.gov/files/structureddata/data/insider-transactions-data-sets/{year}q{quarter}_form345.zip",
]

BONDS_US = {
    "US 13-Week":  "^IRX",
    "US 5-Year":   "^FVX",
    "US 10-Year":  "^TNX",
    "US 30-Year":  "^TYX",
}

# FRED OECD "long-term interest rate" series -- the standard free, no-API-key
# proxy for each country's ~10-year government bond yield. Published monthly,
# not daily, so no longer render_bonds()'s primary source for these
# countries (see BONDS_INTL_TE below) -- kept as the automatic fallback if
# the live TE scrape fails, and as the sole source for the Chart tab's long
# history (TE's free page has no downloadable history). China deliberately
# excluded: it isn't an OECD member, and FRED has no equivalent 10-year
# series for it (confirmed -- IRLTLT01CNM156N does not exist).
BONDS_INTL = {
    "Germany 10Y":     "IRLTLT01DEM156N",
    "France 10Y":      "IRLTLT01FRM156N",
    "UK 10Y":          "IRLTLT01GBM156N",
    "Japan 10Y":       "IRLTLT01JPM156N",
    "Switzerland 10Y": "IRLTLT01CHM156N",
    "Euro Area 10Y":   "IRLTLT01EZM156N",
}

# Live TradingEconomics scrape targets for the same six countries -- the
# PRIMARY source for their Bonds tab card now (real, daily, not the ~6-8
# week-lagged FRED point). Just the url_slug -- the country-name label used
# to live here too, but was dropped 2026-08-31 (see fetch_bond_yield_te's
# docstring: TE's own pages are inconsistent about it, e.g. UK's 10Y page
# says "United Kingdom" but its 5Y/30Y pages say "UK", so matching it was
# actively wrong for some maturity pages).
BONDS_INTL_TE = {
    "Germany 10Y":     "germany",
    "France 10Y":      "france",
    "UK 10Y":          "united-kingdom",
    "Japan 10Y":       "japan",
    "Switzerland 10Y": "switzerland",
    "Euro Area 10Y":   "euro-area",
}

# TradingEconomics page path + the exact maturity marker text that appears
# in that page's own <meta name="description"> tag ("The yield on <country>
# <marker> Bond Yield ..."). Confirmed live against Japan/Germany/France/
# UK/China's 5Y and 30Y pages on 2026-08-31 -- Switzerland and Euro Area's
# maturity sub-pages were NOT individually verified (network timeouts while
# building this), but fetch_bond_yield_te() degrades cleanly to the old
# US-curve-spread estimate if a given page/pattern doesn't match, so this
# is safe even if those two don't follow the same template.
TE_MATURITY_PAGES = {
    "s": ("5-year-note-yield",     "5 Year"),  # short bucket
    "m": ("government-bond-yield", "10Y"),     # medium bucket
    "l": ("30-year-bond-yield",    "30 Year"), # long bucket
}

# Switzerland has no TE page at the standard 5-year maturity the other six
# countries use for the SHORT bucket -- confirmed live 2026-09-02:
# /switzerland/5-year-note-yield redirects to TE's own homepage (i.e.
# doesn't exist; TE's own /bonds country table and this page's own link
# list confirm Switzerland is only tracked at 2Y and 10Y). The real page
# that does exist is /switzerland/2-year-note-yield. Keyed by TE url_slug;
# (page_path, maturity_label, display_tag).
TE_SHORT_OVERRIDES = {
    "switzerland": ("2-year-note-yield", "2Y", "2Y"),
}

# Switzerland also has no TE page at the 30-year maturity (confirmed live
# 2026-09-02: /switzerland/30-year-bond-yield also redirects to TE's
# homepage), so its LONG bucket always fell back to the US-curve-spread
# estimate. Checked that estimate against a real published Swiss 30Y figure
# (~0.59% via live web search, 2026-09-02) and it was off by roughly 0.4
# points that day -- the same category of miss the pre-fix Japan estimate
# had. Rather than keep showing a number confirmed to be meaningfully
# wrong, Switzerland's LONG bucket shows "unavailable" instead, the same
# way China's medium bucket does when it has no real reading.
TE_LONG_NO_ESTIMATE = {"switzerland"}

# 2-year government bond yield -- added 2026-09-18 at the user's request.
# TradingEconomics has a dedicated 2-year page for every TE country tracked
# here EXCEPT Euro Area (confirmed live 2026-09-18: /euro-area/2-year-note-yield
# redirects to TE's own homepage, i.e. it doesn't exist -- the same kind of
# gap as Switzerland's missing 5Y/30Y pages). Also confirmed live that TE's
# own maturity-marker wording for the 2-year page is inconsistent across
# countries the same way the country-name wording already was (see
# fetch_bond_yield_te's docstring): Germany/France/Japan/China's pages all
# say "<country> 2 Year Bond Yield", but United Kingdom's says "United
# Kingdom 2Y Bond Yield" (no space) -- TE_2Y_LABEL_OVERRIDES captures the
# one country that doesn't match the "2 Year" default. Switzerland is
# deliberately NOT in this dict: it already has a real 2-year scrape (see
# TE_SHORT_OVERRIDES, used there because Switzerland has no 5-year page),
# so render_bonds() reuses that existing value for its 2Y bucket instead of
# scraping the same page a second time. The US has no yfinance/CBOE 2-year
# index at all (that family is limited to 13-week/5-year/10-year/30-year --
# see BONDS_US) -- its 2Y point comes from FRED's DGS2 series instead
# (confirmed live 2026-09-18: real, daily, H.15-sourced, typically 1-2
# business days behind rather than same-day like the other three US points).
TE_2Y_PAGE = "2-year-note-yield"
TE_2Y_LABEL_DEFAULT = "2 Year"
TE_2Y_LABEL_OVERRIDES = {"united-kingdom": "2Y"}
TE_2Y_UNAVAILABLE = {"euro-area"}
US_2Y_FRED_SERIES = "DGS2"

# ---------------------------------------------------------------------------
# Cash & Liquidity signals.
#
# Market-wide cash gauge (all free FRED series, verified live 2026-08-19):
# WRMFNS (Retail Money Market Fund Assets, weekly, not seasonally adjusted)
# and MMMFFAQ027S (Total Money Market Fund industry assets -- retail +
# institutional combined, quarterly, from the Fed's Z.1 Flow of Funds) both
# track investor cash parked in money funds. RRPONTSYD (Fed overnight
# reverse repo usage, daily) tracks cash parked directly at the Fed.
# Confirmed: the Fed discontinued its INSTITUTIONAL-only money fund series
# (WIMFSL/WIMFNS/IMFSL/IMFNS) in Feb 2021 and never replaced it -- there is
# no free live "institutional cash alone" figure, only the combined
# quarterly total via MMMFFAQ027S.
CASH_GAUGE_FRED = {
    "Retail MMF Assets (weekly)": ("WRMFNS", 1.0),          # already $B
    "Total MMF Industry Assets (qtrly)": ("MMMFFAQ027S", 0.001),  # $M -> $B
    "Fed O/N Reverse Repo Usage (daily)": ("RRPONTSYD", 1.0),      # already $B
}

# Ratio versions: MMF assets as a share of M2SL (the Fed's M2 money-supply
# series, confirmed live -- both WRMFNS and M2SL are already in $B, no
# conversion needed for that pair; MMMFFAQ027S is in $M so it needs the
# same 0.001 scale as above). Dividing by M2 nets out generic nominal
# growth (inflation, monetary-base expansion) that would otherwise
# inflate a raw dollar figure regardless of any real shift in cash
# preference -- a materially cleaner signal than the raw levels above.
# Note: retail MMF assets are technically already A COMPONENT of M2's own
# definition, so "Retail MMF Share of M2" is a compositional ratio (how
# much of the money supply sits in MMFs vs. checking/savings), which is
# the intended, standard use of this ratio, not an apples-to-oranges
# comparison of two independent series.
CASH_GAUGE_RATIOS = {
    "Retail MMF Share of M2": ("WRMFNS", "M2SL", 1.0, 1.0),
    "Total MMF Industry vs M2": ("MMMFFAQ027S", "M2SL", 0.001, 1.0),
}

# Corporate cash piles via SEC EDGAR's free XBRL company-concept API (no
# key). CIKs are stable, well-known identifiers. IMPORTANT: unlike every
# other source in this notebook, this integration could not be live-tested
# from the environment that built it (SEC EDGAR requests wouldn't complete
# there) -- see fetch_company_cash()'s docstring.
SEC_USER_AGENT = "Personal Markets Dashboard research-contact@example.com"  # EDIT this with your own name/email -- SEC's fair-access policy asks for a real identifying contact, not a placeholder.

COMPANY_CASH_CIKS = {
    "Berkshire Hathaway": "0001067983",
}

# ---------------------------------------------------------------------------
# "Smart Money Cash Position" -- aggregate cash-as-%-of-assets across
# SEC-registered N-PORT filers (mutual funds & ETFs, money market funds
# excluded since they file a different form). Built from SEC's free bulk
# N-PORT Data Sets -- confirmed live, one ZIP per quarter, ~400-480MB each.
# See fetch_smart_money_cash()'s docstring for the full methodology and
# disclosed simplifications. Heavy and opt-in only -- NOT part of the
# normal Refresh cycle.
_NPORT_BASE_URL = "https://www.sec.gov/files/dera/data/form-n-port-data-sets/{year}q{quarter}_nport.zip"

SECTOR_ETFS = {
    "Technology": "XLK", "Financials": "XLF", "Health Care": "XLV",
    "Industrials": "XLI", "Consumer Discretionary": "XLY", "Consumer Staples": "XLP",
    "Energy": "XLE", "Utilities": "XLU", "Materials": "XLB",
}

# Broader real candidate pools per sector -- ranked live by actual return,
# so the displayed top 6 can change (not just reorder) across horizons.
SECTOR_POOLS = {
    "Technology":             ["NVDA","AAPL","MSFT","AVGO","ORCL","PLTR","CSCO","ADBE","CRM","QCOM","AMD","INTC"],
    "Financials":              ["JPM","V","MA","BRK-B","GS","WFC","BAC","SCHW","AXP","C"],
    "Health Care":             ["LLY","UNH","JNJ","MRK","ABBV","MRNA","PFE","TMO","AMGN"],
    "Industrials":             ["GE","CAT","HON","GEV","RTX","ETN","LMT","BA","UNP","DE"],
    "Consumer Discretionary":  ["AMZN","TSLA","HD","MCD","BKNG","NKE","SBUX","LOW"],
    "Consumer Staples":        ["PG","COST","WMT","KO","PEP","PM","CL","MDLZ"],
    "Energy":                  ["XOM","CVX","COP","SLB","EOG","WMB","OXY","PSX"],
    "Utilities":               ["NEE","SO","DUK","CEG","VST","AEP","D"],
    "Materials":               ["LIN","SHW","FCX","APD","NEM","DOW"],
}

ALL_STOCKS = sorted({t for pool in SECTOR_POOLS.values() for t in pool})

# Static display names (avoids a slow/fragile extra API call per ticker for .info)
NAME_MAP = {
    "NVDA":"NVIDIA","AAPL":"Apple","MSFT":"Microsoft","AVGO":"Broadcom","ORCL":"Oracle",
    "PLTR":"Palantir","CSCO":"Cisco Systems","ADBE":"Adobe","CRM":"Salesforce","QCOM":"Qualcomm",
    "AMD":"AMD","INTC":"Intel","JPM":"JPMorgan Chase","V":"Visa","MA":"Mastercard",
    "BRK-B":"Berkshire Hathaway B","GS":"Goldman Sachs","WFC":"Wells Fargo","BAC":"Bank of America",
    "SCHW":"Charles Schwab","AXP":"American Express","C":"Citigroup","LLY":"Eli Lilly",
    "UNH":"UnitedHealth","JNJ":"Johnson & Johnson","MRK":"Merck","ABBV":"AbbVie","MRNA":"Moderna",
    "PFE":"Pfizer","TMO":"Thermo Fisher","AMGN":"Amgen","GE":"GE Aerospace","CAT":"Caterpillar",
    "HON":"Honeywell","GEV":"GE Vernova","RTX":"RTX","ETN":"Eaton","LMT":"Lockheed Martin",
    "BA":"Boeing","UNP":"Union Pacific","DE":"Deere & Co","AMZN":"Amazon","TSLA":"Tesla",
    "HD":"Home Depot","MCD":"McDonald's","BKNG":"Booking Holdings","NKE":"Nike","SBUX":"Starbucks",
    "LOW":"Lowe's","PG":"Procter & Gamble","COST":"Costco","WMT":"Walmart","KO":"Coca-Cola",
    "PEP":"PepsiCo","PM":"Philip Morris","CL":"Colgate-Palmolive","MDLZ":"Mondelez",
    "XOM":"ExxonMobil","CVX":"Chevron","COP":"ConocoPhillips","SLB":"SLB","EOG":"EOG Resources",
    "WMB":"Williams Companies","OXY":"Occidental Petroleum","PSX":"Phillips 66","NEE":"NextEra Energy",
    "SO":"Southern Company","DUK":"Duke Energy","CEG":"Constellation Energy","VST":"Vistra",
    "AEP":"American Electric Power","D":"Dominion Energy","LIN":"Linde","SHW":"Sherwin-Williams",
    "FCX":"Freeport-McMoRan","APD":"Air Products","NEM":"Newmont","DOW":"Dow Inc",
    "XLK":"Technology Sector","XLF":"Financials Sector","XLV":"Health Care Sector",
    "XLI":"Industrials Sector","XLY":"Consumer Discretionary Sector","XLP":"Consumer Staples Sector",
    "XLE":"Energy Sector","XLU":"Utilities Sector","XLB":"Materials Sector",
}

# ---------------------------------------------------------------------------
# Master ticker registry for the Chart tab -- every real symbol tracked
# anywhere in the dashboard, in one searchable list. "kind" tells the chart
# function how to fetch/scale it: price (divide by scale, e.g. cents->$),
# yield_yf (Yahoo bond index, already a direct % -- no scaling), or
# yield_fred (FRED series, %).
# ---------------------------------------------------------------------------
def _build_master_tickers():
    m = {}
    for name, (sym, scale, unit) in list(METALS.items()) + list(ENERGY.items()) + list(AGRI.items()):
        m[f"{name} ({sym}) {unit}"] = ("price", sym, scale)
    for name, (sym, prefix) in METALS_ETF.items():
        m[f"{name} ({sym})"] = ("price", sym, 1)
    for name, (sym, prefix) in AGRI_ETF.items():
        m[f"{name} ({sym})"] = ("price", sym, 1)
    for name, sym in CURRENCIES.items():
        # USD/EUR and USD/GBP are displayed inverted from Yahoo's own
        # EURUSD=X/GBPUSD=X series (see CURRENCIES_INVERTED) -- the Chart
        # tab plots the same inverted series for consistency with the
        # Currencies tab's cards, not the raw Yahoo quote.
        kind = "price_inverted" if name in CURRENCIES_INVERTED else "price"
        m[f"{name} ({sym})"] = (kind, sym, 1)
    for name, sym in CURRENCY_BASKET_ETF.items():
        m[f"{name} ({sym})"] = ("price", sym, 1)
    for name, sym in CRYPTO.items():
        m[f"{name} ({sym})"] = ("price", sym, 1)
    for name, sym in INDICES.items():
        m[f"{name} ({sym})"] = ("price", sym, 1)
    for name, sym in SECTOR_ETFS.items():
        m[f"{name} Sector ({sym})"] = ("price", sym, 1)
    for sym in ALL_STOCKS:
        m[f"{NAME_MAP.get(sym, sym)} ({sym})"] = ("price", sym, 1)
    m["Diesel Crack Spread (HO=F x42 - CL=F) $/bbl"] = ("diesel_crack", None, 1)
    m["WTI-Brent Spread (CL=F - BZ=F) $/bbl"] = ("wti_brent_spread", None, 1)
    m["Gold/Silver Ratio (GC=F / SI=F)"] = ("gold_silver_ratio", None, 1)
    m["Copper/Gold Ratio x1000 (HG=F / GC=F)"] = ("copper_gold_ratio", None, 1)
    for name, sym in SIGNALS_YF.items():
        m[f"{name} ({sym})"] = ("price", sym, 1)
    for name, sym in BONDS_US.items():
        m[f"{name} Yield ({sym})"] = ("yield_yf", sym, 1)
    # US 2-Year -- added 2026-09-18. Not in BONDS_US: Yahoo/CBOE's yield-index
    # family has no 2-year maturity, so this one comes from FRED (see
    # US_2Y_FRED_SERIES / render_bonds()'s docstring notes on this).
    m["US 2-Year Yield (FRED)"] = ("yield_fred", US_2Y_FRED_SERIES, 1)
    for name, series_id in BONDS_INTL.items():
        m[f"{name} Yield (FRED)"] = ("yield_fred", series_id, 1)
    for name, series_id in SIGNALS_FRED.items():
        m[f"{name} (FRED)"] = ("yield_fred", series_id, 1)
    return dict(sorted(m.items()))

MASTER_TICKERS = _build_master_tickers()


In [21]:
# ---------------------------------------------------------------------------
# Data fetching. Everything is computed from real Yahoo Finance / FRED
# history -- current price/level plus actual 1mo/1yr/5yr change, cached so
# toggling the horizon doesn't re-hit the network.
# ---------------------------------------------------------------------------

_CACHE = {}

def _yahoo_live_quote(symbol):
    """Direct call to the same public chart-meta endpoint Yahoo Finance's
    own site uses (query1.finance.yahoo.com/v8/finance/chart/<symbol>),
    read straight via `requests` rather than through yfinance's `fast_info`
    property wrapper.

    Added 2026-09-01 after the user reported Brent (BZ=F) showing a
    materially lower price than the real market, and -- as with the Bonds
    "Japan is wrong" report -- explicitly asked for a systemic fix rather
    than checking each instrument individually. Root-caused by directly
    querying this exact endpoint live: BZ=F's `regularMarketPrice` matched
    an independent source (TradingEconomics) to the cent, meaning Yahoo's
    own real-time data is fine -- the problem is specifically yfinance's
    `fast_info` wrapper, a documented, known-fragile summary layer that
    can lag or mis-map fields for some instruments (particularly
    continuous futures contracts like BZ=F/CL=F/GC=F etc.) without
    raising any error, so the notebook had no way to detect it silently
    serving a stale figure.

    Rather than special-case Brent alone, this bypasses `fast_info`
    entirely for EVERY symbol fetched anywhere in the notebook (stocks,
    indices, commodities, currencies, crypto), using the one path already
    directly verified correct. `fast_info` is kept only as an automatic
    fallback in `_fast_quote()` below, for whatever this direct call
    doesn't fill in (network hiccup, rate limit, or a field genuinely
    missing for a given symbol) -- never as the primary source anymore.

    Cache-busting (added 2026-09-01, same day): the user then reported
    Brent looking "stuck" -- unmoving across repeated refreshes -- right
    after this function shipped. Directly re-querying this exact endpoint
    live confirmed Yahoo's own data keeps moving over real elapsed time
    (BZ=F's own regularMarketTime/regularMarketPrice both advanced across
    two live checks), which rules out Yahoo's data itself being frozen --
    but a plain, unparameterized GET to the *same* URL on every refresh is
    a textbook trigger for an intermediate cache (a CDN edge, a corporate
    proxy, even some HTTP client/session layers) to keep serving back
    whatever it cached the first time, regardless of what Yahoo's origin
    server would return fresh. Every request now appends a throwaway,
    always-different query param (`_=<current time in ms>`) purely to
    make each request's URL unique, plus explicit no-cache headers -- a
    standard, low-risk technique for exactly this failure mode, applied
    once here so it covers every symbol in the notebook at once rather
    than needing to chase this down instrument by instrument.
    """
    out = {"last_price": None, "year_high": None, "year_low": None}
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                          "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
            "Cache-Control": "no-cache, no-store, must-revalidate",
            "Pragma": "no-cache",
        }
        resp = requests.get(
            f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}",
            params={"range": "1d", "interval": "1d", "_": str(int(time.time() * 1000))},
            headers=headers, timeout=10,
        )
        resp.raise_for_status()
        meta = resp.json()["chart"]["result"][0]["meta"]
        if meta.get("regularMarketPrice") is not None:
            out["last_price"] = float(meta["regularMarketPrice"])
        if meta.get("fiftyTwoWeekHigh") is not None:
            out["year_high"] = float(meta["fiftyTwoWeekHigh"])
        if meta.get("fiftyTwoWeekLow") is not None:
            out["year_low"] = float(meta["fiftyTwoWeekLow"])
    except Exception:
        pass
    return out


# ---------------------------------------------------------------------------
# Brent nearest-month contract (added 2026-09-22). User reported a Brent
# price "difference of 3-5" -- investigated live and confirmed NOT a bug:
# Yahoo's continuous `BZ=F` symbol had already rolled to the December '26
# contract (~$97.5) days before Investing.com/oilprice.com/Barchart's own
# headline "Brent" figures rolled, which were all still quoting the
# soon-to-expire November '26 contract (~$101.5-101.8) -- confirmed by
# cross-checking all three sites plus Yahoo's own specific-month ticker for
# November (`BZX26.NYM`) live, side by side, same moment. Both numbers were
# real and correct for their own contract -- just two different delivery
# months, ~$4 apart in a currently backwardated market. User then asked for
# the nearest-month contract specifically, since that's what most other
# sites default to showing.
#
# This adds that as its own card, alongside (not replacing) the existing
# continuous `BZ=F` reading above: BZ=F is what every other Brent-dependent
# feature in this notebook already relies on (WTI-Brent Spread, the Chart
# tab's history, beta/alpha/R²/volume/trend-persistence), all of which need
# long continuous history that one expiring contract-month ticker can't
# provide -- individual contracts typically only carry meaningful liquidity
# for a few months before expiry. This new card is a live SPOT READING
# ONLY -- no history, no beta/alpha/z-score/trend signals.
#
# Yahoo's specific-month futures tickers follow the standard industry
# month-code convention (F/G/H/J/K/M/N/Q/U/V/X/Z = Jan..Dec) plus a 2-digit
# year, e.g. "BZX26.NYM" = Brent, November 2026, NY Mercantile listing --
# confirmed live 2026-09-22 via Yahoo's own symbol-search endpoint, and
# cross-checked against Investing.com/oilprice.com/Barchart's own
# November-contract figures (all ~$101.5-101.8 at the same moment).
#
# WHICH month counts as "nearest" changes every few weeks as contracts
# expire -- hardcoding one ticker would silently go stale (this is exactly
# the fragility the WTI-Brent Spread note elsewhere in this notebook
# warned about when explaining why that spread uses continuous contracts
# instead). So this is computed fresh on every refresh instead of
# hardcoded: starting from next calendar month, it tries each successive
# month's ticker against Yahoo's live chart-meta endpoint and picks the
# first one that comes back with BOTH a real price AND a quote timestamp
# from the last 5 days -- an expired/delisted contract can still return a
# frozen last-traded price with a stale timestamp, which this filters out.
# This way it always tracks the genuine nearest actively-quoted contract
# without needing to hardcode ICE's exact expiry calendar (a
# business-day-before-the-15th-two-months-out rule that would be one more
# thing to get subtly wrong and never revisit).
# ---------------------------------------------------------------------------
_BRENT_MONTH_CODES = "FGHJKMNQUVXZ"  # index 0 = Jan ... 11 = Dec (standard futures month-code convention)
_BRENT_MONTH_ABBR = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

def _brent_contract_symbol(year, month):
    """Yahoo ticker for a specific Brent (ICE) contract month, e.g. (2026, 11) -> 'BZX26.NYM'."""
    code = _BRENT_MONTH_CODES[month - 1]
    yy = str(year % 100).zfill(2)
    return f"BZ{code}{yy}.NYM"

def _brent_contract_quote(symbol):
    """Like `_yahoo_live_quote()`, but also returns the quote timestamp and
    volume -- needed here to tell a genuinely live contract apart from an
    expired one that still returns a frozen last price. Same direct
    chart-meta endpoint, cache-busted the same way; kept deliberately
    separate from `_yahoo_live_quote()` rather than changing that shared,
    already-verified function's return shape for every other caller."""
    out = {"last_price": None, "quote_time": None, "volume": None}
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                          "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
            "Cache-Control": "no-cache, no-store, must-revalidate",
            "Pragma": "no-cache",
        }
        resp = requests.get(
            f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}",
            params={"range": "1d", "interval": "1d", "_": str(int(time.time() * 1000))},
            headers=headers, timeout=10,
        )
        resp.raise_for_status()
        meta = resp.json()["chart"]["result"][0]["meta"]
        if meta.get("regularMarketPrice") is not None:
            out["last_price"] = float(meta["regularMarketPrice"])
        if meta.get("regularMarketTime") is not None:
            out["quote_time"] = meta["regularMarketTime"]
        if meta.get("regularMarketVolume") is not None:
            out["volume"] = meta["regularMarketVolume"]
    except Exception:
        pass
    return out

def fetch_brent_nearest_month(max_tries=6):
    """Finds the nearest-expiry Brent contract that's still genuinely
    trading, live, on every refresh -- see the block comment above this
    function for the full reasoning. Returns a dict with the live price,
    a human label ("Nov '26"), the Yahoo symbol used, and an 'ok' flag.
    Cached in `_CACHE` like the Bonds tab's TradingEconomics scrapes --
    fetched lazily the first time the Energy tab renders after a refresh,
    not pre-fetched in `refresh_all()`, so toggling the horizon doesn't
    re-hit the network but this also isn't counted in the progress bar."""
    cache_key = "brent_nearest_month"
    if cache_key in _CACHE:
        return _CACHE[cache_key]
    now = datetime.utcnow()
    y, m = now.year, now.month
    out = {"ok": False, "price": None, "symbol": None, "label": None, "volume": None, "quote_time": None}
    for i in range(1, max_tries + 1):
        total_month = m + i
        yy = y + (total_month - 1) // 12
        mm = ((total_month - 1) % 12) + 1
        symbol = _brent_contract_symbol(yy, mm)
        q = _brent_contract_quote(symbol)
        if q["last_price"] is None or q["quote_time"] is None:
            continue
        age_days = (now - datetime.utcfromtimestamp(q["quote_time"])).total_seconds() / 86400.0
        if age_days > 5:
            continue  # stale/expired contract -- keep trying the next month out
        out = {"ok": True, "price": q["last_price"], "symbol": symbol,
               "label": f"{_BRENT_MONTH_ABBR[mm - 1]} '{str(yy)[2:]}",
               "volume": q.get("volume"), "quote_time": q["quote_time"]}
        break
    _CACHE[cache_key] = out
    return out


def _fast_quote(ticker):
    """Live quote snapshot fields -- last price plus 52-week high/low --
    used both to avoid history()'s bar-lag issue (see fetch_returns) and
    to power the 52-week-range signal.

    PRIMARY source (as of 2026-09-01): a direct call to Yahoo's own public
    chart-meta endpoint via `_yahoo_live_quote()` -- see that function's
    docstring for why (a real, confirmed-live bug: yfinance's `fast_info`
    property understated Brent's price; this direct endpoint didn't).
    FALLBACK: yfinance's `fast_info` property, used only to fill in
    whatever the direct call didn't return, so this degrades no worse
    than the old behavior if the direct endpoint is ever unreachable.
    Tries a couple of key spellings across yfinance versions for the
    fallback path; any field that can't be read from either source comes
    back None so callers degrade gracefully instead of crashing.

    Also returns `source` -- "direct" if the primary endpoint supplied the
    displayed price, "fast_info" if it had to fall back, "none" if neither
    worked. Added 2026-09-07 after a user reported Energy tab prices
    looking stale with no visible error: the fast_info fallback above is
    silent by design (so a card degrades gracefully instead of breaking),
    but that silence also meant there was no way to tell, from the
    dashboard itself, whether a given refresh actually used the verified
    direct feed or quietly slipped back to the older, known-fragile
    fast_info path (the same wrapper responsible for the original Brent
    bug) -- e.g. if the user's network blocks/throttles Yahoo's direct
    chart-meta endpoint specifically but not yfinance's own internal
    calls. fetch_returns() surfaces this as `price_source`, and
    fmt_signal_line() shows a visible flag on any card where it's not
    "direct", so a real fallback is now something the user can actually
    see and report instead of a silent, hard-to-diagnose staleness."""
    out = {"last_price": None, "year_high": None, "year_low": None, "source": "none"}
    symbol = getattr(ticker, "ticker", None) or getattr(ticker, "symbol", None)
    if symbol:
        direct = _yahoo_live_quote(symbol)
        for k, v in direct.items():
            if v is not None:
                out[k] = v
        if out["last_price"] is not None:
            out["source"] = "direct"
    if any(out[k] is None for k in ("last_price", "year_high", "year_low")):
        try:
            fi = ticker.fast_info
            field_map = {
                "last_price": ("last_price", "lastPrice"),
                "year_high": ("year_high", "yearHigh"),
                "year_low": ("year_low", "yearLow"),
            }
            for out_key, candidates in field_map.items():
                if out.get(out_key) is not None:
                    continue
                for key in candidates:
                    try:
                        v = fi[key]
                        if v is not None:
                            out[out_key] = float(v)
                            if out_key == "last_price":
                                out["source"] = "fast_info"
                            break
                    except Exception:
                        continue
        except Exception:
            pass
    return out


def _fast_last_price(ticker):
    """Backwards-compatible thin wrapper around _fast_quote -- just the
    live last price. See _fast_quote's docstring for why fast_info is
    preferred over history()'s last bar."""
    return _fast_quote(ticker)["last_price"]


def _zscore(closes, last_price, lookback_days=365, min_points=20):
    """How many standard deviations the current price/yield/level sits from
    its own trailing mean over `lookback_days` -- a simple, transparent
    mean-reversion signal. Requires at least `min_points` data points in
    the window, otherwise returns None rather than a noisy estimate from
    too little data. `min_points` is lowered for low-frequency series
    (e.g. quarterly company filings) where 20 daily-style points isn't a
    realistic bar."""
    if closes is None or closes.empty or last_price is None:
        return None
    cutoff = closes.index[-1] - pd.Timedelta(days=lookback_days)
    window = closes[closes.index >= cutoff]
    if len(window) < min_points:
        return None
    std = window.std()
    if std == 0 or pd.isna(std):
        return None
    return (last_price - window.mean()) / std


_CLOSES_CACHE = {}
_VOLUME_CACHE = {}

def _get_closes(symbol, period="6y"):
    """Fetches (and caches) a symbol's raw historical close-price series --
    shared by fetch_returns() and any derived/computed metric (e.g. the
    diesel crack spread) that needs two tickers' full history aligned by
    date, so a derived metric never triggers a second, redundant network
    call for a leg that's already being fetched elsewhere on the same tab.

    Retries on empty/failed responses: Yahoo's API occasionally returns an
    empty response for a well-established, actively-traded symbol (e.g.
    CL=F) due to a transient rate-limit or connection hiccup, not an actual
    delisting -- yfinance's own "No data found, symbol may be delisted"
    message is misleading in that specific case. Originally this function
    made exactly two back-to-back attempts (period, then "max") with no
    delay between them, which doesn't give a genuinely transient hiccup any
    real chance to clear -- both attempts land in the same instant and fail
    for the same underlying reason. Now retries the same period once more
    after a short delay before falling back to "max", and wraps every
    attempt in try/except so a raised connection error (not just an empty
    result) is retried too, rather than propagating uncaught to whichever
    function called this.

    Final fallback to "5d": confirmed live (2026-08-31, direct query
    against Yahoo's own chart API) that some thin/less-common symbols --
    e.g. CNH=X (USD/CNH) -- have a genuinely, permanently restricted valid
    range on Yahoo's backend (CNH=X's own API response: validRanges =
    ["1d","5d"], firstTradeDate = null, volume = 0). Every longer period,
    including "max", is rejected for a symbol like that -- not a rate
    limit or a transient hiccup, Yahoo simply doesn't carry more history
    for it. Trying "5d" as a last resort after "max" fails means such a
    symbol can still show a live price and a real 1-day change instead of
    "n/a" everywhere; 1mo/1yr/5yr legitimately stay "n/a" for it since no
    more history exists to compute them from."""
    if symbol in _CLOSES_CACHE:
        return _CLOSES_CACHE[symbol]
    closes = pd.Series(dtype=float)
    attempts = [period, period, "max", "5d"]
    for attempt, this_period in enumerate(attempts):
        try:
            ticker = yf.Ticker(symbol)
            hist = ticker.history(period=this_period, auto_adjust=True)
            if not hist.empty:
                closes = hist["Close"].dropna()
                # Capture Volume alongside Close from this SAME history()
                # call -- added 2026-09-08 at the user's request, no extra
                # network call needed since it's already in the DataFrame
                # this function fetches. See _get_volume()'s docstring for
                # which instrument types genuinely have none (FX, yields).
                if "Volume" in hist.columns:
                    _VOLUME_CACHE[symbol] = hist["Volume"].reindex(closes.index)
                else:
                    _VOLUME_CACHE[symbol] = pd.Series(dtype=float)
                break
        except Exception:
            pass
        if attempt < len(attempts) - 1:
            time.sleep(1.5)
    _CLOSES_CACHE[symbol] = closes
    return closes


def _get_volume(symbol, period="6y"):
    """Daily trading volume for `symbol` -- captured as a side effect of
    _get_closes()'s already-fetched history() call above, so calling this
    never triggers its own network request. Populated the first time
    _get_closes(symbol, ...) runs, whether called directly or indirectly
    (fetch_returns()/fetch_returns_inverted() already call it for every
    tracked instrument), so this calls _get_closes() itself first to
    guarantee that's happened.

    Genuinely, permanently empty for some instrument types -- not a fetch
    failure: FX pairs (EURUSD=X, GBPUSD=X, JPY=X, CHF=X, CNH=X, EURCHF=X
    -- spot currency trading is OTC/interbank, so there's no centralized
    tape for Yahoo to report volume from; confirmed 0 directly against
    the raw chart API earlier this session for CNH=X specifically) and
    Treasury/bond-yield tickers (^IRX/^FVX/^TNX/^TYX -- those go through
    fetch_yield(), a separate path, and a yield reading isn't a traded
    security with its own volume regardless). A handful of thinly-tracked
    foreign ETF listings can also come back empty depending on what
    Yahoo carries for that specific exchange -- also real, not hidden."""
    if symbol not in _VOLUME_CACHE:
        _get_closes(symbol, period)
    return _VOLUME_CACHE.get(symbol, pd.Series(dtype=float))


def _volume_stats(volume_series, lookback_days=30):
    """Latest daily volume and a trailing-30-day average daily volume,
    from the same series _get_volume() already has in hand -- no extra
    computation beyond a slice and a mean. Returns (None, None) when the
    series is empty OR every value in it is exactly 0 -- the genuine,
    permanent no-volume-data case (FX pairs, yield tickers) described in
    _get_volume()'s docstring, distinguished from an ordinary single
    holiday's zero-volume bar sitting inside an otherwise-active series
    (that case still has other days > 0, so it proceeds normally and that
    one zero day just pulls the 30-day average down a little, which is
    the correct, real behavior, not a gap)."""
    if volume_series is None or volume_series.empty:
        return None, None
    v = volume_series.dropna()
    if v.empty or float(v.max()) <= 0:
        return None, None
    latest = float(v.iloc[-1])
    cutoff = v.index[-1] - pd.Timedelta(days=lookback_days)
    window = v[v.index >= cutoff]
    avg_30d = float(window.mean()) if not window.empty else None
    return latest, avg_30d


def _beta_alpha_r2(closes, benchmark_closes, lookback_days=365, min_points=60):
    """Beta, (simplified, annualized) alpha, and R² vs the S&P 500
    (^GSPC), from trailing daily returns over `lookback_days` -- a fixed
    ~1-year window, independent of the horizon toggle, the same
    convention already used for the z-score and 52-week range signals on
    this line. Added 2026-09-07 at the user's request to add these across
    every tracked instrument; R² added the same day after a follow-up
    question about how much of a stock's move beta actually explains.

    Beta = Cov(instrument's daily return, benchmark's daily return) /
    Var(benchmark's daily return) -- how much this instrument tends to
    move for each 1% move in the S&P 500 (>1 = more volatile than the
    market, <1 = less, negative = tends to move opposite the market, e.g.
    VIX). Beta is a SLOPE, not a measure of fit -- it says nothing on its
    own about how much of the instrument's day-to-day variance the market
    actually explains.

    R² = the squared correlation between the instrument's daily returns
    and the benchmark's over the same window (equivalent to the
    coefficient of determination from the same simple regression beta
    comes from) -- the piece beta alone can't tell you: what fraction of
    this instrument's own return variance the market's moves explain, as
    opposed to idiosyncratic/company-specific noise. A stock can have a
    large beta but a low R² (moves are amplified when they do track the
    market, but are mostly untethered from it day to day -- e.g. driven by
    its own earnings/news), or a small beta with a high R² (moves closely
    with the market, just less amplified). A large beta paired with a low
    R² is exactly the case where that beta is least reliable as a
    day-to-day predictor, even though it's still a correctly-computed
    historical average.

    Alpha is a simplified Jensen's alpha: the instrument's own average
    daily return minus what its beta alone would predict from the
    benchmark's average daily return, annualized (x252 trading days,
    shown as a %). Simplified because it skips subtracting the risk-free
    rate (textbook CAPM regresses *excess* returns over T-bills) -- for a
    dashboard-level "is this beating what its market exposure alone would
    explain" signal, the risk-free rate is small next to typical return
    noise, and skipping it means this needs no extra data source beyond
    the price history already being fetched. Disclosed as a simplification,
    not textbook-precise Jensen's alpha.

    Benchmark dates are forward-filled onto the instrument's own calendar
    (same alignment approach as the WTI-Brent spread elsewhere in this
    file) since some instruments trade on days the S&P 500 doesn't
    (weekends for crypto/FX) -- those days get the benchmark's last known
    return. Needs at least `min_points` overlapping daily returns in the
    window, otherwise returns (None, None, None) rather than a noisy
    regression from too little data. All three are computed from the same
    aligned return series in one pass rather than three separate ones."""
    if closes is None or closes.empty or benchmark_closes is None or benchmark_closes.empty:
        return None, None, None
    cutoff = closes.index[-1] - pd.Timedelta(days=lookback_days)
    inst = closes[closes.index >= cutoff]
    if len(inst) < min_points:
        return None, None, None
    bench = benchmark_closes.reindex(inst.index.union(benchmark_closes.index)).sort_index().ffill().reindex(inst.index)
    inst_ret = inst.pct_change().dropna()
    bench_ret = bench.pct_change().dropna()
    common = inst_ret.index.intersection(bench_ret.index)
    if len(common) < min_points:
        return None, None, None
    inst_ret = inst_ret.loc[common]
    bench_ret = bench_ret.loc[common]
    bench_var = bench_ret.var()
    if not bench_var or pd.isna(bench_var):
        return None, None, None
    beta = inst_ret.cov(bench_ret) / bench_var
    if pd.isna(beta):
        return None, None, None
    alpha_annualized_pct = (inst_ret.mean() - beta * bench_ret.mean()) * 252 * 100.0
    if pd.isna(alpha_annualized_pct):
        return None, None, None
    corr = inst_ret.corr(bench_ret)
    r_squared = (corr ** 2) if not pd.isna(corr) else None
    return float(beta), float(alpha_annualized_pct), (float(r_squared) if r_squared is not None else None)


def _trend_persistence(closes, lookback_days=90, min_points=40):
    """Lag-1 autocorrelation of daily returns over a trailing ~90-day
    window -- a rough proxy for whether the current price action is in a
    self-reinforcing ("trending") regime or a mean-reverting one. Added
    2026-09-13 after a conversation about Soros's reflexivity theory:
    reflexivity describes a two-way feedback loop between market
    perception and underlying fundamentals that can produce sustained,
    self-reinforcing moves (rather than the efficient-market assumption
    that returns are independent/random from one day to the next) --
    there's no agreed formula for "reflexivity" itself, so this is a
    deliberately narrow, disclosed proxy for one piece of it (trend
    persistence in the return series), not a measurement of the theory.
    It does NOT indicate the direction, size, or timing of any future
    move -- see the intro markdown's disclosure for exactly what this
    can and can't tell you.

    Positive autocorrelation means a day's return has recently tended to
    be followed by another return in the same direction (momentum/
    trending -- the kind of behavior reflexivity describes while a
    feedback loop is actively running). Negative autocorrelation means
    moves have recently tended to reverse day to day (mean-reverting,
    closer to the classical efficient-market pattern Soros argued most
    markets aren't actually in most of the time). Near zero means
    neither pattern dominates recently.

    Uses a shorter ~90-day window rather than the ~1-year window beta/
    alpha/R² use, since a "regime" reading is meant to describe recent
    behavior -- a full year would blend together whatever trending and
    mean-reverting stretches happened to occur within it. `min_points=40`
    keeps the ~1/sqrt(N) noise floor for a correlation coefficient at
    this sample size around ~0.16 -- the display threshold in
    fmt_signal_line() (+/-0.15) is set just under that, i.e. close to
    the edge of what's actually distinguishable from a purely random
    return series here, not an arbitrary round number.

    Needs only the instrument's own price series -- no benchmark, unlike
    beta/alpha/R² -- since this is a property of the series' own day-to-
    day dependence, not a relationship to the S&P 500. Returns None when
    there isn't enough trailing history (e.g. a very new listing) rather
    than reporting a noisy reading from too few points."""
    if closes is None or closes.empty:
        return None
    cutoff = closes.index[-1] - pd.Timedelta(days=lookback_days)
    window = closes[closes.index >= cutoff]
    if len(window) < min_points:
        return None
    rets = window.pct_change().dropna()
    if len(rets) < min_points - 1:
        return None
    autocorr = rets.autocorr(lag=1)
    if pd.isna(autocorr):
        return None
    return float(autocorr)


def _compute_return_stats(closes, last_price, year_high, year_low, benchmark_closes=None,
                           volume_series=None):
    """Shared statistics engine used by both fetch_returns() and
    fetch_returns_inverted(): given a closes series (already inverted, for
    the latter) and live quote fields, computes % change over each
    horizon, z-score, 52-week range position, MA trend/crossover,
    (when `benchmark_closes` is supplied) beta/alpha/R² vs the S&P 500,
    and (when `volume_series` is supplied) latest-day and 30-day-average
    trading volume. Kept as one function so an inverted pair (e.g.
    USD/EUR, built from EURUSD=X) gets every derived signal recomputed
    directly on its own real series rather than algebraically guessed at
    from the original -- a z-score, MA crossover, or beta doesn't simply
    flip sign or timing under 1/x, it has to be computed on the actual
    reciprocal data. Volume itself is never inverted (it isn't a price --
    see fetch_returns_inverted())."""
    out = {"price": last_price, "d": None, "s": None, "m": None, "l": None,
           "zscore": None, "range_pct": None, "year_high": year_high, "year_low": year_low,
           "ma_trend": None, "ma_cross_recent": False, "above_200dma": None,
           "beta": None, "alpha": None, "r_squared": None, "autocorr_1": None,
           "volume_1d": None, "volume_30d_avg": None}
    if volume_series is not None:
        out["volume_1d"], out["volume_30d_avg"] = _volume_stats(volume_series)
    if closes.empty or last_price is None:
        return out
    last_date = closes.index[-1]

    def pct_change_since(days):
        target = last_date - pd.Timedelta(days=days)
        sub = closes[closes.index <= target]
        if sub.empty:
            return None
        base = float(sub.iloc[-1])
        if base == 0:
            return None
        return (last_price - base) / base * 100.0

    out["d"] = pct_change_since(1)
    out["s"] = pct_change_since(30)
    out["m"] = pct_change_since(365)
    out["l"] = pct_change_since(365 * 5)

    # Mean-reversion signal: z-score of current price vs its own
    # trailing-year mean/stdev, computed from the same history bars
    # already fetched above -- no extra network call.
    out["zscore"] = _zscore(closes, last_price)

    # 52-week range position: prefer fast_info's year high/low: if
    # unavailable, fall back to computing it directly from the history
    # already in hand.
    year_high, year_low = out["year_high"], out["year_low"]
    if year_high is None or year_low is None:
        cutoff = last_date - pd.Timedelta(days=365)
        window = closes[closes.index >= cutoff]
        if not window.empty:
            if year_high is None:
                year_high = float(window.max())
            if year_low is None:
                year_low = float(window.min())
    out["year_high"] = year_high
    out["year_low"] = year_low
    if year_high is not None and year_low is not None and year_high > year_low:
        out["range_pct"] = (last_price - year_low) / (year_high - year_low) * 100.0

    # 50/200-day moving average trend + crossover-recency, and simple
    # "above its own 200-day average" breadth flag -- all computed from
    # the same history bars already fetched above, no extra network
    # call. Needs a reasonable amount of history to be meaningful, so
    # this silently stays None for thin/short series (e.g. very new
    # listings) rather than reporting a noisy signal.
    if len(closes) >= 210:
        ma50 = closes.rolling(50).mean()
        ma200 = closes.rolling(200).mean()
        if not pd.isna(ma50.iloc[-1]) and not pd.isna(ma200.iloc[-1]):
            out["ma_trend"] = "bullish" if ma50.iloc[-1] > ma200.iloc[-1] else "bearish"
            out["above_200dma"] = bool(last_price > ma200.iloc[-1])
            # "recent" crossover = the 50/200 relationship flipped sign
            # at some point in the last 10 trading days -- flags a fresh
            # golden/death cross rather than a long-standing trend.
            recent_diff = (ma50 - ma200).iloc[-11:].dropna()
            if len(recent_diff) >= 2:
                signs = (recent_diff > 0).astype(int)
                out["ma_cross_recent"] = bool(signs.diff().abs().sum() >= 1)

    # Beta/alpha/R² vs the S&P 500 -- see _beta_alpha_r2()'s docstring.
    # Skipped (stays None) when the caller has no benchmark series to
    # compare against, e.g. a synthetic derived spread/ratio rather than a
    # single priced instrument -- see render notes on which cards show this.
    if benchmark_closes is not None:
        out["beta"], out["alpha"], out["r_squared"] = _beta_alpha_r2(closes, benchmark_closes)

    # Trend-persistence / reflexivity-regime proxy -- see _trend_persistence()'s
    # docstring. Needs only the instrument's own price series (no benchmark),
    # so this is computed whenever there's a usable `closes` series, unlike
    # beta/alpha/R² which also require benchmark_closes.
    out["autocorr_1"] = _trend_persistence(closes)

    return out


def fetch_returns(symbol, period="6y"):
    if symbol in _CACHE:
        return _CACHE[symbol]
    out = {"price": None, "d": None, "s": None, "m": None, "l": None, "ok": False, "error": None,
           "zscore": None, "range_pct": None, "year_high": None, "year_low": None,
           "ma_trend": None, "ma_cross_recent": False, "above_200dma": None, "price_source": None,
           "beta": None, "alpha": None, "r_squared": None, "autocorr_1": None,
           "volume_1d": None, "volume_30d_avg": None}
    try:
        ticker = yf.Ticker(symbol)
        closes = _get_closes(symbol, period)
        if closes.empty:
            out["error"] = "no data"
            _CACHE[symbol] = out
            return out

        # Prefer the live quote snapshot for the *displayed* price; only
        # fall back to the last history bar if fast_info isn't available.
        quote = _fast_quote(ticker)
        if quote["last_price"] is not None:
            last_price = quote["last_price"]
            price_source = quote["source"]
        else:
            last_price = float(closes.iloc[-1])
            price_source = "history_bar"  # worse than either live-quote path -- see fmt_signal_line()

        # Benchmark for beta/alpha (see _beta_alpha_r2()) -- ^GSPC's own
        # closes are already fetched/cached the same way as any other
        # symbol here, so this is free after the first call each session.
        # ^GSPC vs itself is a deliberate, harmless sanity check (comes out
        # beta=1.0, alpha=0.0), not special-cased away.
        benchmark_closes = _get_closes("^GSPC", period)
        # Volume (see _get_volume()) -- also free, already captured as a
        # side effect of the _get_closes() call above.
        volume_series = _get_volume(symbol, period)
        out = _compute_return_stats(closes, last_price, quote["year_high"], quote["year_low"],
                                     benchmark_closes, volume_series)
        out["ok"] = True
        out["error"] = None
        out["price_source"] = price_source
    except Exception as e:
        out["error"] = str(e)
    _CACHE[symbol] = out
    return out


def fetch_returns_inverted(symbol, period="6y"):
    """Same live-quote + statistics pipeline as fetch_returns(), but on the
    reciprocal (1/x) of the price series -- used to display a Yahoo pair
    like EURUSD=X (USD per 1 EUR) as USD/EUR (EUR per 1 USD) instead, at
    the user's request that every row on the Currencies tab read
    USD-first. Cached separately from the non-inverted fetch_returns(sym)
    entry for the same symbol (different cache key) since they're
    genuinely different series with their own independent stats -- not
    just a relabeled version of each other."""
    cache_key = f"{symbol}::inverted"
    if cache_key in _CACHE:
        return _CACHE[cache_key]
    out = {"price": None, "d": None, "s": None, "m": None, "l": None, "ok": False, "error": None,
           "zscore": None, "range_pct": None, "year_high": None, "year_low": None,
           "ma_trend": None, "ma_cross_recent": False, "above_200dma": None, "price_source": None,
           "beta": None, "alpha": None, "r_squared": None, "autocorr_1": None,
           "volume_1d": None, "volume_30d_avg": None}
    try:
        ticker = yf.Ticker(symbol)
        closes = _get_closes(symbol, period)
        if closes.empty:
            out["error"] = "no data"
            _CACHE[cache_key] = out
            return out
        inv_closes = (1.0 / closes.replace(0, pd.NA)).dropna()

        quote = _fast_quote(ticker)
        if quote["last_price"] is not None:
            last_price = quote["last_price"]
            price_source = quote["source"]
        else:
            last_price = float(closes.iloc[-1])
            price_source = "history_bar"
        inv_last_price = (1.0 / last_price) if last_price else None
        # High/low swap AND invert under reciprocal: 1/(old low) = new high.
        inv_year_high = (1.0 / quote["year_low"]) if quote["year_low"] else None
        inv_year_low  = (1.0 / quote["year_high"]) if quote["year_high"] else None

        # Beta/alpha computed on the actual inverted series (USD/EUR's own
        # returns), not the original EURUSD=X -- consistent with z-score/
        # MA trend above; the benchmark itself (^GSPC) is never inverted.
        # Volume also isn't inverted (not a price) -- expected to come back
        # empty here anyway, since Yahoo reports no volume for FX pairs.
        benchmark_closes = _get_closes("^GSPC", period)
        volume_series = _get_volume(symbol, period)
        out = _compute_return_stats(inv_closes, inv_last_price, inv_year_high, inv_year_low,
                                     benchmark_closes, volume_series)
        out["ok"] = True
        out["error"] = None
        out["price_source"] = price_source
    except Exception as e:
        out["error"] = str(e)
    _CACHE[cache_key] = out
    return out


def fetch_diesel_crack_spread():
    """1:1 diesel (ULSD) crack spread, in $/bbl -- the standard refining-
    margin signal: (Heating Oil futures price, $/gal, x 42 gal/bbl) minus
    WTI crude price ($/bbl). CME/NYMEX's Heating Oil contract (HO=F) has
    tracked ultra-low-sulfur diesel (ULSD) since a 2013 spec change, so
    it's used here as diesel's proxy futures contract -- the same
    substitution already made for the "Heating Oil / Diesel proxy" card on
    this tab. A rising spread means refiners are earning more per barrel to
    turn crude into diesel (often a sign of tight diesel supply or strong
    demand -- worth watching ahead of winter heating season and during
    trucking-demand cycles); a falling or negative spread means diesel
    refining is squeezed or unprofitable at the margin.

    Reuses the SAME historical closes already fetched for the HO=F and CL=F
    cards on this tab (via _get_closes()'s shared cache, and fetch_returns()
    for the live quote) -- no extra network calls. % change and z-score are
    computed on the spread's OWN historical series, not derived from either
    leg's separate % change, since the spread's own volatility -- not
    either underlying commodity's -- is what a refining-margin signal
    should be measured against."""
    cache_key = "diesel_crack_spread"
    if cache_key in _CACHE:
        return _CACHE[cache_key]
    out = {"value": None, "d": None, "s": None, "m": None, "l": None, "ok": False, "error": None,
           "zscore": None, "range_pct": None}
    try:
        ho_d = fetch_returns("HO=F")
        cl_d = fetch_returns("CL=F")
        if not ho_d["ok"] or not cl_d["ok"] or ho_d["price"] is None or cl_d["price"] is None:
            out["error"] = "underlying HO=F/CL=F fetch failed"
            _CACHE[cache_key] = out
            return out
        ho_closes = _get_closes("HO=F")
        cl_closes = _get_closes("CL=F")
        if ho_closes.empty or cl_closes.empty:
            out["error"] = "no data"
            _CACHE[cache_key] = out
            return out
        cl_aligned = cl_closes.reindex(ho_closes.index.union(cl_closes.index)).sort_index().ffill().reindex(ho_closes.index)
        spread = (ho_closes * 42.0 - cl_aligned).dropna()
        if spread.empty:
            out["error"] = "no overlapping data after alignment"
            _CACHE[cache_key] = out
            return out
        last_val = ho_d["price"] * 42.0 - cl_d["price"]
        last_date = spread.index[-1]

        def chg_since(days):
            target = last_date - pd.Timedelta(days=days)
            sub = spread[spread.index <= target]
            if sub.empty:
                return None
            base = float(sub.iloc[-1])
            if base == 0:
                return None
            return (last_val - base) / abs(base) * 100.0  # abs() since the spread can go negative

        out["value"] = last_val
        out["d"] = chg_since(1)
        out["s"] = chg_since(30)
        out["m"] = chg_since(365)
        out["l"] = chg_since(365 * 5)
        out["zscore"] = _zscore(spread, last_val)

        cutoff = last_date - pd.Timedelta(days=365)
        window = spread[spread.index >= cutoff]
        if not window.empty:
            yr_high, yr_low = float(window.max()), float(window.min())
            if yr_high > yr_low:
                out["range_pct"] = (last_val - yr_low) / (yr_high - yr_low) * 100.0
        out["ok"] = True
    except Exception as e:
        out["error"] = str(e)
    _CACHE[cache_key] = out
    return out


def fetch_price_ratio(num_symbol, den_symbol, multiplier=1.0, zscore_lookback_days=365, zscore_min_points=20):
    """Generic ratio of two already-tracked yfinance price series (Gold/
    Silver, Copper/Gold) -- reuses fetch_returns()'s live price and
    _get_closes()'s shared history cache, so this triggers NO extra network
    calls beyond what the tab's own cards already fetch. `multiplier`
    rescales the ratio for readability (e.g. Copper $/lb over Gold $/oz is
    a tiny number, so x1000 is the conventional way this ratio is quoted).
    % change and z-score are computed on the ratio's own historical series,
    matching fetch_fred_ratio()'s convention for the same reason: a ratio's
    own volatility is what a relative-value signal should be measured
    against, not either leg's separate volatility."""
    cache_key = f"{num_symbol}::{den_symbol}::price_ratio"
    if cache_key in _CACHE:
        return _CACHE[cache_key]
    out = {"value": None, "d": None, "s": None, "m": None, "l": None, "ok": False, "error": None,
           "zscore": None, "range_pct": None}
    try:
        num_d, den_d = fetch_returns(num_symbol), fetch_returns(den_symbol)
        if not num_d["ok"] or not den_d["ok"] or num_d["price"] is None or den_d["price"] is None:
            out["error"] = "underlying fetch failed"
            _CACHE[cache_key] = out
            return out
        num_c, den_c = _get_closes(num_symbol), _get_closes(den_symbol)
        if num_c.empty or den_c.empty:
            out["error"] = "no data"
            _CACHE[cache_key] = out
            return out
        den_aligned = den_c.reindex(num_c.index.union(den_c.index)).sort_index().ffill().reindex(num_c.index)
        ratio = (num_c / den_aligned * multiplier).dropna()
        if ratio.empty:
            out["error"] = "no overlapping data after alignment"
            _CACHE[cache_key] = out
            return out
        last_val = num_d["price"] / den_d["price"] * multiplier
        last_date = ratio.index[-1]

        def chg_since(days):
            target = last_date - pd.Timedelta(days=days)
            sub = ratio[ratio.index <= target]
            if sub.empty:
                return None
            base = float(sub.iloc[-1])
            if base == 0:
                return None
            return (last_val - base) / base * 100.0

        out["value"] = last_val
        out["d"] = chg_since(1)
        out["s"] = chg_since(30)
        out["m"] = chg_since(365)
        out["l"] = chg_since(365 * 5)
        out["zscore"] = _zscore(ratio, last_val, lookback_days=zscore_lookback_days, min_points=zscore_min_points)
        cutoff = last_date - pd.Timedelta(days=365)
        window = ratio[ratio.index >= cutoff]
        if not window.empty:
            yr_high, yr_low = float(window.max()), float(window.min())
            if yr_high > yr_low:
                out["range_pct"] = (last_val - yr_low) / (yr_high - yr_low) * 100.0
        out["ok"] = True
    except Exception as e:
        out["error"] = str(e)
    _CACHE[cache_key] = out
    return out


def fetch_price_spread(leg1_symbol, leg1_mult, leg2_symbol, leg2_mult,
                        zscore_lookback_days=365, zscore_min_points=20):
    """Generic spread (leg1*mult1 - leg2*mult2) of two already-tracked
    yfinance price series (WTI-Brent) -- same shared-cache/no-extra-
    network-call pattern as fetch_price_ratio(), just subtraction instead
    of division."""
    cache_key = f"{leg1_symbol}::{leg2_symbol}::price_spread"
    if cache_key in _CACHE:
        return _CACHE[cache_key]
    out = {"value": None, "d": None, "s": None, "m": None, "l": None, "ok": False, "error": None,
           "zscore": None, "range_pct": None}
    try:
        d1, d2 = fetch_returns(leg1_symbol), fetch_returns(leg2_symbol)
        if not d1["ok"] or not d2["ok"] or d1["price"] is None or d2["price"] is None:
            out["error"] = "underlying fetch failed"
            _CACHE[cache_key] = out
            return out
        c1, c2 = _get_closes(leg1_symbol), _get_closes(leg2_symbol)
        if c1.empty or c2.empty:
            out["error"] = "no data"
            _CACHE[cache_key] = out
            return out
        c2_aligned = c2.reindex(c1.index.union(c2.index)).sort_index().ffill().reindex(c1.index)
        spread = (c1 * leg1_mult - c2_aligned * leg2_mult).dropna()
        if spread.empty:
            out["error"] = "no overlapping data after alignment"
            _CACHE[cache_key] = out
            return out
        last_val = d1["price"] * leg1_mult - d2["price"] * leg2_mult
        last_date = spread.index[-1]

        def chg_since(days):
            target = last_date - pd.Timedelta(days=days)
            sub = spread[spread.index <= target]
            if sub.empty:
                return None
            base = float(sub.iloc[-1])
            if base == 0:
                return None
            return (last_val - base) / abs(base) * 100.0

        out["value"] = last_val
        out["d"] = chg_since(1)
        out["s"] = chg_since(30)
        out["m"] = chg_since(365)
        out["l"] = chg_since(365 * 5)
        out["zscore"] = _zscore(spread, last_val, lookback_days=zscore_lookback_days, min_points=zscore_min_points)
        cutoff = last_date - pd.Timedelta(days=365)
        window = spread[spread.index >= cutoff]
        if not window.empty:
            yr_high, yr_low = float(window.max()), float(window.min())
            if yr_high > yr_low:
                out["range_pct"] = (last_val - yr_low) / (yr_high - yr_low) * 100.0
        out["ok"] = True
    except Exception as e:
        out["error"] = str(e)
    _CACHE[cache_key] = out
    return out


# ---------------------------------------------------------------------------
# Liquidity Stress gauge (Amihud illiquidity ratio, cross-sectional) --
# added 2026-09-22 at the user's request, after a discussion of whether the
# academic Pastor-Stambaugh liquidity factor could power a dashboard signal.
# Conclusion from that discussion, worth repeating here: PS's own measure
# regresses next-day return reversal on the PRIOR day's signed dollar
# volume (was trading buyer- or seller-initiated), which needs intraday
# trade-sign data -- not available for free, and not something yfinance's
# daily bars can provide. The user's own research separately found that
# even the real PS factor is trailing/coincident with liquidity conditions
# around a recession or boom, not a leading indicator of one.
#
# This builds the well-known, much simpler COUSIN of PS instead: Amihud's
# (2002) illiquidity ratio, |daily return| / dollar volume -- same
# underlying economic idea (price impact of trading = a market-
# microstructure view of liquidity) but needs only daily close and volume,
# both already cached for every tracked stock via _get_closes()/
# _get_volume() (the volume feature shipped 2026-09-08) -- so this adds NO
# new network calls. Explicitly disclosed as Amihud's specific measure, not
# a reproduction of "the Pastor-Stambaugh factor" -- a real but honest
# substitution, the same way the WTI-Brent Spread is disclosed as a
# substitute for true futures term structure elsewhere in this notebook.
#
# TRAILING BY CONSTRUCTION, same as the real PS measure, and surfaced on
# this tab (not the faster Signals tab) for exactly that reason: a reading
# here confirms/sizes a liquidity-stress regime already suspected from
# faster gauges (VIX, HY OAS on the Signals tab), it does not get ahead of
# one. Rising = the market is currently paying more in price impact to
# trade (liquidity thinning); falling = liquidity healthy.
# ---------------------------------------------------------------------------
def fetch_liquidity_stress(lookback_days=365, min_stocks_per_day=20, history_buffer_days=45):
    """Cross-sectional Amihud illiquidity gauge across ALL_STOCKS -- see the
    block comment above this function for the full methodology and the
    disclosed distinction from the actual Pastor-Stambaugh liquidity factor.

    For each tracked stock: daily |return| / (close x volume), scaled by
    1e6 (Amihud's own convention, purely for readability -- otherwise these
    are vanishingly small numbers for large, heavily-traded names). Days
    with zero volume are excluded rather than producing a division-by-zero
    infinity. Each stock's own daily series is then aggregated cross-
    sectionally by taking the MEDIAN across all stocks with a valid reading
    that day (not the mean -- a handful of thinly-traded names can otherwise
    swamp a straight average with outsized ratios), producing one daily
    market-wide series. A day is only kept if at least `min_stocks_per_day`
    stocks contributed a reading that day, so the early/sparse tail of the
    combined history doesn't produce a noisy, thin aggregate.

    That aggregate daily series is then treated exactly like any other
    derived series in this notebook (WTI-Brent Spread, Gold/Silver Ratio):
    % change over d/s/m/l is computed on the aggregate's OWN history, and
    z-score / 52wk-range are computed the same way too, via the same
    `_zscore()` helper used everywhere else -- so this integrates with
    `fmt_signal_line()` and the card-frame coloring with no special-casing.

    Cached in `_CACHE` like every other derived series -- computed once per
    refresh (first time the Cash & Liquidity tab renders), not pre-fetched
    in `refresh_all()`, matching the Bonds tab's TradingEconomics-scrape
    caching pattern."""
    cache_key = "liquidity_stress_amihud"
    if cache_key in _CACHE:
        return _CACHE[cache_key]
    out = {"value": None, "d": None, "s": None, "m": None, "l": None, "ok": False, "error": None,
           "zscore": None, "range_pct": None, "n_stocks": None, "as_of": None}
    try:
        daily_frames = {}
        for sym in ALL_STOCKS:
            closes = _get_closes(sym)
            vol = _get_volume(sym)
            if closes is None or closes.empty or vol is None or vol.empty:
                continue
            df = pd.DataFrame({"close": closes, "volume": vol}).dropna()
            if len(df) < 30:
                continue
            ret = df["close"].pct_change()
            dollar_vol = df["close"] * df["volume"]
            dollar_vol = dollar_vol.where(dollar_vol > 0)  # zero-volume days -> NaN, not a division-by-zero inf
            illiq = (ret.abs() / dollar_vol) * 1_000_000.0
            illiq = illiq.dropna()
            if illiq.empty:
                continue
            cutoff = illiq.index.max() - pd.Timedelta(days=lookback_days + history_buffer_days)
            illiq = illiq[illiq.index >= cutoff]
            if not illiq.empty:
                daily_frames[sym] = illiq
        if len(daily_frames) < min_stocks_per_day:
            out["error"] = f"only {len(daily_frames)} stocks had usable price/volume history (need {min_stocks_per_day}+)"
            _CACHE[cache_key] = out
            return out
        panel = pd.DataFrame(daily_frames).sort_index()
        counts = panel.count(axis=1)
        agg = panel.median(axis=1, skipna=True)
        agg = agg[counts >= min_stocks_per_day].dropna().sort_index()
        if agg.empty:
            out["error"] = "insufficient cross-sectional overlap across stocks' histories"
            _CACHE[cache_key] = out
            return out

        last_val = float(agg.iloc[-1])
        last_date = agg.index[-1]

        def chg_since(days):
            target = last_date - pd.Timedelta(days=days)
            sub = agg[agg.index <= target]
            if sub.empty:
                return None
            base = float(sub.iloc[-1])
            if base == 0:
                return None
            return (last_val - base) / base * 100.0

        out["value"] = last_val
        out["as_of"] = str(last_date.date()) if hasattr(last_date, "date") else str(last_date)
        out["n_stocks"] = int(counts.loc[last_date]) if last_date in counts.index else None
        out["d"] = chg_since(1)
        out["s"] = chg_since(30)
        out["m"] = chg_since(365)
        out["l"] = chg_since(365 * 5)
        out["zscore"] = _zscore(agg, last_val, lookback_days=lookback_days, min_points=60)
        cutoff = last_date - pd.Timedelta(days=365)
        window = agg[agg.index >= cutoff]
        if not window.empty:
            yr_high, yr_low = float(window.max()), float(window.min())
            if yr_high > yr_low:
                out["range_pct"] = (last_val - yr_low) / (yr_high - yr_low) * 100.0
        out["ok"] = True
    except Exception as e:
        out["error"] = str(e)
    _CACHE[cache_key] = out
    return out


def fetch_cot(commodity_name, years=6):
    """Weekly futures positioning from the CFTC's Legacy "Futures Only"
    Commitment of Traders report -- free, no key, via CFTC's public Socrata
    API (confirmed live 2026-08-21). Reports the "Non-Commercial" bucket
    (speculators -- hedge funds, CTAs, etc., as opposed to "Commercial"
    hedgers) as a NET position (long minus short) scaled to % of total open
    interest, which is the standard way this is normalized across time
    since raw contract counts aren't comparable as open interest itself
    grows or shrinks. Positive/rising = speculators net long and adding
    (more bullish positioning, but also more one-sided/crowded and prone to
    a squeeze if sentiment reverses); negative/falling = net short.

    Several thin secondary contracts can share the same `commodity_name`
    (e.g. ICE basis swaps or micro contracts alongside the main NYMEX/COMEX
    contract) -- rather than hardcoding one contract code, this pulls every
    contract under that name and keeps only the single largest-open-
    interest one on each report date, which reliably identifies the
    primary, most-liquid contract without guessing a specific code.

    CAVEAT: the Legacy report's "Non-Commercial" category is a broad,
    decades-old bucket that mixes hedge funds with other large speculators
    -- CFTC's newer Disaggregated report has a narrower "Managed Money"
    category that's a closer read on hedge-fund-specific positioning, but
    lives on a different API endpoint; this uses the Legacy report as a
    real, live, and simpler-to-verify proxy rather than adding a second
    data source for a modest precision gain."""
    cache_key = commodity_name + "::cot"
    if cache_key in _CACHE:
        return _CACHE[cache_key]
    # No "d" (1-day) figure: COT is a weekly-cadence report, so a genuine
    # 1-day change doesn't exist here -- stays None (displays "n/a"),
    # same honest-gap convention as the SEC insider filings signal below.
    out = {"net_pct": None, "d": None, "s": None, "m": None, "l": None, "ok": False, "error": None,
           "zscore": None, "range_pct": None, "report_date": None, "contract_name": None}
    try:
        cutoff = (datetime.now() - pd.Timedelta(days=365 * years)).strftime("%Y-%m-%dT00:00:00.000")
        url = (f"{_COT_BASE_URL}?commodity_name={commodity_name}"
               f"&$where=report_date_as_yyyy_mm_dd >= '{cutoff}'"
               f"&$order=report_date_as_yyyy_mm_dd DESC&$limit=50000")
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        rows = resp.json()
        if not rows:
            out["error"] = "no data"
            _CACHE[cache_key] = out
            return out
        df = pd.DataFrame(rows)
        df["report_date_as_yyyy_mm_dd"] = pd.to_datetime(df["report_date_as_yyyy_mm_dd"])
        for col in ("open_interest_all", "noncomm_positions_long_all", "noncomm_positions_short_all"):
            df[col] = pd.to_numeric(df[col], errors="coerce")
        idx = df.groupby("report_date_as_yyyy_mm_dd")["open_interest_all"].idxmax()
        primary = df.loc[idx].sort_values("report_date_as_yyyy_mm_dd")
        primary = primary[primary["open_interest_all"] > 0]
        primary = primary.assign(net_pct=(
            (primary["noncomm_positions_long_all"] - primary["noncomm_positions_short_all"])
            / primary["open_interest_all"] * 100.0
        ))
        series = primary.set_index("report_date_as_yyyy_mm_dd")["net_pct"].dropna()
        if series.empty:
            out["error"] = "no usable rows"
            _CACHE[cache_key] = out
            return out
        last_val = float(series.iloc[-1])
        last_date = series.index[-1]

        def pp_change_since(days):
            target = last_date - pd.Timedelta(days=days)
            sub = series[series.index <= target]
            if sub.empty:
                return None
            return last_val - float(sub.iloc[-1])  # percentage-point change (net_pct is already a %)

        out["net_pct"] = last_val
        out["s"] = pp_change_since(30)
        out["m"] = pp_change_since(365)
        out["l"] = pp_change_since(365 * 5)
        out["zscore"] = _zscore(series, last_val, lookback_days=365 * 2, min_points=20)
        cutoff_1y = last_date - pd.Timedelta(days=365)
        window = series[series.index >= cutoff_1y]
        if not window.empty:
            yr_high, yr_low = float(window.max()), float(window.min())
            if yr_high > yr_low:
                out["range_pct"] = (last_val - yr_low) / (yr_high - yr_low) * 100.0
        out["report_date"] = last_date.strftime("%Y-%m-%d")
        out["contract_name"] = str(primary.iloc[-1]["market_and_exchange_names"])
        out["ok"] = True
    except Exception as e:
        out["error"] = str(e)
    _CACHE[cache_key] = out
    return out


def fetch_yield(symbol, period="6y"):
    """^IRX/^FVX/^TNX/^TYX on Yahoo are quoted DIRECTLY as the yield in
    percent (e.g. 4.68 means 4.68%) -- confirmed against the live Yahoo
    Finance quote page, no /10 scaling needed.

    Note on ^IRX specifically: it's the 13-week T-bill quoted on a discount-
    yield basis, which is a slightly different (and structurally always a
    bit lower) convention than the "bond-equivalent yield" some other public
    sources quote for the same maturity (e.g. Treasury.gov's par yield
    curve). A small persistent gap there is a real convention difference,
    not a data error.

    Like fetch_returns(), prefers fast_info's live quote snapshot for the
    displayed yield over history()'s last daily bar, which can lag."""
    cache_key = symbol + "::yield"
    if cache_key in _CACHE:
        return _CACHE[cache_key]
    out = {"yield": None, "d": None, "s": None, "m": None, "l": None, "ok": False, "error": None}
    try:
        ticker = yf.Ticker(symbol)
        hist = ticker.history(period=period, auto_adjust=True)
        if hist.empty:
            out["error"] = "no data"
            _CACHE[cache_key] = out
            return out
        closes = hist["Close"].dropna()
        last_date = closes.index[-1]
        fast_price = _fast_last_price(ticker)
        last_yield = fast_price if fast_price is not None else float(closes.iloc[-1])

        def chg_since(days):
            target = last_date - pd.Timedelta(days=days)
            sub = closes[closes.index <= target]
            if sub.empty:
                return None
            base = float(sub.iloc[-1])
            return (last_yield - base) * 100.0  # basis points

        out["yield"] = last_yield
        out["d"] = chg_since(1)
        out["s"] = chg_since(30)
        out["m"] = chg_since(365)
        out["l"] = chg_since(365 * 5)
        out["ok"] = True
    except Exception as e:
        out["error"] = str(e)
    _CACHE[cache_key] = out
    return out


def fetch_fred_yield(series_id, years=6, zscore_lookback_days=365, zscore_min_points=20):
    """International 10-year-equivalent government bond yield from FRED
    (OECD long-term interest rate series) -- also reused for any other
    daily/monthly FRED %-level series that wants bp-change + z-score/range
    signals (yield curve slope, credit spreads, breakeven inflation).
    Monthly or daily cadence depending on series, % already.

    IMPORTANT for monthly-cadence series (the OECD 10Y set used on the
    Bonds tab): FRED itself reports a "Last Updated" timestamp that's
    recent, but that only means OECD's *release* was recent -- the
    underlying observation date lags by 6-8 weeks (confirmed directly
    against FRED: on 2026-08-24, Germany's IRLTLT01DEM156N series' newest
    point was dated 2026-06-01, not August). `out["date"]` returns that
    real observation date so callers can show "as of <date>" instead of
    silently implying the number is today's rate."""
    cache_key = series_id + "::fred"
    if cache_key in _CACHE:
        return _CACHE[cache_key]
    out = {"yield": None, "date": None, "d": None, "s": None, "m": None, "l": None, "ok": False, "error": None,
           "zscore": None, "range_pct": None}
    try:
        end = datetime.now()
        start = end - pd.Timedelta(days=365 * years)
        df = pdr.DataReader(series_id, "fred", start, end)
        series = df[series_id].dropna()
        if series.empty:
            out["error"] = "no data"
            _CACHE[cache_key] = out
            return out
        last_val = float(series.iloc[-1])
        last_date = series.index[-1]

        def chg_since(days):
            target = last_date - pd.Timedelta(days=days)
            sub = series[series.index <= target]
            if sub.empty:
                return None
            base = float(sub.iloc[-1])
            return (last_val - base) * 100.0  # basis points

        out["yield"] = last_val
        out["date"] = last_date.strftime("%Y-%m-%d")
        out["d"] = chg_since(1)
        out["s"] = chg_since(30)
        out["m"] = chg_since(365)
        out["l"] = chg_since(365 * 5)
        out["zscore"] = _zscore(series, last_val, lookback_days=zscore_lookback_days, min_points=zscore_min_points)
        cutoff = last_date - pd.Timedelta(days=365)
        window = series[series.index >= cutoff]
        if not window.empty:
            yr_high, yr_low = float(window.max()), float(window.min())
            if yr_high > yr_low:
                out["range_pct"] = (last_val - yr_low) / (yr_high - yr_low) * 100.0
        out["ok"] = True
    except Exception as e:
        out["error"] = str(e)
    _CACHE[cache_key] = out
    return out


def fetch_fred_level(series_id, years=8, zscore_lookback_days=365, zscore_min_points=20):
    """Generic FRED dollar-level series fetch (money fund assets, RRP
    usage, etc.) -- not a yield. Computes % change over each horizon and a
    z-score vs its own trailing history, same signal system as
    fetch_returns(). `zscore_lookback_days`/`zscore_min_points` are tuned
    per-series at the call site since these run at very different
    frequencies (daily/weekly/quarterly)."""
    cache_key = series_id + "::fred_level"
    if cache_key in _CACHE:
        return _CACHE[cache_key]
    out = {"value": None, "d": None, "s": None, "m": None, "l": None, "ok": False, "error": None, "zscore": None}
    try:
        end = datetime.now()
        start = end - pd.Timedelta(days=365 * years)
        df = pdr.DataReader(series_id, "fred", start, end)
        series = df[series_id].dropna()
        if series.empty:
            out["error"] = "no data"
            _CACHE[cache_key] = out
            return out
        last_val = float(series.iloc[-1])
        last_date = series.index[-1]

        def pct_change_since(days):
            target = last_date - pd.Timedelta(days=days)
            sub = series[series.index <= target]
            if sub.empty:
                return None
            base = float(sub.iloc[-1])
            if base == 0:
                return None
            return (last_val - base) / base * 100.0

        out["value"] = last_val
        out["d"] = pct_change_since(1)
        out["s"] = pct_change_since(30)
        out["m"] = pct_change_since(365)
        out["l"] = pct_change_since(365 * 5)
        out["zscore"] = _zscore(series, last_val, lookback_days=zscore_lookback_days, min_points=zscore_min_points)
        out["ok"] = True
    except Exception as e:
        out["error"] = str(e)
    _CACHE[cache_key] = out
    return out


def fetch_fred_ratio(numerator_id, denominator_id, numerator_scale=1.0, denominator_scale=1.0,
                      years=8, zscore_lookback_days=365, zscore_min_points=20):
    """numerator/denominator as a %, from two FRED series that may run at
    different frequencies (e.g. WRMFNS is weekly, M2SL is monthly) --
    aligned by forward-filling the lower-frequency series onto the
    higher-frequency one's dates before dividing. `numerator_scale` /
    `denominator_scale` convert each series to matching dollar units
    (e.g. millions -> billions) first.

    Dividing a cash aggregate by a broad monetary aggregate like M2 nets
    out generic nominal growth (inflation, money-supply expansion) that
    would otherwise inflate the raw dollar level regardless of any real
    shift in cash preference -- this is a materially cleaner signal than
    fetch_fred_level()'s raw level for exactly that reason. `s`/`m`/`l`
    here are ABSOLUTE percentage-point changes in the ratio (not a
    relative % change of a %, which would be confusing for a share
    metric) -- use fmt_pp_change() to display them, not fmt_pct()."""
    cache_key = f"{numerator_id}::{denominator_id}::ratio"
    if cache_key in _CACHE:
        return _CACHE[cache_key]
    out = {"value": None, "d": None, "s": None, "m": None, "l": None, "ok": False, "error": None, "zscore": None}
    try:
        end = datetime.now()
        start = end - pd.Timedelta(days=365 * years)
        num = pdr.DataReader(numerator_id, "fred", start, end)[numerator_id].dropna() * numerator_scale
        den = pdr.DataReader(denominator_id, "fred", start, end)[denominator_id].dropna() * denominator_scale
        if num.empty or den.empty:
            out["error"] = "no data"
            _CACHE[cache_key] = out
            return out
        den_aligned = den.reindex(num.index.union(den.index)).sort_index().ffill().reindex(num.index)
        ratio = (num / den_aligned * 100.0).dropna()
        if ratio.empty:
            out["error"] = "no overlapping data after alignment"
            _CACHE[cache_key] = out
            return out
        last_val = float(ratio.iloc[-1])
        last_date = ratio.index[-1]

        def pp_change_since(days):
            target = last_date - pd.Timedelta(days=days)
            sub = ratio[ratio.index <= target]
            if sub.empty:
                return None
            return last_val - float(sub.iloc[-1])  # percentage-point change

        out["value"] = last_val
        out["d"] = pp_change_since(1)
        out["s"] = pp_change_since(30)
        out["m"] = pp_change_since(365)
        out["l"] = pp_change_since(365 * 5)
        out["zscore"] = _zscore(ratio, last_val, lookback_days=zscore_lookback_days, min_points=zscore_min_points)
        out["ok"] = True
    except Exception as e:
        out["error"] = str(e)
    _CACHE[cache_key] = out
    return out


_SEC_CASH_TAGS = [
    "CashAndCashEquivalentsAtCarryingValue",
    "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents",
]

def fetch_company_cash(cik):
    """Cash & equivalents for a public company, from SEC EDGAR's free XBRL
    company-concept API -- quarterly/annual (from 10-Q/10-K filings), lags
    roughly 40 days after each quarter-end for the filing to post. No API
    key, but SEC's fair-access policy requires a real identifying
    User-Agent -- edit SEC_USER_AGENT above with your own name/email.

    IMPORTANT CAVEAT, unlike everything else in this notebook: this
    integration could NOT be live-tested from the environment that built
    it -- SEC EDGAR requests hung rather than completing there, so the
    exact response shape/field names below follow SEC's documented, stable
    API format rather than a fetch I personally confirmed this session.
    The code degrades cleanly (shows "unavailable" per company, tries a
    fallback tag first) if anything doesn't match on your machine, but
    treat the first real run as this feature's actual verification, and
    flag back if a number looks obviously wrong.

    Note on Berkshire Hathaway specifically: the "cash pile" figure widely
    quoted in the media usually ALSO includes short-term Treasury bill
    holdings on top of cash & equivalents -- this only pulls the cash &
    equivalents line (the one part reliably tagged the same way across
    companies), so it will likely run lower than headline news figures.
    """
    cache_key = cik + "::sec_cash"
    if cache_key in _CACHE:
        return _CACHE[cache_key]
    out = {"value": None, "date": None, "s": None, "m": None, "l": None,
           "ok": False, "error": None, "zscore": None, "tag_used": None}
    headers = {"User-Agent": SEC_USER_AGENT}
    cik_padded = str(cik).zfill(10)
    last_error = None
    for tag in _SEC_CASH_TAGS:
        try:
            url = f"https://data.sec.gov/api/xbrl/companyconcept/CIK{cik_padded}/us-gaap/{tag}.json"
            resp = requests.get(url, headers=headers, timeout=15)
            if resp.status_code != 200:
                last_error = f"HTTP {resp.status_code} for tag {tag}"
                continue
            data = resp.json()
            facts = data.get("units", {}).get("USD", [])
            filed = [f for f in facts if f.get("form") in ("10-Q", "10-K") and f.get("end")]
            if not filed:
                last_error = f"no 10-Q/10-K facts for tag {tag}"
                continue
            filed.sort(key=lambda f: f["end"])
            series = pd.Series({pd.Timestamp(f["end"]): float(f["val"]) for f in filed}).sort_index()
            series = series[~series.index.duplicated(keep="last")]
            last_val = float(series.iloc[-1])
            last_date = series.index[-1]

            def pct_change_since(days):
                target = last_date - pd.Timedelta(days=days)
                sub = series[series.index <= target]
                if sub.empty:
                    return None
                base = float(sub.iloc[-1])
                if base == 0:
                    return None
                return (last_val - base) / base * 100.0

            out["value"] = last_val
            out["date"] = last_date.strftime("%Y-%m-%d")
            out["s"] = pct_change_since(95)  # ~1 quarter -- filings are quarterly, not monthly
            out["m"] = pct_change_since(365)
            out["l"] = pct_change_since(365 * 5)
            out["zscore"] = _zscore(series, last_val, lookback_days=365 * 5, min_points=6)
            out["tag_used"] = tag
            out["ok"] = True
            break
        except Exception as e:
            last_error = str(e)
            continue
    if not out["ok"]:
        out["error"] = last_error or "unknown error"
    _CACHE[cache_key] = out
    return out


def _nport_quarter_candidates(n=4):
    """(year, quarter) pairs to try, starting from the most recently
    COMPLETED calendar quarter (the current, still-in-progress quarter's
    data isn't posted yet) and walking backward -- handles the case where
    the most recent completed quarter hasn't been published yet either."""
    now = datetime.now()
    year, quarter = now.year, (now.month - 1) // 3 + 1
    quarter -= 1
    if quarter == 0:
        quarter, year = 4, year - 1
    out = []
    for _ in range(n):
        out.append((year, quarter))
        quarter -= 1
        if quarter == 0:
            quarter, year = 4, year - 1
    return out


def _classify_cash_like_asset_cats(unique_cats):
    """Given the ACTUAL unique ASSET_CAT values found in a real N-PORT
    filing this run, identify which ones represent cash-like holdings --
    short-term investment vehicles (money market funds, liquidity pools,
    cash management vehicles) and repurchase agreements -- per Form
    N-PORT Item C.4.a's defined category list. SEC's public technical
    spec for the bulk data files doesn't enumerate the literal short
    codes used in the TSV, so this checks a set of plausible candidates
    (exact match, case-insensitive) plus a conservative keyword fallback,
    rather than trusting one guessed string blindly. Returns the matched
    subset of the actual values passed in (so callers can display exactly
    what was used) -- empty if nothing matched, which callers must treat
    as a hard "couldn't verify" rather than silently reporting zero."""
    stiv_candidates = {"STIV", "ST", "STV", "SHORT-TERM INVESTMENT VEHICLE"}
    ra_candidates = {"RA", "REPO", "REPURCHASE AGREEMENT"}
    matched = set()
    for cat in unique_cats:
        if cat is None or (isinstance(cat, float) and pd.isna(cat)):
            continue
        c = str(cat).strip().upper()
        if c in stiv_candidates or "STIV" in c or ("SHORT" in c and "TERM" in c and "INVEST" in c):
            matched.add(cat)
        elif c in ra_candidates or "REPO" in c or "REPURCHASE" in c:
            matched.add(cat)
    return matched


def fetch_smart_money_cash(progress_cb=None):
    """Aggregate cash-as-%-of-assets across SEC-registered N-PORT filers
    (mutual funds & ETFs; money market funds excluded, they file a
    different form) -- a real, regulator-sourced proxy in the same spirit
    as the classic "mutual fund cash ratio" sentiment gauge: low = funds
    are fully invested with little dry powder (historically nearer market
    tops), high = more cash cushion sitting out (historically nearer
    troughs).

    Built from SEC's free bulk N-PORT Data Sets (no key) -- confirmed live
    at sec.gov/data-research/sec-markets-data/form-n-port-data-sets, one
    ZIP per quarter, ~400-480MB each (confirmed at build time).

    Computes TWO figures per fund, both asset-weighted and simple-averaged
    across the industry:
    - "loose cash": CASH_NOT_RPTD_IN_C_OR_D / TOTAL_ASSETS -- cash not
      already itemized as a security holding (from FUND_REPORTED_INFO,
      small table).
    - "near-cash": loose cash PLUS holdings tagged as short-term investment
      vehicles (money market funds, liquidity pools, cash management
      vehicles) or repurchase agreements -- the two categories Form N-PORT
      Item C.4.a defines specifically for cash-like positions (confirmed
      against SEC's own Form N-PORT text) -- divided by TOTAL_ASSETS. This
      requires reading FUND_REPORTED_HOLDING, by far the largest table in
      the ZIP (one row per security position across ~14,000+ funds), so
      it's meaningfully heavier on memory/time than the loose-cash figure
      alone.

    IMPORTANT caveat on the near-cash figure specifically: SEC's public
    technical spec for the bulk files doesn't enumerate the literal short
    codes used for ASSET_CAT in the TSV (only the human-readable category
    list in the form instructions). This code detects the matching
    categories from whatever ASSET_CAT values actually appear in the real
    data each run (see _classify_cash_like_asset_cats()) rather than
    trusting one guessed string blindly, and reports exactly which literal
    values it matched so you can sanity-check them. If nothing matches,
    the near-cash figure comes back as unavailable (`near_cash_ok=False`)
    rather than silently reporting zero -- the loose-cash figure still
    works independently either way.

    Other disclosed simplifications (both figures are still real but
    rougher, broader proxies than the textbook version):
    - Covers ALL N-PORT filers, not just "actively managed domestic equity
      mutual funds" specifically -- no free way to filter to equity-only
      funds without classifying every individual holding by issuer type,
      which this doesn't attempt.
    - Amendments (NPORT-P/A) are excluded to avoid double-counting; only
      original NPORT-P filings are used.
    - Only the latest available quarter loads -- no automatic historical
      trend/z-score, since each additional quarter is another
      ~400-480MB download.

    Heavy and opt-in only -- this is NOT part of the normal Refresh cycle,
    it only runs when the "Load Smart Money Cash Signal" button is clicked.
    """
    cache_key = "smart_money_cash"
    if cache_key in _CACHE:
        return _CACHE[cache_key]
    out = {"weighted_pct": None, "simple_pct": None, "n_funds": None,
           "total_assets_usd": None, "report_date": None, "quarter_label": None,
           "near_cash_weighted_pct": None, "near_cash_simple_pct": None,
           "near_cash_ok": False, "near_cash_cats_used": None,
           "ok": False, "error": None}
    headers = {"User-Agent": SEC_USER_AGENT}
    last_error = None
    for year, quarter in _nport_quarter_candidates(4):
        url = _NPORT_BASE_URL.format(year=year, quarter=quarter)
        tmp_path = None
        try:
            if progress_cb:
                progress_cb(f"Trying {year} Q{quarter} -- downloading (~450MB, this can take a few minutes)...")
            resp = requests.get(url, headers=headers, timeout=300, stream=True)
            if resp.status_code != 200:
                last_error = f"{year}Q{quarter}: HTTP {resp.status_code} (likely not posted yet)"
                continue

            tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".zip")
            tmp_path = tmp.name
            downloaded = 0
            next_report_at = 25 * 1024 * 1024
            for chunk in resp.iter_content(chunk_size=1024 * 1024):
                if not chunk:
                    continue
                tmp.write(chunk)
                downloaded += len(chunk)
                if progress_cb and downloaded >= next_report_at:
                    progress_cb(f"{year} Q{quarter}: downloaded {downloaded / (1024 * 1024):.0f} MB...")
                    next_report_at += 25 * 1024 * 1024
            tmp.close()

            if progress_cb:
                progress_cb(f"{year} Q{quarter}: extracting SUBMISSION + FUND_REPORTED_INFO...")
            with zipfile.ZipFile(tmp_path) as zf:
                with zf.open("SUBMISSION.tsv") as f:
                    sub = pd.read_csv(f, sep="\t", usecols=["ACCESSION_NUMBER", "SUB_TYPE", "REPORT_DATE"],
                                       low_memory=False)
                with zf.open("FUND_REPORTED_INFO.tsv") as f:
                    info = pd.read_csv(f, sep="\t",
                                        usecols=["ACCESSION_NUMBER", "TOTAL_ASSETS", "CASH_NOT_RPTD_IN_C_OR_D"],
                                        low_memory=False)

                sub = sub[sub["SUB_TYPE"] == "NPORT-P"]
                df = sub.merge(info, on="ACCESSION_NUMBER", how="inner")
                df["CASH_NOT_RPTD_IN_C_OR_D"] = df["CASH_NOT_RPTD_IN_C_OR_D"].fillna(0)
                df = df.dropna(subset=["TOTAL_ASSETS"])
                df = df[df["TOTAL_ASSETS"] > 0]
                df["cash_pct"] = df["CASH_NOT_RPTD_IN_C_OR_D"] / df["TOTAL_ASSETS"] * 100.0
                df = df[(df["cash_pct"] >= 0) & (df["cash_pct"] <= 100)]

                if df.empty:
                    last_error = f"{year}Q{quarter}: no usable rows after filtering"
                    continue

                weighted_pct = float((df["cash_pct"] * df["TOTAL_ASSETS"]).sum() / df["TOTAL_ASSETS"].sum())
                simple_pct = float(df["cash_pct"].mean())
                report_dates = pd.to_datetime(df["REPORT_DATE"], errors="coerce").dropna()
                report_date = report_dates.max().strftime("%Y-%m-%d") if not report_dates.empty else None

                out.update({
                    "weighted_pct": weighted_pct, "simple_pct": simple_pct, "n_funds": int(len(df)),
                    "total_assets_usd": float(df["TOTAL_ASSETS"].sum()), "report_date": report_date,
                    "quarter_label": f"{year} Q{quarter}", "ok": True,
                })

                # Heavier step: near-cash breakdown from the holdings-level
                # table. Failure here doesn't undo the loose-cash result above.
                try:
                    if progress_cb:
                        progress_cb(f"{year} Q{quarter}: reading holdings-level table for near-cash "
                                     "categories (largest file, may take a while)...")
                    with zf.open("FUND_REPORTED_HOLDING.tsv") as f:
                        holdings = pd.read_csv(
                            f, sep="\t", usecols=["ACCESSION_NUMBER", "ASSET_CAT", "CURRENCY_VALUE"],
                            dtype={"ACCESSION_NUMBER": "string", "ASSET_CAT": "category"},
                            low_memory=False,
                        )
                    unique_cats = list(holdings["ASSET_CAT"].cat.categories) if hasattr(holdings["ASSET_CAT"], "cat") \
                        else holdings["ASSET_CAT"].dropna().unique().tolist()
                    cash_like_cats = _classify_cash_like_asset_cats(unique_cats)
                    if cash_like_cats:
                        mask = holdings["ASSET_CAT"].isin(cash_like_cats)
                        per_fund_nc = holdings.loc[mask].groupby("ACCESSION_NUMBER")["CURRENCY_VALUE"].sum()
                        df["near_cash_holdings"] = df["ACCESSION_NUMBER"].map(per_fund_nc).fillna(0)
                        df["near_cash_total"] = df["CASH_NOT_RPTD_IN_C_OR_D"] + df["near_cash_holdings"]
                        df["near_cash_pct"] = df["near_cash_total"] / df["TOTAL_ASSETS"] * 100.0
                        valid_nc = df[(df["near_cash_pct"] >= 0) & (df["near_cash_pct"] <= 100)]
                        if not valid_nc.empty:
                            nc_weighted = float((valid_nc["near_cash_pct"] * valid_nc["TOTAL_ASSETS"]).sum()
                                                 / valid_nc["TOTAL_ASSETS"].sum())
                            nc_simple = float(valid_nc["near_cash_pct"].mean())
                            out.update({
                                "near_cash_weighted_pct": nc_weighted, "near_cash_simple_pct": nc_simple,
                                "near_cash_ok": True, "near_cash_cats_used": sorted(str(c) for c in cash_like_cats),
                            })
                    else:
                        out["near_cash_cats_used"] = []  # signals "looked, found nothing plausible"
                except Exception:
                    pass  # near-cash is a bonus on top of the already-successful loose-cash result

                break
        except Exception as e:
            last_error = f"{year}Q{quarter}: {e}"
        finally:
            if tmp_path and os.path.exists(tmp_path):
                try:
                    os.unlink(tmp_path)
                except Exception:
                    pass
    if not out["ok"]:
        out["error"] = last_error or "unknown error"
    _CACHE[cache_key] = out
    return out


def _insider_quarter_candidates(n=4):
    """Same walk-back logic as _nport_quarter_candidates(): start from the
    most recently COMPLETED calendar quarter and step backward, since the
    current in-progress quarter's data isn't posted yet."""
    now = datetime.now()
    year, quarter = now.year, (now.month - 1) // 3 + 1
    quarter -= 1
    if quarter == 0:
        quarter, year = 4, year - 1
    out = []
    for _ in range(n):
        out.append((year, quarter))
        quarter -= 1
        if quarter == 0:
            quarter, year = 4, year - 1
    return out


def _fetch_insider_quarter(year, quarter, progress_cb=None):
    """Downloads and parses ONE quarter of SEC's free bulk Insider
    Transactions (Form 3/4/5) data set -- confirmed live 2026-08-21,
    ~8-15MB per quarter (much lighter than the N-PORT smart-money
    download). Cached per-quarter (not just per-"latest fetch"), so
    plotting a multi-quarter trend later never re-downloads a quarter
    that's already been fetched once.

    Restricts to TRANS_CODE 'P' (open-market purchase) and 'S' (open-
    market sale) only -- deliberately excluding option exercises,
    grants/awards, gifts, and other non-discretionary codes (A, F, M, G,
    etc.), since those don't reflect a genuine buy/sell conviction call
    the way an open-market trade does. Also restricts to DOCUMENT_TYPE '4'
    (the as-it-happens transaction report), not Form 3 (initial ownership)
    or Form 5 (annual catch-up), to focus on real-time trading activity.

    Reports buy_pct_of_dollar_volume (total $ bought / total $ bought+sold
    -- insiders structurally sell far more than they buy, so this is
    normally well under 50%; watch the trend/extremes, not the absolute
    level) and n_companies_net_buying/n_companies_net_selling (breadth --
    how many distinct companies had more $ bought than sold that quarter).

    Simplifications (real but rougher than a hand-curated feed): doesn't
    weight by company size/sector, doesn't exclude 10b5-1 pre-scheduled
    sale plans (SEC's data doesn't reliably flag which trades are
    pre-scheduled vs. discretionary), and covers all SEC filers, not just
    a specific index."""
    cache_key = f"insider::{year}q{quarter}"
    if cache_key in _CACHE:
        return _CACHE[cache_key]
    out = {"buy_pct_of_dollar_volume": None, "total_buy_usd": None, "total_sell_usd": None,
           "n_companies_net_buying": None, "n_companies_net_selling": None,
           "n_transactions": None, "quarter_label": f"{year} Q{quarter}", "year": year, "quarter": quarter,
           "ok": False, "error": None, "data_quality_flag": None}
    headers = {"User-Agent": SEC_USER_AGENT}
    tmp_path, resp, last_error = None, None, None
    for url_template in _INSIDER_URL_TEMPLATES:
        url = url_template.format(year=year, quarter=quarter)
        try:
            if progress_cb:
                progress_cb(f"{year} Q{quarter}: downloading (~10-15MB)...")
            r = requests.get(url, headers=headers, timeout=60, stream=True)
            if r.status_code == 200:
                resp = r
                break
            last_error = f"HTTP {r.status_code} at {url}"
        except Exception as e:
            last_error = str(e)
    if resp is None:
        out["error"] = last_error or "not posted yet"
        _CACHE[cache_key] = out
        return out
    try:
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".zip")
        tmp_path = tmp.name
        for chunk in resp.iter_content(chunk_size=1024 * 1024):
            if chunk:
                tmp.write(chunk)
        tmp.close()

        if progress_cb:
            progress_cb(f"{year} Q{quarter}: extracting SUBMISSION + NONDERIV_TRANS...")
        # dtype forced to str for the filter/join key columns -- SUBMISSION's
        # DOCUMENT_TYPE and NONDERIV_TRANS's TRANS_CODE are short alphanumeric
        # codes ("4", "4/A", "P", "S", ...); letting pandas auto-infer risks a
        # silent int64 misread if a particular file happens to contain only
        # numeric-looking values in a given column, which would silently
        # break the string equality filters below.
        with zipfile.ZipFile(tmp_path) as zf:
            with zf.open("SUBMISSION.tsv") as f:
                sub = pd.read_csv(f, sep="\t", usecols=["ACCESSION_NUMBER", "DOCUMENT_TYPE", "ISSUERCIK"],
                                   dtype=str, low_memory=False)
            with zf.open("NONDERIV_TRANS.tsv") as f:
                trans = pd.read_csv(
                    f, sep="\t",
                    usecols=["ACCESSION_NUMBER", "TRANS_CODE", "TRANS_SHARES", "TRANS_PRICEPERSHARE"],
                    dtype={"ACCESSION_NUMBER": str, "TRANS_CODE": str}, low_memory=False,
                )
        # Strip stray whitespace before filtering/merging -- some quarterly
        # exports have been observed to have padded values in these short
        # code fields (e.g. "S " instead of "S"), which would otherwise
        # silently fail an exact-match filter/join for just that quarter's
        # file and quietly zero out one side of the buy/sell split.
        sub["ACCESSION_NUMBER"] = sub["ACCESSION_NUMBER"].str.strip()
        sub["DOCUMENT_TYPE"] = sub["DOCUMENT_TYPE"].str.strip()
        trans["ACCESSION_NUMBER"] = trans["ACCESSION_NUMBER"].str.strip()
        trans["TRANS_CODE"] = trans["TRANS_CODE"].str.strip()

        # De-duplicate before merging. SEC's structured Form 4 datasets have a
        # documented quirk: a single filing covering multiple reporting owners
        # (e.g. an insider + a family trust filing jointly) can produce more
        # than one SUBMISSION row for the same ACCESSION_NUMBER, and/or the
        # identical transaction line repeated once per owner in
        # NONDERIV_TRANS. Left alone, the merge below fans out and silently
        # multiplies (or badly skews) the dollar volume on one side of the
        # buy/sell split -- a much more likely explanation for a
        # too-clean-looking 100%/0% quarter than the whitespace issue alone.
        # Collapsing both to one row per (filing) / one row per (distinct
        # transaction) is the conservative fix: a truly repeated identical
        # transaction line is far more likely a duplicate than two separate
        # real trades.
        sub = sub.drop_duplicates(subset=["ACCESSION_NUMBER"])
        trans = trans.drop_duplicates(subset=["ACCESSION_NUMBER", "TRANS_CODE", "TRANS_SHARES", "TRANS_PRICEPERSHARE"])

        sub = sub[sub["DOCUMENT_TYPE"] == "4"]
        trans = trans[trans["TRANS_CODE"].isin(["P", "S"])]
        df = sub.merge(trans, on="ACCESSION_NUMBER", how="inner")
        df["TRANS_SHARES"] = pd.to_numeric(df["TRANS_SHARES"], errors="coerce")
        df["TRANS_PRICEPERSHARE"] = pd.to_numeric(df["TRANS_PRICEPERSHARE"], errors="coerce")
        df = df.dropna(subset=["TRANS_SHARES", "TRANS_PRICEPERSHARE"])
        df["dollar_value"] = df["TRANS_SHARES"] * df["TRANS_PRICEPERSHARE"]
        df = df[df["dollar_value"] > 0]

        if df.empty:
            out["error"] = "no usable P/S transactions after filtering"
            _CACHE[cache_key] = out
            return out

        buys = df[df["TRANS_CODE"] == "P"]
        sells = df[df["TRANS_CODE"] == "S"]
        total_buy = float(buys["dollar_value"].sum())
        total_sell = float(sells["dollar_value"].sum())
        per_company_buy = buys.groupby("ISSUERCIK")["dollar_value"].sum()
        per_company_sell = sells.groupby("ISSUERCIK")["dollar_value"].sum()
        per_company_net = per_company_buy.add(per_company_sell.mul(-1), fill_value=0)

        # Sanity guard: real quarterly insider data always has BOTH buying
        # and selling -- a quarter coming back with either side at exactly
        # $0 (giving a degenerate 0% or 100% split) is far more likely a
        # parsing/format quirk specific to that file than a genuine market
        # reading, so it's flagged rather than reported as a real number.
        # This was observed in practice (three real quarters came back
        # pinned at exactly 100%) even after the whitespace fix above, so
        # this guard is the honest fallback: don't silently show a
        # misleading extreme value.
        buy_pct = (total_buy / (total_buy + total_sell) * 100.0) if (total_buy + total_sell) > 0 else None
        data_quality_flag = None
        if len(buys) > 0 and len(sells) == 0:
            data_quality_flag = "no sell-side ($) transactions survived parsing -- likely a data artifact, not a real 100% buying quarter"
            buy_pct = None
        elif len(sells) > 0 and len(buys) == 0:
            data_quality_flag = "no buy-side ($) transactions survived parsing -- likely a data artifact, not a real 0% buying quarter"
            buy_pct = None

        out.update({
            "buy_pct_of_dollar_volume": buy_pct,
            "total_buy_usd": total_buy, "total_sell_usd": total_sell,
            "n_companies_net_buying": int((per_company_net > 0).sum()),
            "n_companies_net_selling": int((per_company_net < 0).sum()),
            "n_transactions": int(len(df)), "n_buy_transactions": int(len(buys)),
            "n_sell_transactions": int(len(sells)), "data_quality_flag": data_quality_flag, "ok": True,
        })
    except Exception as e:
        out["error"] = str(e)
    finally:
        if tmp_path and os.path.exists(tmp_path):
            try:
                os.unlink(tmp_path)
            except Exception:
                pass
    _CACHE[cache_key] = out
    return out


def fetch_insider_aggregate(progress_cb=None):
    """Latest available quarter only -- thin wrapper over
    _fetch_insider_quarter() that walks back from the most recently
    completed quarter until one succeeds. See _fetch_insider_quarter()'s
    docstring for the full methodology/caveats, shared with
    fetch_insider_history() below."""
    cache_key = "insider_aggregate"
    if cache_key in _CACHE:
        return _CACHE[cache_key]
    last_error = None
    for year, quarter in _insider_quarter_candidates(4):
        d = _fetch_insider_quarter(year, quarter, progress_cb=progress_cb)
        if d["ok"]:
            _CACHE[cache_key] = d
            return d
        last_error = d.get("error")
    out = {"ok": False, "error": last_error or "unknown error"}
    _CACHE[cache_key] = out
    return out


def fetch_insider_history(n_quarters=5, progress_cb=None):
    """Same insider buy/sell signal as fetch_insider_aggregate(), across
    the last `n_quarters` available quarters (5 quarters ~ the last year
    plus one for context) -- for plotting a trend rather than reading a
    single snapshot. Walks back further than n_quarters so a quarter that
    simply hasn't posted yet doesn't shrink the window. Each quarter is
    cached independently via _fetch_insider_quarter(), so this never
    re-downloads a quarter already fetched (by this or the single-quarter
    button), and extending the window later only fetches the new part."""
    cache_key = f"insider_history::{n_quarters}"
    if cache_key in _CACHE:
        return _CACHE[cache_key]
    results = []
    for year, quarter in _insider_quarter_candidates(n_quarters + 3):
        if len(results) >= n_quarters:
            break
        d = _fetch_insider_quarter(year, quarter, progress_cb=progress_cb)
        if d["ok"]:
            results.append(d)
    results.sort(key=lambda d: (d["year"], d["quarter"]))
    _CACHE[cache_key] = results
    return results


def fetch_bond_yield_te(url_slug, maturity_label, page_path="government-bond-yield"):
    """Live government bond yield scraped from TradingEconomics' public
    page, for any country AND any maturity -- the 10-year page by default
    (page_path="government-bond-yield", maturity_label="10Y"), or a
    country's 5-year/30-year maturity sub-page via TE_MATURITY_PAGES.

    Root-cause fix, 2026-08-31: for every non-US country, the Bonds tab's
    SHORT (~5Y) and LONG (~30Y) buckets were never scraped at all -- they
    were *estimated* by shifting that country's real 10Y point using the
    real US curve's own 5Y-10Y / 30Y-10Y spread that day. That's a
    reasonable proxy only when a country's curve shape resembles the US
    curve's; confirmed live it does NOT for several countries right now
    (Japan's real 30Y was 4.15% on 2026-08-31 -- the US-spread estimate
    would have put it near 3.15%, a full percentage point off; France and
    Germany diverge by 0.5-0.8pp too, just less dramatically). Rather than
    estimate, every bucket now tries a real scrape of that maturity's own
    TE page first (same technique already proven for the 10-year page),
    and only falls back to the old US-curve-spread estimate if that
    specific maturity's scrape fails on a given refresh -- see
    render_bonds()'s use of TE_MATURITY_PAGES / _te_bucket_value().

    Country name is deliberately NOT part of the match (only the maturity
    marker is): confirmed live that TE's own pages are inconsistent about
    the country-name wording (UK's 10-year page says "United Kingdom", but
    its 5-year/30-year pages say "UK" -- matching the exact country label
    used to silently break UK's non-10Y scrapes). Matching `.+?` for the
    country name, still anchored inside the single scoped meta-description
    tag right before the maturity marker, is exactly as precise per page
    and robust to that inconsistency across pages.

    Originally this only existed for China's 10-year point, which has no
    free FRED/yfinance source at all (not an OECD member,
    IRLTLT01CNM156N doesn't exist). FRED remains the medium-bucket
    fallback for the other six countries if TE's 10-year scrape fails,
    and remains the sole source for the Chart tab's long history, since
    TE's free page gives no downloadable history -- only a current
    snapshot.

    Two honest caveats, unchanged: (1) TradingEconomics' terms of use are
    written for manual/API-key access, not automated scraping of their
    pages, so this is a gray area, not a sanctioned integration. (2) it's
    fragile: if TE changes a page's wording/structure or blocks the
    request, this silently returns ok=False rather than crashing, and the
    caller falls back cleanly (to FRED for the medium bucket, or to the
    old US-curve-spread estimate for short/long, or to "unavailable" for
    China's medium bucket, which has no fallback at all).
    """
    cache_key = f"{url_slug}::te_bond::{page_path}"
    if cache_key in _CACHE:
        return _CACHE[cache_key]
    out = {"yield": None, "date": None, "ok": False, "error": None}
    try:
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                                  "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"}
        resp = requests.get(f"https://tradingeconomics.com/{url_slug}/{page_path}",
                             headers=headers, timeout=10)
        resp.raise_for_status()
        # Matches on structure (a country name, then the maturity marker,
        # then "Bond Yield", then the number right before "%", then "on
        # <date>") rather than an exact verb or exact country-name string --
        # resilient to "rose to"/"fell to"/"held steady at"/etc. phrasing,
        # and to TE's own inconsistent country-name wording across pages
        # (see docstring above).
        m = re.search(
            r'name="description"\s+content="The yield on .+? ' + re.escape(maturity_label) +
            r' Bond Yield [^%]*?([\d.]+)%\s+on\s+([A-Za-z]+ \d{1,2}, \d{4})',
            resp.text,
        )
        if m:
            out["yield"] = float(m.group(1))
            out["date"] = m.group(2)
            out["ok"] = True
        else:
            out["error"] = "expected pattern not found (TE may have changed their page)"
    except Exception as e:
        out["error"] = str(e)
    _CACHE[cache_key] = out
    return out


def refresh_all(progress_cb=None):
    """Clear the cache and re-pull every tracked ticker/series once."""
    _CACHE.clear()
    commodity_syms = ([t[0] for t in METALS.values()] + [t[0] for t in ENERGY.values()]
                       + [t[0] for t in AGRI.values()])
    all_syms = (commodity_syms + [t[0] for t in METALS_ETF.values()]
                + [t[0] for t in AGRI_ETF.values()]
                + list(CURRENCIES.values()) + list(CURRENCY_BASKET_ETF.values()) + list(CRYPTO.values())
                + list(INDICES.values()) + list(SECTOR_ETFS.values()) + ALL_STOCKS
                + list(SIGNALS_YF.values()))
    total = (len(all_syms) + len(BONDS_US) + len(BONDS_INTL)
              + len(BONDS_INTL_TE) * len(TE_MATURITY_PAGES)  # 3 maturities (5Y/10Y/30Y) per TE country
              + len(TE_MATURITY_PAGES)  # + China, same 3 maturities
              + 1  # US 2-Year (FRED DGS2) -- see render_bonds()'s 2Y-bucket notes
              + len(CASH_GAUGE_FRED) + len(CASH_GAUGE_RATIOS) + len(COMPANY_CASH_CIKS)
              + 4  # diesel crack spread, WTI-Brent spread, Gold/Silver ratio, Copper/Gold ratio (all derived)
              + len(SIGNALS_FRED) + len(COT_COMMODITIES)
              + len(CURRENCIES_INVERTED))  # USD/EUR, USD/GBP inverted variants (see fetch_returns_inverted)
    i = 0
    for sym in all_syms:
        fetch_returns(sym)
        i += 1
        if progress_cb:
            progress_cb(i, total)
    for name, sym in CURRENCIES.items():
        if name in CURRENCIES_INVERTED:
            fetch_returns_inverted(sym)
            i += 1
            if progress_cb:
                progress_cb(i, total)
    fetch_diesel_crack_spread()  # derived from HO=F/CL=F above -- no extra network call
    fetch_price_spread("CL=F", 1.0, "BZ=F", 1.0)  # WTI-Brent spread, also derived
    fetch_price_ratio("GC=F", "SI=F")              # Gold/Silver ratio, derived
    fetch_price_ratio("HG=F", "GC=F", multiplier=1000.0)  # Copper/Gold ratio (x1000), derived
    i += 4
    if progress_cb:
        progress_cb(i, total)
    for series_id in SIGNALS_FRED.values():
        fetch_fred_yield(series_id)
        i += 1
        if progress_cb:
            progress_cb(i, total)
    for commodity_name in COT_COMMODITIES.values():
        fetch_cot(commodity_name)
        i += 1
        if progress_cb:
            progress_cb(i, total)
    for sym in BONDS_US.values():
        fetch_yield(sym)
        i += 1
        if progress_cb:
            progress_cb(i, total)
    fetch_fred_yield(US_2Y_FRED_SERIES)  # US 2-Year -- see render_bonds()'s 2Y-bucket notes
    i += 1
    if progress_cb:
        progress_cb(i, total)
    for series_id in BONDS_INTL.values():
        fetch_fred_yield(series_id)  # fallback source + Chart tab history
        i += 1
        if progress_cb:
            progress_cb(i, total)
    for url_slug in BONDS_INTL_TE.values():
        for page_path, maturity_label in TE_MATURITY_PAGES.values():
            fetch_bond_yield_te(url_slug, maturity_label, page_path)  # primary, live source for short/medium/long
            i += 1
            if progress_cb:
                progress_cb(i, total)
    for page_path, maturity_label in TE_MATURITY_PAGES.values():
        fetch_bond_yield_te("china", maturity_label, page_path)
        i += 1
        if progress_cb:
            progress_cb(i, total)
    for series_id, _scale in CASH_GAUGE_FRED.values():
        zs_days, zs_min = (365 * 5, 6) if series_id == "MMMFFAQ027S" else (365, 20)
        fetch_fred_level(series_id, zscore_lookback_days=zs_days, zscore_min_points=zs_min)
        i += 1
        if progress_cb:
            progress_cb(i, total)
    for num_id, den_id, num_scale, den_scale in CASH_GAUGE_RATIOS.values():
        zs_days, zs_min = (365 * 5, 6) if num_id == "MMMFFAQ027S" else (365, 20)
        fetch_fred_ratio(num_id, den_id, num_scale, den_scale, zscore_lookback_days=zs_days, zscore_min_points=zs_min)
        i += 1
        if progress_cb:
            progress_cb(i, total)
    for cik in COMPANY_CASH_CIKS.values():
        fetch_company_cash(cik)
        i += 1
        if progress_cb:
            progress_cb(i, total)


In [22]:
# ---------------------------------------------------------------------------
# Rendering helpers -- styled HTML cards using the same palette as the
# original dashboard: cream page background, white cards, dark ink text.
# ---------------------------------------------------------------------------

PAPER = "#F5F4EF"   # page/background
INK   = "#14161A"   # primary text
CARD  = "#FFFFFF"   # card background
LINE  = "#D9D7CD"   # card border
MUTED = "#6B6A63"   # secondary text
RED   = "#D8321F"   # down / accent
GREEN = "#1F6B4C"   # up
AMBER = "#B8862B"   # estimated flag
LIGHT_GREEN = "#7FAE93"  # mild-outperformance card frame
LIGHT_RED   = "#E2897E"  # mild-underperformance card frame
COMPARE_COLORS = [RED, GREEN, AMBER, "#3B6EA5", INK]  # up to 5 distinguishable
# lines for the Chart tab's Compare overlay (added 2026-09-16) -- reuses the
# existing palette plus one new steel-blue so it stays visually consistent
# with the rest of the dashboard rather than introducing an unrelated set of
# colors just for this one chart.

def _frame_from_zscore(z):
    """Card-frame color from a mean-reversion z-score (stdevs from the
    trailing-year mean) -- used for commodities/FX/rates/crypto where
    there's no clean equity benchmark to compare against. |z|>=2 = deep
    color (statistically extended), 1<=|z|<2 = light color, else neutral."""
    if z is None:
        return LINE
    if z >= 2:
        return GREEN
    if z >= 1:
        return LIGHT_GREEN
    if z <= -2:
        return RED
    if z <= -1:
        return LIGHT_RED
    return LINE

def _frame_from_relstrength(diff_pp):
    """Card-frame color from excess return vs the S&P 500 (percentage
    points, same horizon) -- used for individual stocks/sector ETFs.
    >=5pp = deep color, 1-5pp = light color, else neutral."""
    if diff_pp is None:
        return LINE
    if diff_pp >= 5:
        return GREEN
    if diff_pp >= 1:
        return LIGHT_GREEN
    if diff_pp <= -5:
        return RED
    if diff_pp <= -1:
        return LIGHT_RED
    return LINE

def _row_frame_color(r):
    """Rows opt in by including a 'relstrength_pp' key (preferred, for
    equities) or a 'zscore' key (everything else). Rows with neither key
    render with the original neutral border -- no visual change."""
    if "relstrength_pp" in r:
        return _frame_from_relstrength(r["relstrength_pp"])
    if "zscore" in r:
        return _frame_from_zscore(r["zscore"])
    return LINE

def fmt_pct(v, suffix="%"):
    if v is None:
        return f"<span style='color:{MUTED}'>n/a</span>"
    color = GREEN if v >= 0 else RED
    arrow = "▲" if v >= 0 else "▼"
    return f"<span style='color:{color};font-weight:600'>{arrow} {v:+.2f}{suffix}</span>"

def fmt_bp(v):
    if v is None:
        return f"<span style='color:{MUTED}'>n/a</span>"
    color = GREEN if v >= 0 else RED
    sign = "+" if v >= 0 else ""
    return f"<span style='color:{color};font-weight:600'>{sign}{v:.0f} bp</span>"

def fmt_price(v, prefix="$", decimals=2):
    if v is None:
        return f"<span style='color:{MUTED}'>n/a &mdash; click Refresh</span>"
    return f"{prefix}{v:,.{decimals}f}"

CARD_STYLE = """
<div style="display:inline-block;width:225px;margin:6px;padding:12px 14px;
border:1px solid """ + LINE + """;border-left:4px solid {frame};border-bottom:4px solid {frame};
border-radius:6px;background:""" + CARD + """;
font-family:-apple-system,Segoe UI,Helvetica,Arial,sans-serif;vertical-align:top;box-shadow:0 1px 2px rgba(0,0,0,.05)">
  <div style="font-size:11px;color:""" + MUTED + """;text-transform:uppercase;letter-spacing:.03em">{ticker}</div>
  <div style="font-size:14px;font-weight:700;margin:2px 0 6px;color:""" + INK + """">{name}</div>
  <div style="font-size:18px;font-weight:700;color:""" + INK + """">{price}</div>
  <div style="font-size:13px;margin-top:4px">{pct}</div>
  {extra}
</div>
"""

def render_grid(rows):
    """rows: list of dicts with keys name, ticker, price, pct, and an
    optional extra (additional HTML -- signal line / relative strength).
    A row may also include 'zscore' or 'relstrength_pp' to color-code its
    left/bottom border -- see _row_frame_color()."""
    if not rows:
        return widgets.HTML(f"<p style='color:{MUTED}'>No data (fetch failed or not yet loaded -- click Refresh).</p>")
    cards = "".join(
        CARD_STYLE.format(ticker=r["ticker"], name=r["name"], price=r["price"],
                           pct=r["pct"], extra=r.get("extra", ""), frame=_row_frame_color(r))
        for r in rows
    )
    return widgets.HTML(f"<div>{cards}</div>")

def fmt_volume(v):
    """Formats a raw shares/contracts trading-volume figure as e.g.
    12.4M, 856.3K, or a plain integer for small values -- added
    2026-09-08 alongside volume_1d/volume_30d_avg. Not a dollar amount --
    see fmt_big_dollars() for that -- just a share/contract count."""
    if v is None:
        return None
    v = float(v)
    if v >= 1_000_000_000:
        return f"{v / 1_000_000_000:.2f}B"
    if v >= 1_000_000:
        return f"{v / 1_000_000:.1f}M"
    if v >= 1_000:
        return f"{v / 1_000:.1f}K"
    return f"{v:.0f}"

def fmt_signal_line(d):
    """Compact 'how stretched is this vs its own history' line: 52-week
    range position (0% = at the year low, 100% = at the year high) and a
    z-score mean-reversion signal (standard deviations from the trailing-
    year mean). Highlights |z| >= 1.5 in amber as a simple "notable, worth
    a look" flag -- not a buy/sell signal, just a stretch indicator.

    Also appends a visible fallback-quote flag when `price_source` isn't
    "direct" -- added 2026-09-07 after a user reported Energy tab prices
    looking stale with no error shown. _fast_quote() has always silently
    fallen back to yfinance's fast_info (or, if even that fails, the last
    history bar) when the direct Yahoo endpoint doesn't return a price for
    that refresh -- fine for graceful degradation, but it meant a
    persistent fallback (e.g. the user's network blocking/throttling that
    one endpoint specifically) was invisible from the dashboard itself,
    with nothing to notice or report. This makes it visible per-card
    instead of silent.

    Also appends beta, R², and alpha vs the S&P 500 (beta/alpha added
    2026-09-07, R² added the same day as a follow-up -- see
    _beta_alpha_r2()'s docstring for the full methodology and its
    simplifications) when the card's data has them. Beta and R² are both
    shown in plain/neutral color normally -- neither is inherently
    "good" or "bad", they're a risk-exposure measure and a fit/reliability
    measure, not stretch signals -- except R² is flagged amber when it's
    below 10%, since that's the case where the beta shown right next to
    it is least reliable (most of this instrument's own variance is
    idiosyncratic, not market-driven, so beta's day-to-day predictive
    value is genuinely weak even though it's still a correctly-computed
    historical average). Alpha is colored green/red by sign like the
    "vs S&P 500" relative-strength line elsewhere, since it's a real
    excess-return number. All three stay blank for instruments this
    notebook doesn't compute them for (bond yield levels, cash gauges,
    COT %, derived ratios/spreads) -- see the Opportunity Signals note
    for the exact scope.

    Also appends a trading-volume line (latest day, and a trailing
    30-day average) when `volume_1d`/`volume_30d_avg` have them -- added
    2026-09-08, captured for free from the same history() call already
    made for price (see _get_volume()'s docstring). Shown plain/neutral,
    no color coding -- volume level alone isn't a directional signal.
    Genuinely absent (not shown at all) for FX pairs and yield tickers,
    which have no real volume concept, and for a handful of thin foreign
    ETF listings -- a real gap, not a fetch failure.

    Also appends a "Trend: Reinforcing/Neutral/Reverting" tag from
    `autocorr_1` (added 2026-09-13 -- see _trend_persistence()'s
    docstring) when present. This is a deliberately narrow, disclosed
    proxy for one piece of Soros's reflexivity theory (self-reinforcing
    feedback between price and perception), not a measurement of the
    theory itself, and it is NOT a buy/sell signal or a forecast of
    magnitude/timing -- both extremes are flagged amber purely as
    "notable, worth a closer look", the same non-directional treatment
    as the z-score stretch indicator above. If anything, a strong
    Reinforcing reading is the regime reflexivity theory says is most
    prone to an eventual sharp reversal once the underlying gap becomes
    unsustainable -- not a reason for more confidence a move continues."""
    parts = []
    rp = d.get("range_pct")
    if rp is not None:
        rp_clamped = max(0, min(100, rp))
        parts.append(f"52wk {rp_clamped:.0f}%")
    z = d.get("zscore")
    if z is not None:
        z_color = AMBER if abs(z) >= 1.5 else MUTED
        sign = "+" if z >= 0 else ""
        parts.append(f"<span style='color:{z_color};font-weight:600'>z {sign}{z:.1f}&sigma;</span>")
    ma_trend = d.get("ma_trend")
    if ma_trend is not None:
        bullish = ma_trend == "bullish"
        color = GREEN if bullish else RED
        if d.get("ma_cross_recent"):
            label = "Golden Cross" if bullish else "Death Cross"
        else:
            label = "50/200: Bullish" if bullish else "50/200: Bearish"
        parts.append(f"<span style='color:{color};font-weight:600'>{label}</span>")
    beta = d.get("beta")
    if beta is not None:
        parts.append(f"&beta; {beta:.2f}")
    r2 = d.get("r_squared")
    if r2 is not None:
        r2_color = AMBER if r2 < 0.10 else MUTED
        parts.append(f"<span style='color:{r2_color};font-weight:600'>R&sup2; {r2*100:.0f}%</span>")
    alpha = d.get("alpha")
    if alpha is not None:
        a_color = GREEN if alpha >= 0 else RED
        sign = "+" if alpha >= 0 else ""
        parts.append(f"<span style='color:{a_color};font-weight:600'>&alpha; {sign}{alpha:.1f}%</span>")
    autocorr = d.get("autocorr_1")
    if autocorr is not None:
        if autocorr >= 0.15:
            regime_label, regime_color = "Reinforcing", AMBER
        elif autocorr <= -0.15:
            regime_label, regime_color = "Reverting", AMBER
        else:
            regime_label, regime_color = "Neutral", MUTED
        sign = "+" if autocorr >= 0 else ""
        parts.append(f"<span style='color:{regime_color};font-weight:600'>Trend: {regime_label} ({sign}{autocorr:.2f})</span>")

    line = ""
    if parts:
        line = f"<div style='font-size:11px;color:{MUTED};margin-top:5px;padding-top:5px;border-top:1px solid {LINE}'>{' &middot; '.join(parts)}</div>"

    vol_1d = d.get("volume_1d")
    vol_30d = d.get("volume_30d_avg")
    if vol_1d is not None:
        vol_bits = [f"Vol {fmt_volume(vol_1d)}"]
        if vol_30d is not None:
            vol_bits.append(f"30d avg {fmt_volume(vol_30d)}")
        line += f"<div style='font-size:10px;color:{MUTED};margin-top:3px'>{' &middot; '.join(vol_bits)}</div>"

    src = d.get("price_source")
    if src == "fast_info":
        line += (f"<div style='font-size:10px;color:{AMBER};font-weight:600;margin-top:3px'>&#9888; "
                  "fallback quote -- direct live feed unreachable this refresh, using yfinance's older "
                  "fast_info snapshot instead (can lag or mis-map for some instruments)</div>")
    elif src == "history_bar":
        line += (f"<div style='font-size:10px;color:{RED};font-weight:600;margin-top:3px'>&#9888; "
                  "last close only -- both live-quote sources failed this refresh, this price may be from "
                  "a prior session</div>")
    return line

def fmt_big_dollars(value, scale_to_billions=1.0):
    """Formats a raw dollar figure (already in $B if scale_to_billions=1,
    otherwise multiplied by scale_to_billions first -- e.g. 0.001 to
    convert millions to billions, 1e-9 to convert raw dollars to billions)
    as $X.XB or $X.XXT."""
    if value is None:
        return f"<span style='color:{MUTED}'>n/a &mdash; click Refresh</span>"
    v = value * scale_to_billions
    if abs(v) >= 1000:
        return f"${v / 1000:,.2f}T"
    return f"${v:,.1f}B"

def fmt_pp_change(v):
    """Absolute percentage-point change, explicitly labeled 'pp' so it's
    never confused with a relative % change -- used for ratio-of-ratio
    metrics like 'MMF share of M2' where the value is already a %."""
    if v is None:
        return f"<span style='color:{MUTED}'>n/a</span>"
    color = GREEN if v >= 0 else RED
    sign = "+" if v >= 0 else ""
    return f"<span style='color:{color};font-weight:600'>{sign}{v:.3f}pp</span>"

def fmt_period_changes(d, short_label):
    """Small line showing the short-horizon and 5yr % change for a level
    series (cash gauge / company cash), independent of the global horizon
    toggle -- these tabs hide that toggle since a single mo/yr/5yr framing
    doesn't fit both a daily RRP series and quarterly company filings."""
    bits = []
    for key, label in ((("s"), short_label), (("l"), "5yr")):
        v = d.get(key)
        if v is None:
            continue
        color = GREEN if v >= 0 else RED
        sign = "+" if v >= 0 else ""
        bits.append(f"{label} <span style='color:{color};font-weight:600'>{sign}{v:.2f}%</span>")
    if not bits:
        return ""
    return f"<div style='font-size:11px;color:{MUTED};margin-top:4px'>{' &middot; '.join(bits)}</div>"

def fmt_relstrength(pct, bench_pct):
    """Excess return vs the S&P 500 over the same horizon, in percentage
    points -- separates real outperformance from a market-wide move."""
    if pct is None or bench_pct is None:
        return ""
    diff = pct - bench_pct
    color = GREEN if diff >= 0 else RED
    sign = "+" if diff >= 0 else ""
    return f"<div style='font-size:11px;color:{color};font-weight:600;margin-top:2px'>{sign}{diff:.2f}pp vs S&amp;P 500</div>"


In [23]:
# ---------------------------------------------------------------------------
# UI assembly
# ---------------------------------------------------------------------------

horizon = widgets.ToggleButtons(
    options=[("1 Day", "d"), ("Short · 1 Month", "s"), ("Medium · 1 Year", "m"), ("Long · 5 Year", "l")],
    value="s",
    description="Horizon:",
    tooltips=["Most recent daily move -- reads n/a for weekly/monthly/quarterly-cadence "
              "data (COT positioning, international bond yields, insider filings, some cash "
              "gauges), since there's no genuine daily figure to show between their own releases.",
              "1 month", "1 year", "5 year"],
)

refresh_btn = widgets.Button(description="\U0001f504 Refresh live data", button_style="")

def _status_html(text, spinning=False):
    """Status line as an HTML widget instead of a plain Label so it can
    show a small CSS spinner while refresh_all() is running. The spinner
    is pure CSS (@keyframes in the STYLE block below), so it keeps
    animating in the browser even while the kernel is busy fetching data --
    no threading needed, since the animation runs client-side once the
    comm message with this HTML has been sent."""
    spinner_html = "<span class='md-spinner'></span> " if spinning else ""
    return f"<span style='font-size:14px;color:{INK}'>{spinner_html}{text}</span>"

status_label = widgets.HTML(value=_status_html("Not yet loaded -- click Refresh, or Run All to auto-load."))

TAB_NAMES = ["Metals", "Energy", "Agriculture", "Currencies", "Crypto", "Indices", "Signals", "Top Performers", "Industries", "Bonds", "Cash & Liquidity", "Chart"]
outputs = {name: widgets.Output() for name in TAB_NAMES}
tabs = widgets.Tab(children=[outputs[n] for n in TAB_NAMES])
for i, n in enumerate(TAB_NAMES):
    tabs.set_title(i, n)

industries_state = {"sector": None}

def on_tab_change(change):
    # Bonds, Cash & Liquidity, and Chart show current levels / historical
    # plots (or mixed-frequency level series), not a return over a single
    # horizon, so the Short/Medium/Long toggle doesn't apply to them.
    if change["name"] == "selected_index":
        name = TAB_NAMES[change["new"]]
        horizon.layout.display = "none" if name in ("Bonds", "Cash & Liquidity", "Chart") else None

tabs.observe(on_tab_change, names="selected_index")


def render_simple_category(registry, unit_prefix="$", decimals=2):
    h = horizon.value
    rows = []
    for name, sym in registry.items():
        d = fetch_returns(sym)
        rows.append({
            "name": name, "ticker": sym,
            "price": fmt_price(d["price"], unit_prefix, decimals),
            "pct": fmt_pct(d.get(h)),
            "extra": fmt_signal_line(d),
            "zscore": d.get("zscore"),
        })
    return render_grid(rows)

def render_commodity_category(registry, decimals=2):
    """For registries shaped name -> (ticker, scale, unit): divides the raw
    exchange-quoted price by `scale` (e.g. cents -> dollars for grains/softs)
    and appends a unit label so the $ figure matches real-world quotes."""
    h = horizon.value
    rows = []
    for name, (sym, scale, unit) in registry.items():
        d = fetch_returns(sym)
        price = d["price"]
        shown = (price / scale) if price is not None else None
        price_html = fmt_price(shown, "$", decimals)
        if shown is not None and unit:
            price_html += f"<span style='font-size:12px;color:{MUTED};font-weight:400'> {unit}</span>"
        rows.append({
            "name": name, "ticker": sym, "price": price_html,
            "pct": fmt_pct(d.get(h)), "extra": fmt_signal_line(d),
            "zscore": d.get("zscore"),
        })
    return render_grid(rows)

def fmt_cot_card(name, d):
    """One COT positioning card: net speculative (non-commercial) position
    as % of open interest, with its own signal line. `pct` shows the 1yr
    (medium) pp change since COT is weekly data and doesn't cleanly fit the
    global horizon toggle's 1mo/1yr/5yr framing the way daily prices do."""
    price_html = f"{d['net_pct']:.1f}%" if d["ok"] else fmt_price(None)
    pct_html = (f"<span style='font-size:11px;color:{MUTED}'>1yr </span>" + fmt_pp_change(d.get("m"))) if d["ok"] else fmt_pct(None)
    extra = fmt_signal_line(d)
    if d["ok"] and d.get("report_date"):
        extra += f"<div style='font-size:10px;color:{MUTED};margin-top:2px'>as of {d['report_date']}</div>"
    return {"name": name, "ticker": "COT: Net Spec. Position", "price": price_html, "pct": pct_html, "extra": extra,
            "zscore": d.get("zscore")}


def render_metals():
    grid = render_commodity_category(METALS)
    h = horizon.value
    gs_d = fetch_price_ratio("GC=F", "SI=F")
    cg_d = fetch_price_ratio("HG=F", "GC=F", multiplier=1000.0)
    ratio_rows = [
        {"name": "Gold/Silver Ratio", "ticker": "GC=F / SI=F",
         "price": f"{gs_d['value']:.2f}" if gs_d["value"] is not None else fmt_price(None),
         "pct": fmt_pct(gs_d.get(h)), "extra": fmt_signal_line(gs_d), "zscore": gs_d.get("zscore")},
        {"name": "Copper/Gold Ratio (\"Dr. Copper\", x1000)", "ticker": "HG=F / GC=F x1000",
         "price": f"{cg_d['value']:.3f}" if cg_d["value"] is not None else fmt_price(None),
         "pct": fmt_pct(cg_d.get(h)), "extra": fmt_signal_line(cg_d), "zscore": cg_d.get("zscore")},
    ]
    cot_rows = [fmt_cot_card(name, fetch_cot(COT_COMMODITIES[name])) for name in ("Gold", "Silver", "Copper")]
    note = widgets.HTML(
        f"<p style='color:{MUTED};font-size:12px;max-width:820px;line-height:1.5'>"
        "<b>Gold/Silver Ratio</b> and <b>Copper/Gold Ratio</b> are derived from prices already tracked "
        "above (no new data source) -- classic relative-value/macro-regime signals: Gold/Silver rising "
        "usually reflects a flight-to-safety bid (gold outperforming its higher-beta, more industrial "
        "cousin); Copper/Gold (\"Dr. Copper\") rising reflects growth optimism outweighing safety-seeking, "
        "and is watched as a rough global-growth proxy. <b>COT: Net Spec. Position</b> cards show the "
        "CFTC's weekly Commitment of Traders data -- non-commercial (speculative) net long/short as a % "
        "of total open interest, free and live via CFTC's public API. Rising/positive = speculators net "
        "long and adding (bullish positioning, but also more crowded/squeeze-prone); falling/negative = "
        "net short. Uses the Legacy report's broader \"Non-Commercial\" bucket (mixes hedge funds with "
        "other large speculators) rather than the narrower \"Managed Money\" category from CFTC's newer "
        "Disaggregated report -- a real but rougher proxy.</p>"
    )
    etf_rows = []
    for name, (sym, prefix) in METALS_ETF.items():
        d = fetch_returns(sym)
        etf_rows.append({
            "name": name, "ticker": sym,
            "price": fmt_price(d["price"], prefix, 2),
            "pct": fmt_pct(d.get(h)), "extra": fmt_signal_line(d),
            "zscore": d.get("zscore"),
        })
    etf_note = widgets.HTML(
        f"<p style='color:{MUTED};font-size:12px;max-width:820px;line-height:1.5'>"
        "<b>Fund/ETP alternatives</b>: physically-backed, exchange-traded shares rather than futures "
        "contracts. <b>Silver</b> keeps its Swisscanto/ZKB Swiss listing (SIX Swiss Exchange, CHF, full "
        "physical bar replication, 0.40% TER). <b>Gold</b> is now Invesco's Physical Gold ETC (SGLD, "
        "London Stock Exchange, USD, 0.12% TER) -- legally an ETC (a collateralized debt obligation, i.e. "
        "a secured note), not a fund like the Swiss listing, Ireland-domiciled, custodied by JPMorgan "
        "Chase Bank, benchmarked to the LBMA Gold Price. Still fully allocated, ring-fenced physical gold "
        "(same investor protection against issuer credit risk as the Swiss fund structure), just a "
        "different legal wrapper and notably cheaper. Two earlier gold picks were tried and replaced: "
        "Swisscanto/ZKB Gold (same Swiss fund structure as Silver, but 0.40% TER) and iShares Gold Trust "
        "Micro (IAUM, cheap per-share at ~$45-50 and the lowest TER of any major US gold ETF at 0.09%, "
        "but dropped once the ask shifted to comparing directly against ZGLD). Note SGLD is <i>not</i> "
        "cheap per-share (~$420, similar magnitude to GLD) -- if per-share affordability matters again, "
        "IAUM is the one to bring back. No Swiss-listed single-commodity Wheat product could be confirmed "
        f"for the Agriculture tab (see there) -- iShares doesn't appear to offer one either. <span "
        f"style='color:{AMBER};font-weight:600'>Unlike every other source in this notebook, these tickers "
        "could not be live-tested end-to-end</span> from the environment that built this (network access "
        "to Yahoo Finance itself was blocked there) -- their existence, currency, and approximate share "
        "price were confirmed via live web search instead. Treat your first real run as the actual "
        "verification, and flag back if a price looks obviously wrong. These are <b>fund/ETC share "
        "prices</b>, not the spot commodity price -- they move with the currency conversion (where "
        "applicable) and fees/tracking on top of the underlying gold/silver move, so don't expect them to "
        "match GC=F/SI=F 1:1.</p>"
    )
    return widgets.VBox([grid, widgets.HTML(f"<h4 style='margin:14px 0 4px;color:{INK}'>Relative-Value &amp; Positioning</h4>"),
                          note, render_grid(ratio_rows + cot_rows),
                          widgets.HTML(f"<h4 style='margin:14px 0 4px;color:{INK}'>Fund/ETP Alternatives</h4>"),
                          etf_note, render_grid(etf_rows)])

def render_energy():
    h = horizon.value
    vendor_note = widgets.HTML(
        f"<p style='color:{MUTED};font-size:12px;max-width:820px;line-height:1.5;margin:0 0 8px'>"
        "<b>On cross-site price checks:</b> WTI/Brent here are Yahoo Finance's own live quotes for the "
        "<code>CL=F</code>/<code>BZ=F</code> futures contracts -- confirmed 2026-09-01 to match Yahoo's "
        "own site to the cent in real time. A different site (e.g. oilprice.com) can legitimately show a "
        "noticeably different \"Brent\"/\"WTI\" number at the very same moment: these sites aggregate from "
        "their own mix of brokers/trading desks/benchmarks, not Yahoo's specific futures feed, so a real, "
        "persistent basis of a dollar or two (roughly 1-3%) between vendors is normal for commodities, not "
        "a bug here -- the same way two FX or crypto data providers can disagree slightly at one instant. "
        "If a number here ever looks off, the useful check is Yahoo Finance's own quote page for that exact "
        "ticker, not a different vendor's headline figure for the same commodity.</p>"
    )
    grid = render_commodity_category(ENERGY)

    nm = fetch_brent_nearest_month()
    bz_price = None
    try:
        bz_price = fetch_returns("BZ=F").get("price")
    except Exception:
        bz_price = None
    if nm["ok"]:
        nm_price_html = fmt_price(nm["price"], "$", 2) + f"<span style='font-size:12px;color:{MUTED};font-weight:400'> /bbl</span>"
        qt = datetime.utcfromtimestamp(nm["quote_time"]).strftime("%b %d, %H:%M") if nm.get("quote_time") else "n/a"
        extra_bits = [f"{nm['label']} contract &middot; {nm['symbol']} &middot; as of {qt} UTC"]
        if bz_price is not None:
            spread = nm["price"] - bz_price
            sign = "+" if spread >= 0 else ""
            extra_bits.append(f"{sign}{spread:.2f} vs continuous BZ=F (${bz_price:,.2f})")
        nm_extra = f"<div style='font-size:11px;color:{MUTED};margin-top:6px;line-height:1.4'>" + "<br>".join(extra_bits) + "</div>"
    else:
        nm_price_html = fmt_price(None)
        nm_extra = f"<div style='font-size:11px;color:{MUTED};margin-top:6px'>No nearby contract returned a fresh live quote this refresh -- click Refresh to retry.</div>"
    nm_row = [{
        "name": "Brent Crude (Nearest Month)", "ticker": nm.get("symbol") or "BZ..YY.NYM",
        "price": nm_price_html, "pct": "", "extra": nm_extra,
    }]
    nm_grid = render_grid(nm_row)
    nm_note = widgets.HTML(
        f"<p style='color:{MUTED};font-size:12px;max-width:820px;line-height:1.5'>"
        "<b>Added 2026-09-22</b>, after confirming live that the \"Brent price difference of 3-5\" report "
        "was a genuine contract-rollover mismatch, not a bug: Yahoo's continuous <code>BZ=F</code> above had "
        "already rolled forward to the December contract while Investing.com/oilprice.com/Barchart's own "
        "\"Brent\" headline figures were all still quoting the soon-to-expire November contract -- both real, "
        "both live, just ~$4 apart on two different delivery months in a backwardated market. This card shows "
        "that <b>same nearest-expiry contract</b> the other sites default to, computed fresh every refresh "
        "(next calendar month's contract, or the one after if that's already rolled off) rather than a "
        "hardcoded ticker that would silently go stale as contracts expire. <b>Spot reading only</b> -- no "
        "52wk range, z-score, beta/alpha, volume, or trend signal, and not used anywhere else in this "
        "notebook (WTI-Brent Spread and the Chart tab's Brent history still use the continuous <code>BZ=F</code> "
        "above, which needs years of unbroken history a single expiring contract can't provide).</p>"
    )

    d = fetch_diesel_crack_spread()
    price_html = fmt_price(d["value"], "$", 2)
    if d["value"] is not None:
        price_html += f"<span style='font-size:12px;color:{MUTED};font-weight:400'> /bbl</span>"
    spread_row = [{
        "name": "Diesel Crack Spread (1:1)", "ticker": "HO=F × 42 − CL=F",
        "price": price_html, "pct": fmt_pct(d.get(h)), "extra": fmt_signal_line(d),
        "zscore": d.get("zscore"),
    }]
    spread_grid = render_grid(spread_row)
    note = widgets.HTML(
        f"<p style='color:{MUTED};font-size:12px;max-width:820px;line-height:1.5'>"
        "<b>Diesel Crack Spread</b> -- the refining margin for turning crude into diesel: "
        "(Heating Oil futures price &times; 42 gal/bbl) minus WTI crude price, in $/bbl. Uses "
        "<code>HO=F</code> (CME/NYMEX Heating Oil, which has tracked ultra-low-sulfur diesel since "
        "a 2013 contract spec change) as diesel's proxy, and <code>CL=F</code> for crude -- both "
        "already tracked above, so this is a <b>derived</b> signal computed from data already being "
        "fetched, not a new data source. Rising = refiners earning more per barrel to produce "
        "diesel (often signals tight diesel supply or strong demand -- worth watching ahead of "
        "winter heating season and during trucking-demand cycles); falling or negative = diesel "
        "refining is squeezed or unprofitable at the margin. % change and z-score are computed on "
        "the spread's own history, not derived from either leg's separate return.</p>"
    )
    wb_d = fetch_price_spread("CL=F", 1.0, "BZ=F", 1.0)
    wb_price_html = fmt_price(wb_d["value"], "$", 2)
    if wb_d["value"] is not None:
        wb_price_html += f"<span style='font-size:12px;color:{MUTED};font-weight:400'> /bbl</span>"
    wb_row = {"name": "WTI-Brent Spread", "ticker": "CL=F − BZ=F",
              "price": wb_price_html, "pct": fmt_pct(wb_d.get(h)), "extra": fmt_signal_line(wb_d),
              "zscore": wb_d.get("zscore")}
    cot_rows = [fmt_cot_card(name, fetch_cot(COT_COMMODITIES[name])) for name in ("WTI Crude Oil", "Natural Gas")]
    wb_note = widgets.HTML(
        f"<p style='color:{MUTED};font-size:12px;max-width:820px;line-height:1.5'>"
        "<b>WTI-Brent Spread</b>: a clean, reliable free version of a real oil-futures \"term structure\" "
        "(contango/backwardation across specific contract months) isn't buildable from yfinance -- "
        "continuous futures contracts don't cleanly expose individual expiry months for free, and "
        "hardcoding specific contract-month tickers is fragile (codes change every roll). This spread "
        "between the two major benchmarks is a genuine, commonly-watched substitute instead: it reflects "
        "regional supply/logistics dynamics (US shale supply, pipeline/export capacity) rather than "
        "calendar term structure, but is real and free. <b>COT: Net Spec. Position</b> cards use the same "
        "CFTC Commitment of Traders methodology described on the Metals tab.</p>"
    )
    return widgets.VBox([vendor_note, grid,
                          widgets.HTML(f"<h4 style='margin:14px 0 4px;color:{INK}'>Nearest-Month Contract</h4>"),
                          nm_grid, nm_note,
                          widgets.HTML(f"<h4 style='margin:14px 0 4px;color:{INK}'>Refining Margin</h4>"),
                          note, spread_grid,
                          widgets.HTML(f"<h4 style='margin:14px 0 4px;color:{INK}'>Regional Spread &amp; Positioning</h4>"),
                          wb_note, render_grid([wb_row] + cot_rows)])
def render_agri():
    h = horizon.value
    grid = render_commodity_category(AGRI)
    cot_rows = [fmt_cot_card(name, fetch_cot(COT_COMMODITIES[name])) for name in ("Corn", "Wheat", "Sugar", "Rice")]
    cot_note = widgets.HTML(
        f"<p style='color:{MUTED};font-size:12px;max-width:820px;line-height:1.5'>"
        "<b>COT: Net Spec. Position</b> cards use the same CFTC Commitment of Traders methodology "
        "described on the Metals tab -- non-commercial (speculative) net long/short as a % of total "
        "open interest, free and live via CFTC's public API. <b>Corn</b> and <b>Wheat</b> (SRW contract) "
        "trade on CBOT; <b>Sugar</b> uses ICE's Sugar No. 11 (world raw sugar) contract; <b>Rice</b> uses "
        "CBOT's Rough Rice contract. Rising/positive = speculators net long and adding (bullish "
        "positioning, but also more crowded/squeeze-prone); falling/negative = net short.</p>"
    )
    etf_rows = []
    for name, (sym, prefix) in AGRI_ETF.items():
        d = fetch_returns(sym)
        etf_rows.append({
            "name": name, "ticker": sym,
            "price": fmt_price(d["price"], prefix, 2),
            "pct": fmt_pct(d.get(h)), "extra": fmt_signal_line(d),
            "zscore": d.get("zscore"),
        })
    etf_note = widgets.HTML(
        f"<p style='color:{MUTED};font-size:12px;max-width:820px;line-height:1.5'>"
        "<b>Fund/ETP alternatives</b>: <b>Corn</b> and <b>Soybean</b> use Teucrium's long-established, "
        "US-listed, USD-denominated funds (CORN trading since 2010, SOYB since 2011), which hold futures "
        "baskets rather than physical grain -- confirmed real via live web search. <b>Wheat</b> has no "
        "confirmed Swiss-listed or iShares single-commodity product -- WisdomTree's London Stock "
        "Exchange-listed Wheat ETC is the nearest verified real substitute instead (not Swiss, not "
        f"iShares, disclosed as neither). <span style='color:{AMBER};font-weight:600'>Could not be "
        "live-tested end-to-end</span> from the environment that built this (network access to Yahoo "
        "Finance was blocked there) -- existence and currency confirmed via web search instead; Wheat's "
        "exact quote convention (commonly GBX/pence for LSE lines) wasn't independently confirmed either, "
        "so verify against a live quote on first run. These are fund share prices, not the spot commodity "
        "price -- they won't track ZC=F/ZS=F/ZW=F 1:1 due to currency (where applicable) and fund "
        "fees/tracking on top of the underlying move.</p>"
    )
    return widgets.VBox([grid,
                          widgets.HTML(f"<h4 style='margin:14px 0 4px;color:{INK}'>Positioning</h4>"),
                          cot_note, render_grid(cot_rows),
                          widgets.HTML(f"<h4 style='margin:14px 0 4px;color:{INK}'>Fund/ETP Alternatives</h4>"),
                          etf_note, render_grid(etf_rows)])

def render_currencies():
    h = horizon.value
    rows = []
    for name, sym in CURRENCIES.items():
        d = fetch_returns_inverted(sym) if name in CURRENCIES_INVERTED else fetch_returns(sym)
        rows.append({
            "name": name, "ticker": sym,
            "price": fmt_price(d["price"], "", 4),
            "pct": fmt_pct(d.get(h)),
            "extra": fmt_signal_line(d),
            "zscore": d.get("zscore"),
        })
    grid = render_grid(rows)
    inv_note = widgets.HTML(
        f"<p style='color:{MUTED};font-size:11px;max-width:820px;line-height:1.5;margin:2px 0 8px'>"
        "<b>USD/EUR</b> and <b>USD/GBP</b> are Yahoo's own EURUSD=X/GBPUSD=X pairs shown inverted "
        "(1/x) so every row here reads USD-first, matching USD/JPY, USD/CHF, and USD/CNH -- those three "
        "already read USD-first natively on Yahoo. Every signal below (52wk range, z-score, 50/200 "
        "trend) is computed on the actual inverted series, not flipped from the original pair's numbers. "
        "<b>EUR/CHF</b> has no USD leg, so it's kept as its own real cross rather than forced into a "
        "USD-first shape that wouldn't mean anything.</p>"
    )
    basket_rows = []
    for name, sym in CURRENCY_BASKET_ETF.items():
        d = fetch_returns(sym)
        basket_rows.append({
            "name": name, "ticker": sym,
            "price": fmt_price(d["price"], "$", 2),
            "pct": fmt_pct(d.get(h)), "extra": fmt_signal_line(d),
            "zscore": d.get("zscore"),
        })
    basket_note = widgets.HTML(
        f"<p style='color:{MUTED};font-size:12px;max-width:820px;line-height:1.5'>"
        "<b>EM Currency Basket</b>: WisdomTree's CEW, a USD-denominated fund long a basket of ~15 "
        "emerging-market currencies (equal-weighted, quarterly rebalance) funded via short-USD forwards "
        "plus a US money-market portfolio -- confirmed real via live web search. Rising = EM currencies "
        "strengthening vs the dollar; falling = EM currencies weakening vs the dollar -- roughly the "
        "mirror image of what <b>US Dollar Index</b> (Signals tab) does for G10 currencies, but for "
        "emerging markets instead. <b>No EUR-denominated equivalent could be confirmed</b> -- EM currency "
        "baskets are conventionally quoted against USD, not EUR, so a genuine \"EUR vs EM basket\" product "
        "doesn't appear to exist; not approximated here with a synthetic cross calculation since that "
        f"could mislead. <span style='color:{AMBER};font-weight:600'>Could not be live-tested end-to-end"
        "</span> from the environment that built this (Yahoo Finance access was blocked there) -- treat "
        "your first real run as the actual verification. This is an actively-managed fund share price "
        "(subject to fees and forward-roll yield), not a clean index level like DXY -- watch the % change "
        "for the FX signal, not the raw price.</p>"
    )
    return widgets.VBox([inv_note, grid,
                          widgets.HTML(f"<h4 style='margin:14px 0 4px;color:{INK}'>EM Currency Basket</h4>"),
                          basket_note, render_grid(basket_rows)])

def render_indices():     return render_simple_category(INDICES, unit_prefix="", decimals=2)

def _vix_context_line(vix_value):
    """Implied monthly move (VIX / sqrt(12), since VIX is an annualized
    30-day figure) plus a commonly-cited regime label. These bands are
    conventional finance-media shorthand, not an official classification --
    disclosed as such rather than presented as a precise threshold."""
    if vix_value is None:
        return ""
    monthly_move = vix_value / (12 ** 0.5)
    if vix_value < 12:
        regime, color = "Very Low / Complacent", GREEN
    elif vix_value < 20:
        regime, color = "Normal / Calm", GREEN
    elif vix_value < 30:
        regime, color = "Elevated", AMBER
    elif vix_value < 40:
        regime, color = "High Stress", RED
    else:
        regime, color = "Crisis-Level", RED
    return (f"<div style='font-size:11px;color:{MUTED};margin-top:4px'>Implied monthly move ~"
            f"<b>&plusmn;{monthly_move:.1f}%</b> &middot; <span style='color:{color};font-weight:600'>{regime}</span>"
            f"<div style='font-size:10px;color:{MUTED};margin-top:2px'>commonly cited: &lt;12 very low, "
            "12-20 normal (long-run avg ~19-20), 20-30 elevated, 30-40 high stress, 40+ crisis (2008, "
            "Mar 2020) -- rough finance-media convention, not an official scale</div></div>")

def _hy_context_line(hy_value):
    """Same idea as _vix_context_line() but for the HY OAS credit spread --
    commonly-cited stress bands, disclosed as informal convention."""
    if hy_value is None:
        return ""
    if hy_value < 3:
        regime, color = "Tight / Low Stress", GREEN
    elif hy_value < 5:
        regime, color = "Normal", GREEN
    elif hy_value < 8:
        regime, color = "Elevated Stress", AMBER
    else:
        regime, color = "Distressed", RED
    return (f"<div style='font-size:11px;color:{MUTED};margin-top:4px'>"
            f"<span style='color:{color};font-weight:600'>{regime}</span>"
            f"<div style='font-size:10px;color:{MUTED};margin-top:2px'>commonly cited: &lt;3% tight, "
            "3-5% normal, 5-8% elevated, 8%+ distressed (2008 peaked ~20%, Mar 2020 COVID spike ~11%) -- "
            "rough finance-media convention, not an official scale</div></div>")

def render_signals():
    h = horizon.value
    rows = []
    for name, sym in SIGNALS_YF.items():
        d = fetch_returns(sym)
        extra = fmt_signal_line(d)
        if name == "VIX (Fear Gauge)":
            extra += _vix_context_line(d.get("price"))
        rows.append({"name": name, "ticker": sym, "price": fmt_price(d["price"], "", 2),
                     "pct": fmt_pct(d.get(h)), "extra": extra, "zscore": d.get("zscore")})
    for name, series_id in SIGNALS_FRED.items():
        d = fetch_fred_yield(series_id)
        price_html = f"{d['yield']:.2f}%" if d["ok"] else fmt_price(None)
        pct_html = fmt_bp(d.get(h)) if d["ok"] else fmt_pct(None)
        extra = fmt_signal_line(d)
        if name == "HY Credit Spread (OAS)":
            extra += _hy_context_line(d.get("yield"))
        rows.append({"name": name, "ticker": series_id, "price": price_html,
                     "pct": pct_html, "extra": extra, "zscore": d.get("zscore")})
    note = widgets.HTML(
        f"<p style='color:{MUTED};font-size:12px;max-width:850px;line-height:1.5'>Cross-asset risk/"
        "stress gauges, all free and confirmed live 2026-08-20/21. <b>VIX</b>: the classic equity "
        "fear gauge (implied 30-day S&amp;P 500 volatility) -- spikes often mark short-term capitulation "
        "lows, persistent complacency (very low readings) can precede corrections. <b>US Dollar Index</b>: "
        "a lot of commodity/EM moves are really dollar moves in disguise -- a rising dollar is a "
        "headwind for dollar-priced commodities and EM assets, independent of their own fundamentals. "
        "<b>10Y-2Y Yield Curve</b> (FRED <code>T10Y2Y</code>): negative = inverted, historically one of "
        "the more reliable recession-warning signals, though with long and variable lead times. <b>HY "
        "Credit Spread</b> (ICE BofA option-adjusted spread over Treasuries): widening = credit markets "
        "pricing in more default/stress risk, often moves before equities do. <b>10Y Breakeven "
        "Inflation</b>: the bond market's own implied 10-year inflation expectation (from TIPS vs. "
        "nominal Treasuries) -- context for Fed-policy and gold/bond positioning, not a trading signal "
        "on its own. This tab respects the horizon toggle like Currencies/Indices since all five series "
        "have genuine daily/near-daily history to compute real 1mo/1yr/5yr changes.</p>"
    )
    return widgets.VBox([note, render_grid(rows)])

def render_crypto():
    note = widgets.HTML(
        f"<p style='color:{MUTED};font-size:12px;max-width:820px;line-height:1.5'>Top 5 "
        "cryptocurrencies by market cap (checked live 2026-08-20) -- a fixed snapshot list, not "
        "re-ranked automatically each refresh, since that needs a separate live market-cap API "
        "yfinance doesn't provide for free. <b>Stablecoins excluded</b>: Tether (USDT) and USD Coin "
        "(USDC) both rank in the real top 10 by market cap, but a coin pegged to $1 carries no price "
        "signal -- its 52wk range / z-score would just be noise around the peg, not a real stretch "
        "indicator. All prices are direct USD quotes (crypto is quoted natively in USD on Yahoo "
        "Finance, e.g. <code>BTC-USD</code>) -- no FX conversion involved or needed.</p>"
    )
    return widgets.VBox([note, render_simple_category(CRYPTO, unit_prefix="$", decimals=2)])

BOND_CARD_STYLE = """
<div style="display:inline-block;width:335px;margin:8px;padding:14px 16px;
border:1px solid """ + LINE + """;border-radius:6px;background:""" + CARD + """;vertical-align:top;
font-family:-apple-system,Segoe UI,Helvetica,Arial,sans-serif">
  <div style="font-weight:700;font-size:15px;margin-bottom:8px;color:""" + INK + """">{country} <span style="color:""" + MUTED + """;font-weight:400;font-size:12px">{code}</span></div>
  {rows}
</div>
"""

def bond_bar_row(label, value, is_real, color, max_scale=8.0):
    if value is None:
        pct, val_label = 0, f"<span style='color:{MUTED}'>n/a</span>"
    else:
        pct = max(2, min(100, (value / max_scale) * 100))
        val_label = f"{value:.2f}%" + ("" if is_real else " ~")
    return f"""
    <div style="margin:8px 0">
      <div style="font-size:11px;color:{MUTED};letter-spacing:.02em;margin-bottom:3px">{label}</div>
      <div style="display:flex;align-items:center;gap:8px">
        <div style="flex:1;background:#EEECE3;border-radius:3px;height:10px;overflow:hidden">
          <div style="width:{pct}%;background:{color};height:100%"></div>
        </div>
        <div style="width:70px;text-align:right;font-weight:700;font-size:13px;color:{INK}">{val_label}</div>
      </div>
    </div>
    """

def render_bonds():
    # US: SHORT/MEDIUM/LONG are real, live points from Yahoo/CBOE, already
    # quoted directly as % (no /10 scaling -- see fetch_yield docstring).
    us_short_d = fetch_yield("^FVX")   # 5-year  -> "short" bucket
    us_med_d   = fetch_yield("^TNX")   # 10-year -> "medium" bucket
    us_long_d  = fetch_yield("^TYX")   # 30-year -> "long" bucket
    us_short_v, us_med_v, us_long_v = us_short_d.get("yield"), us_med_d.get("yield"), us_long_d.get("yield")

    # US 2-Year -- added 2026-09-18. Yahoo/CBOE's own yield-index family has
    # no 2-year maturity (only 13-week/5-year/10-year/30-year), so this one
    # point comes from FRED's DGS2 instead (see TE_2Y_* constants' comments
    # for the full source rundown) -- real and daily, but typically 1-2
    # business days behind, unlike the other three US points above.
    us_2y_d = fetch_fred_yield(US_2Y_FRED_SERIES)
    us_2y_v = us_2y_d.get("yield")

    # Real US curve shape used ONLY as a disclosed estimation method: for
    # countries where no free live short/long-maturity source exists, shift
    # that country's real 10-year FRED point by the real US curve's own
    # short/long spread on the same day. Explicit, labeled estimate (~).
    short_spread = (us_short_v - us_med_v) if None not in (us_short_v, us_med_v) else None
    long_spread  = (us_long_v  - us_med_v) if None not in (us_long_v,  us_med_v) else None
    y2_spread    = (us_2y_v    - us_med_v) if None not in (us_2y_v,    us_med_v) else None

    # fred_key=None -> United States (yfinance). fred_key="CHINA_TE" -> use
    # the TradingEconomics scrape (no FRED fallback available for China's
    # medium bucket -- not an OECD member, no FRED series). Every other key
    # names a BONDS_INTL/BONDS_INTL_TE entry -- for the MEDIUM (10Y) bucket,
    # TE is tried first (live, daily), falling back to FRED's real but
    # ~6-8-week-lagged monthly point only if TE fails that refresh. For the
    # SHORT (5Y) and LONG (30Y) buckets (all countries incl. China): each
    # tries a real scrape of that country's own 5Y/30Y TE page first (see
    # _te_bucket_value below), falling back to the old US-curve-spread
    # estimate only if that specific maturity's own scrape fails.
    countries = [
        ("United States",  "US", None),
        ("European Union", "EU", "Euro Area 10Y"),
        ("Germany",        "DE", "Germany 10Y"),
        ("France",         "FR", "France 10Y"),
        ("China",          "CN", "CHINA_TE"),
        ("Switzerland",    "CH", "Switzerland 10Y"),
        ("Japan",          "JP", "Japan 10Y"),
        ("United Kingdom", "GB", "UK 10Y"),
    ]

    def _te_bucket_value(url_slug, bucket_key, estimate_base, estimate_spread):
        """One 2Y/SHORT/LONG bucket for a non-US country: real TE scrape of
        that maturity's own page first (Switzerland's SHORT bucket uses its
        2Y page instead of 5Y -- see TE_SHORT_OVERRIDES, no 5Y page exists);
        only on failure, falls back to estimate_base (the medium/10Y value
        already obtained, real or FRED-fallback) shifted by the real US
        curve's own same-bucket spread that day -- the old estimation
        method, now a last resort instead of the default, and disabled
        entirely for Switzerland's LONG bucket (TE_LONG_NO_ESTIMATE --
        confirmed materially wrong for that country specifically).
        Returns (value, is_real, as-of-date).

        bucket_key="y2" (added 2026-09-18): uses TE_2Y_PAGE/TE_2Y_LABEL_*
        instead of TE_MATURITY_PAGES, since the 2-year bucket isn't one of
        the three original SHORT/MEDIUM/LONG buckets -- see those
        constants' own comments for why the label varies by country.
        Callers skip this entirely for Switzerland (reuses its already-
        fetched SHORT-bucket value) and Euro Area (TE_2Y_UNAVAILABLE -- no
        2-year page exists), so this function doesn't need to know about
        either special case itself."""
        if bucket_key == "y2":
            maturity_label = TE_2Y_LABEL_OVERRIDES.get(url_slug, TE_2Y_LABEL_DEFAULT)
            d = fetch_bond_yield_te(url_slug, maturity_label, TE_2Y_PAGE)
            if d.get("ok"):
                return d.get("yield"), True, d.get("date")
            if None not in (estimate_base, estimate_spread):
                return estimate_base + estimate_spread, False, None
            return None, False, None
        if bucket_key == "s" and url_slug in TE_SHORT_OVERRIDES:
            page_path, maturity_label, _tag = TE_SHORT_OVERRIDES[url_slug]
        else:
            page_path, maturity_label = TE_MATURITY_PAGES[bucket_key]
        d = fetch_bond_yield_te(url_slug, maturity_label, page_path)
        if d.get("ok"):
            return d.get("yield"), True, d.get("date")
        if bucket_key == "l" and url_slug in TE_LONG_NO_ESTIMATE:
            return None, False, None
        if None not in (estimate_base, estimate_spread):
            return estimate_base + estimate_spread, False, None
        return None, False, None

    def _y2_note(y2_v, y2_real, y2_date, real_source_label="TradingEconomics live scrape",
                 unavailable_reason=None):
        """One sentence describing the 2Y bucket's source/freshness --
        shared phrasing across all of render_bonds()'s per-country branches
        (US/China/Switzerland/other TE countries), added 2026-09-18."""
        if y2_real:
            return f"2Y as of {y2_date} ({real_source_label})."
        if unavailable_reason:
            return f"2Y unavailable -- {unavailable_reason}"
        if y2_v is not None:
            return ("2Y estimated from the 10Y point using the real US curve's own same-day spread "
                    "(TradingEconomics' own 2Y page unavailable this refresh).")
        return "2Y unavailable this refresh."

    def _bucket_note(short_v, short_real, short_date, long_v, long_real, long_date,
                      short_tag="5Y", long_no_estimate=False):
        bits = []
        if short_real:
            bits.append(f"{short_tag} as of {short_date} (TradingEconomics live scrape).")
        elif short_v is not None:
            bits.append(f"{short_tag} estimated from the 10Y point using the real US curve's same-day "
                        f"spread (TE's own {short_tag} page unavailable this refresh).")
        else:
            bits.append(f"{short_tag} unavailable this refresh.")
        if long_real:
            bits.append(f"30Y as of {long_date} (TradingEconomics live scrape).")
        elif long_no_estimate:
            bits.append("30Y unavailable -- TradingEconomics has no 30-year page for this country, and "
                         "the usual US-curve-spread estimate was checked against a real published figure "
                         "and confirmed meaningfully wrong here, so no number is shown rather than a "
                         "misleading one.")
        elif long_v is not None:
            bits.append("30Y estimated from the 10Y point using the real US curve's same-day "
                         "spread (TE's own 30Y page unavailable this refresh).")
        else:
            bits.append("30Y unavailable this refresh.")
        return " ".join(bits)

    cards = []
    for name, code, fred_key in countries:
        source_note = None
        short_tag = "5Y"  # display tag for the SHORT bucket row/note; overridden below (Switzerland: 2Y)
        if fred_key is None:  # United States
            short_v, short_real = us_short_v, True
            med_v,   med_real   = us_med_v,   True
            long_v,  long_real  = us_long_v,  True
            y2_v, y2_real, y2_date = us_2y_v, us_2y_d.get("ok", False), us_2y_d.get("date")
            source_note = _y2_note(
                y2_v, y2_real, y2_date,
                real_source_label=("FRED's DGS2 series -- a real, daily H.15 figure, but typically 1-2 "
                                    "business days behind today, unlike the other three points above "
                                    "which are live via Yahoo/CBOE (whose own yield-index family has no "
                                    "2-year maturity)"),
            )
        elif fred_key == "CHINA_TE":
            med_page, med_label = TE_MATURITY_PAGES["m"]
            china_med_d = fetch_bond_yield_te("china", med_label, med_page)
            med_v = china_med_d.get("yield")
            med_real = china_med_d.get("ok")
            med_date = china_med_d.get("date")

            short_v, short_real, short_date = _te_bucket_value("china", "s", med_v, short_spread)
            long_v,  long_real,  long_date  = _te_bucket_value("china", "l", med_v, long_spread)
            y2_v, y2_real, y2_date = _te_bucket_value("china", "y2", med_v, y2_spread)

            if med_v is None:
                source_note = (
                    "TradingEconomics scrape unavailable this refresh (page format changed, blocked, "
                    "or offline) -- no fallback exists for China (not an OECD member, no FRED series) "
                    "-- try Refresh again later."
                )
            else:
                source_note = (
                    _y2_note(y2_v, y2_real, y2_date) + " "
                    + f"10Y as of {med_date} (TradingEconomics live scrape, not an official API -- "
                    "see notes). "
                    + _bucket_note(short_v, short_real, short_date, long_v, long_real, long_date, short_tag)
                )
        else:
            te_slug = BONDS_INTL_TE[fred_key]
            short_tag = TE_SHORT_OVERRIDES.get(te_slug, (None, None, "5Y"))[2]
            long_no_estimate = te_slug in TE_LONG_NO_ESTIMATE
            med_page, med_label = TE_MATURITY_PAGES["m"]
            te_d = fetch_bond_yield_te(te_slug, med_label, med_page)
            if te_d.get("ok"):
                med_v = te_d.get("yield")
                med_real = True
                med_date = te_d.get("date")
                med_bit = f"10Y as of {med_date} (TradingEconomics live scrape, not an official API -- see notes)."
            else:
                fred_d = fetch_fred_yield(BONDS_INTL[fred_key])
                med_v = fred_d.get("yield")
                med_real = False  # real datapoint, but not today's -- see note
                med_date = fred_d.get("date")
                if med_v is not None:
                    med_bit = (
                        f"<span style='color:{AMBER};font-weight:600'>TradingEconomics 10Y unavailable "
                        f"this refresh</span> -- fell back to FRED's OECD series, 10Y as of {med_date} "
                        "(not today's rate: FRED publishes monthly with a ~6-8 week real-world lag)."
                    )
                else:
                    med_bit = None

            short_v, short_real, short_date = _te_bucket_value(te_slug, "s", med_v, short_spread)
            long_v,  long_real,  long_date  = _te_bucket_value(te_slug, "l", med_v, long_spread)

            # 2-year bucket -- added 2026-09-18. Switzerland already scrapes
            # a real 2-year point for its SHORT bucket above (see
            # TE_SHORT_OVERRIDES), so reuse that value instead of a second,
            # redundant scrape of the exact same TE page. Euro Area has no
            # 2-year TE page at all (TE_2Y_UNAVAILABLE).
            if te_slug == "switzerland":
                y2_v, y2_real, y2_date = short_v, short_real, short_date
                y2_bit = _y2_note(
                    y2_v, y2_real, y2_date,
                    real_source_label=("TradingEconomics live scrape -- same reading as the SHORT bucket "
                                        "below, since TradingEconomics has no 5-year page for Switzerland"),
                )
            elif te_slug in TE_2Y_UNAVAILABLE:
                y2_v, y2_real, y2_date = None, False, None
                y2_bit = _y2_note(None, False, None,
                                   unavailable_reason="TradingEconomics has no 2-year page for this country.")
            else:
                y2_v, y2_real, y2_date = _te_bucket_value(te_slug, "y2", med_v, y2_spread)
                y2_bit = _y2_note(y2_v, y2_real, y2_date)

            if med_bit is None:
                source_note = (
                    "Both TradingEconomics and FRED fetches failed this refresh for the medium (10Y) "
                    "point -- no value shown, try Refresh again later."
                )
            else:
                source_note = (y2_bit + " " + med_bit + " "
                                + _bucket_note(short_v, short_real, short_date, long_v, long_real,
                                               long_date, short_tag, long_no_estimate))

        rows = (
            bond_bar_row("2Y", y2_v, y2_real, GREEN)
            + bond_bar_row(f"SHORT &middot; ~{short_tag}", short_v, short_real, AMBER)
            + bond_bar_row("MEDIUM &middot; ~10Y", med_v,   med_real,   RED)
            + bond_bar_row("LONG &middot; ~30Y",   long_v,  long_real,  INK)
        )
        if source_note:
            rows += f"<p style='color:{MUTED};font-size:11px;margin:6px 0 0'>{source_note}</p>"
        cards.append(BOND_CARD_STYLE.format(country=name, code=code, rows=rows))

    note = widgets.HTML(
        f"<p style='color:{MUTED};font-size:12px;line-height:1.5;max-width:900px'>"
        "Government bond yields by maturity bucket &mdash; 2-year, short (~5Y), medium (~10Y), long (~30Y). "
        "Shows the current yield level, not a return over time, so the horizon toggle doesn't apply here "
        "(hidden on this tab). <b>United States</b> is real and live on SHORT/MEDIUM/LONG (Yahoo/CBOE, daily); "
        "its <b>2-year</b> point comes from FRED's DGS2 series instead (added 2026-09-18 &mdash; real and "
        f"daily, but <span style='color:{AMBER};font-weight:600'>typically 1-2 business days behind</span>, "
        "since Yahoo/CBOE's own yield-index family has no 2-year maturity to pull from). "
        "For Germany/France/UK/Japan/Switzerland/Euro Area <b>and</b> China, <b>every</b> bucket first tries a "
        "real, live, daily figure read directly from that maturity's own TradingEconomics page (fixed "
        "2026-08-31 -- short/long used to be estimated from the medium point via the US curve's own spread, "
        "which was badly wrong for countries whose curve shape differs a lot from the US's, most visibly "
        "Japan) &mdash; not an official API, so treat these as more fragile/gray-area than every yfinance/FRED "
        "source elsewhere in this notebook (page-wording or blocking changes can silently break a scrape; see "
        "each card's own note for exactly what happened this refresh). "
        f"If the <b>medium (10Y)</b> scrape fails, Germany/France/UK/Japan/Switzerland/Euro Area fall back to "
        f"FRED's OECD long-term rate series instead &mdash; real data, but <span style='color:{AMBER};"
        "font-weight:600'>NOT live</span>: OECD only publishes it monthly with a real-world lag of roughly "
        "6-8 weeks (confirmed directly against FRED while building this: Germany's newest available point "
        "was dated 2026-06-01 on 2026-08-24, not August). China has no FRED equivalent at all (not an OECD "
        "member), so a failed medium-bucket scrape there just shows \"unavailable\" with no fallback. "
        f"If a <b>short (5Y) or long (30Y)</b> scrape specifically fails, that one bucket falls back to the "
        f"old estimate &mdash; the country's medium-term point shifted by the real US curve's own same-day "
        f"spread &mdash; marked <span style='color:{AMBER};font-weight:600'>~</span>. Each card's own note "
        "states the exact date and source behind every bucket &mdash; check that before comparing against a "
        "live source elsewhere. <b>Switzerland is special-cased</b> (fixed 2026-09-02): TradingEconomics has "
        "no 5-year or 30-year page for Switzerland at all (both URLs redirect to TE's own homepage) &mdash; "
        "only 2Y and 10Y are tracked there. So Switzerland's SHORT bucket reads its real 2Y page instead "
        "(labeled <b>~2Y</b>, not 5Y, on that card), and its LONG bucket shows <b>unavailable</b> rather "
        "than the usual estimate &mdash; that estimate was checked against a real published Swiss 30Y figure "
        "and confirmed off by roughly 0.4 points, the same kind of error the pre-fix Japan estimate had. "
        "Switzerland's new <b>2Y</b> bucket (added 2026-09-18) shows the exact same reading as its SHORT "
        "bucket above, rather than a second scrape of the same page &mdash; both exist for the same reason. "
        "<b>Euro Area's 2Y</b> bucket shows <b>unavailable</b>: confirmed live that TradingEconomics has no "
        "2-year page for Euro Area at all (the URL redirects to TE's own homepage), and with no independently-"
        "verified estimate basis for it, this shows no number rather than an unchecked one.</p>"
    )
    grid = widgets.HTML(f"<div>{''.join(cards)}</div>")
    return widgets.VBox([note, grid])

def render_top_performers():
    h = horizon.value
    spx_pct = fetch_returns("^GSPC").get(h)
    scored = []
    breadth_yes, breadth_total = 0, 0
    for sym in ALL_STOCKS:
        d = fetch_returns(sym)
        if d.get("above_200dma") is not None:
            breadth_total += 1
            if d["above_200dma"]:
                breadth_yes += 1
        pct = d.get(h)
        if pct is None:
            continue
        scored.append((sym, d, pct))
    scored.sort(key=lambda x: x[2], reverse=True)
    top = scored[:9]
    rows = []
    for sym, d, pct in top:
        extra = fmt_signal_line(d) + fmt_relstrength(pct, spx_pct)
        rel = (pct - spx_pct) if (pct is not None and spx_pct is not None) else None
        rows.append({
            "name": NAME_MAP.get(sym, sym), "ticker": sym,
            "price": fmt_price(d["price"]), "pct": fmt_pct(pct), "extra": extra,
            "relstrength_pp": rel,
        })
    breadth_html = ""
    if breadth_total:
        breadth_pct = breadth_yes / breadth_total * 100.0
        breadth_color = GREEN if breadth_pct >= 50 else RED
        breadth_html = (
            f"<p style='color:{MUTED};font-size:12px;margin:0 0 6px'><b>Market Breadth:</b> "
            f"<span style='color:{breadth_color};font-weight:700'>{breadth_pct:.0f}%</span> of "
            f"{breadth_total} tracked stocks are trading above their own 200-day moving average -- "
            "independent of any single index's level, this is a market-health gauge: broad participation "
            "(high %) supports a rally's durability, while a shrinking % while the index itself keeps "
            "rising can flag a narrow, top-heavy market propped up by a handful of large names.</p>"
        )
    note = widgets.HTML(f"<p style='color:{MUTED};font-size:12px'>Top 9 by real return this horizon, "
                         f"ranked live across {len(ALL_STOCKS)} tracked stocks. \"vs S&P 500\" is excess "
                         "return over the same horizon, in percentage points -- separates real "
                         "outperformance from a market-wide rally.</p>")
    children = ([widgets.HTML(breadth_html)] if breadth_html else []) + [note, render_grid(rows)]
    return widgets.VBox(children)

def render_industries():
    h = horizon.value
    spx_pct = fetch_returns("^GSPC").get(h)
    sector = industries_state["sector"]
    if sector is None:
        rows = []
        for name, etf in SECTOR_ETFS.items():
            d = fetch_returns(etf)
            pct = d.get(h)
            extra = fmt_signal_line(d) + fmt_relstrength(pct, spx_pct)
            rel = (pct - spx_pct) if (pct is not None and spx_pct is not None) else None
            rows.append({"name": name, "ticker": etf, "price": fmt_price(d["price"]),
                         "pct": fmt_pct(pct), "extra": extra, "relstrength_pp": rel})
        grid = render_grid(rows)
        buttons = [widgets.Button(description=name, layout=widgets.Layout(width="190px", margin="2px"))
                   for name in SECTOR_ETFS]
        def make_handler(sector_name):
            def _handler(b):
                industries_state["sector"] = sector_name
                refresh_view("Industries")
            return _handler
        for btn, name in zip(buttons, SECTOR_ETFS):
            btn.on_click(make_handler(name))
        btn_box = widgets.HBox(buttons, layout=widgets.Layout(flex_flow="row wrap"))
        header = widgets.HTML("<b>Click a sector to drill into its top 6, ranked live by real return for this horizon:</b>")
        return widgets.VBox([header, btn_box, grid])
    else:
        pool = SECTOR_POOLS[sector]
        scored = []
        for sym in pool:
            d = fetch_returns(sym)
            pct = d.get(h)
            if pct is None:
                continue
            scored.append((sym, d, pct))
        scored.sort(key=lambda x: x[2], reverse=True)
        top6 = scored[:6]
        rows = []
        for sym, d, pct in top6:
            extra = fmt_signal_line(d) + fmt_relstrength(pct, spx_pct)
            rel = (pct - spx_pct) if (pct is not None and spx_pct is not None) else None
            rows.append({"name": NAME_MAP.get(sym, sym), "ticker": sym,
                         "price": fmt_price(d["price"]), "pct": fmt_pct(pct), "extra": extra,
                         "relstrength_pp": rel})
        grid = render_grid(rows)
        back_btn = widgets.Button(description="← Back to all sectors")
        def on_back(b):
            industries_state["sector"] = None
            refresh_view("Industries")
        back_btn.on_click(on_back)
        header = widgets.HTML(f"<h3 style='margin-bottom:4px;color:{INK}'>{sector} — top 6 of {len(pool)} tracked, "
                               f"ranked live by real return for this horizon</h3>")
        return widgets.VBox([back_btn, header, grid])

# Smart Money Cash Position widgets -- created once at module scope (like
# the Chart tab's widgets) so the loaded result and button state survive
# being re-parented into a fresh VBox on every re-render.
smart_money_btn = widgets.Button(description="\U0001f4ca Load Smart Money Cash Signal (~450MB, few min)",
                                  layout=widgets.Layout(width="360px"))
smart_money_status = widgets.HTML(
    value=f"<span style='color:{MUTED};font-size:12px'>Not loaded -- click to fetch the latest quarter's "
          f"SEC N-PORT bulk data (~450MB one-time download, cached after that).</span>"
)
smart_money_output = widgets.Output()

def _render_smart_money_result():
    with smart_money_output:
        clear_output(wait=True)
        d = _CACHE.get("smart_money_cash")
        if not d or not d.get("ok"):
            return
        rows = [{
            "name": "Loose Cash (not itemized as a holding)", "ticker": d["quarter_label"],
            "price": f"{d['weighted_pct']:.2f}%",
            "pct": f"<span style='font-size:11px;color:{MUTED}'>asset-weighted, {d['n_funds']:,} funds</span>",
            "extra": (f"<div style='font-size:11px;color:{MUTED};margin-top:4px'>Simple avg "
                      f"{d['simple_pct']:.2f}% &middot; ${d['total_assets_usd'] / 1e12:.2f}T covered "
                      f"&middot; as of {d['report_date']}</div>"),
        }]
        if d.get("near_cash_ok"):
            cats = ", ".join(d["near_cash_cats_used"]) if d.get("near_cash_cats_used") else "n/a"
            rows.append({
                "name": "Near-Cash (+ money funds, repo)", "ticker": d["quarter_label"],
                "price": f"{d['near_cash_weighted_pct']:.2f}%",
                "pct": f"<span style='font-size:11px;color:{MUTED}'>asset-weighted</span>",
                "extra": (f"<div style='font-size:11px;color:{MUTED};margin-top:4px'>Simple avg "
                          f"{d['near_cash_simple_pct']:.2f}% &middot; categories matched: {cats}</div>"),
            })
            display(render_grid(rows))
        else:
            display(render_grid(rows))
            cats_found = d.get("near_cash_cats_used")
            note = (
                "Near-cash breakdown unavailable this run -- couldn't confidently identify the "
                "short-term-investment-vehicle / repurchase-agreement categories in the holdings data "
                + (f"(saw categories: {', '.join(str(c) for c in cats_found[:15])}{'...' if len(cats_found) > 15 else ''})"
                   if cats_found else "(the holdings table itself may not have loaded)")
                + " -- flag this back so the matching logic in _classify_cash_like_asset_cats() can be fixed. "
                "The loose-cash figure above is unaffected."
            )
            display(widgets.HTML(f"<p style='color:{AMBER};font-size:11px;max-width:700px'>{note}</p>"))

def on_smart_money_click(b):
    def progress(msg):
        smart_money_status.value = f"<span style='color:{MUTED};font-size:12px'>{msg}</span>"
    d = fetch_smart_money_cash(progress_cb=progress)
    if d["ok"]:
        smart_money_status.value = (
            f"<span style='color:{GREEN};font-size:12px'>Loaded {d['quarter_label']} -- {d['n_funds']:,} "
            f"funds, ${d['total_assets_usd'] / 1e12:.2f}T covered, as of {d['report_date']}.</span>"
        )
    else:
        smart_money_status.value = f"<span style='color:{RED};font-size:12px'>Failed: {d['error']}</span>"
    _render_smart_money_result()

smart_money_btn.on_click(on_smart_money_click)

# Insider Form 4 aggregate widgets -- same module-scope pattern as Smart
# Money above, so state survives re-renders. Lighter download (~8-15MB vs
# ~450MB) but still opt-in, to keep the normal Refresh cycle fast.
insider_btn = widgets.Button(description="\U0001f4dd Load Insider Buy/Sell Signal (~10MB, seconds)",
                              layout=widgets.Layout(width="360px"))
insider_status = widgets.HTML(
    value=f"<span style='color:{MUTED};font-size:12px'>Not loaded -- click to fetch the latest quarter's "
          f"SEC insider transaction data (Form 3/4/5 bulk set).</span>"
)
insider_output = widgets.Output()

def _render_insider_result():
    with insider_output:
        clear_output(wait=True)
        d = _CACHE.get("insider_aggregate")
        if not d or not d.get("ok"):
            return
        if d.get("data_quality_flag"):
            buy_pct_html = f"<span style='color:{MUTED}'>unavailable</span>"
            extra_flag = (f"<div style='font-size:11px;color:{AMBER};margin-top:4px'>&#9888; "
                          f"{d['data_quality_flag']}</div>")
        else:
            buy_pct_html = f"{d['buy_pct_of_dollar_volume']:.1f}%"
            extra_flag = (f"<div style='font-size:11px;color:{MUTED};margin-top:4px'>"
                          f"${d['total_buy_usd']/1e9:.2f}B bought &middot; ${d['total_sell_usd']/1e9:.2f}B sold</div>")
        rows = [{
            "name": "Insider Buying Share of $ Volume", "ticker": d["quarter_label"],
            "price": buy_pct_html,
            "pct": f"<span style='font-size:11px;color:{MUTED}'>{d['n_transactions']:,} open-market trades</span>",
            "extra": extra_flag,
        }, {
            "name": "Companies: Net Buying vs Net Selling", "ticker": d["quarter_label"],
            "price": f"{d['n_companies_net_buying']:,} : {d['n_companies_net_selling']:,}",
            "pct": f"<span style='font-size:11px;color:{MUTED}'>net buying : net selling companies</span>",
            "extra": "",
        }]
        display(render_grid(rows))

def on_insider_click(b):
    def progress(msg):
        insider_status.value = f"<span style='color:{MUTED};font-size:12px'>{msg}</span>"
    d = fetch_insider_aggregate(progress_cb=progress)
    if d["ok"]:
        insider_status.value = (
            f"<span style='color:{GREEN};font-size:12px'>Loaded {d['quarter_label']} -- "
            f"{d['n_transactions']:,} open-market buy/sell transactions.</span>"
        )
    else:
        insider_status.value = f"<span style='color:{RED};font-size:12px'>Failed: {d['error']}</span>"
    _render_insider_result()

insider_btn.on_click(on_insider_click)

# Insider trend chart widgets -- plots the last ~5 quarters (roughly the
# last year) instead of just the latest snapshot. Heavier (~50-75MB vs
# ~10-15MB for one quarter) since it downloads multiple quarters, but each
# quarter is cached independently so re-plotting or extending the window
# later doesn't re-fetch quarters already loaded.
insider_trend_btn = widgets.Button(description="\U0001f4c8 Plot Insider Trend (Last ~5 Quarters, ~50-75MB)",
                                    layout=widgets.Layout(width="380px"))
insider_trend_status = widgets.HTML(
    value=f"<span style='color:{MUTED};font-size:12px'>Not loaded -- click to fetch the last ~5 quarters "
          f"(~10-15MB each, cached per-quarter) and plot the trend.</span>"
)
insider_trend_output = widgets.Output()

def _render_insider_trend(results):
    with insider_trend_output:
        clear_output(wait=True)
        if not results:
            print("No quarters loaded -- try again, or SEC's bulk files may be temporarily unavailable.")
            return
        labels = [d["quarter_label"] for d in results]
        nan = float("nan")
        flagged = [d["quarter_label"] for d in results if d.get("data_quality_flag")]
        # None -> NaN so matplotlib draws a gap instead of erroring or plotting a
        # misleading 0. A flagged quarter's buy_pct is already forced to None
        # upstream in _fetch_insider_quarter(), so it naturally shows as a gap here.
        buy_pct = [(d["buy_pct_of_dollar_volume"] if d["buy_pct_of_dollar_volume"] is not None else nan)
                   for d in results]
        net_buy_share = []
        for d in results:
            total_companies = d["n_companies_net_buying"] + d["n_companies_net_selling"]
            net_buy_share.append(d["n_companies_net_buying"] / total_companies * 100.0 if total_companies > 0 else nan)

        x = list(range(len(labels)))
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5.6), dpi=110, sharex=True)
        ax1.plot(x, buy_pct, color=RED, linewidth=1.8, marker="o")
        ax1.set_title("Insider Buying Share of $ Volume (%)", fontsize=11, color=INK, loc="left", fontweight="bold")
        ax2.plot(x, net_buy_share, color=RED, linewidth=1.8, marker="o")
        ax2.set_title("Companies Net Buying (% of companies with insider activity)",
                       fontsize=11, color=INK, loc="left", fontweight="bold")
        ax2.set_xticks(x)
        ax2.set_xticklabels(labels, rotation=30, ha="right", fontsize=9, color=MUTED)
        for ax in (ax1, ax2):
            ax.tick_params(colors=MUTED, labelsize=9)
            for spine in ("top", "right"):
                ax.spines[spine].set_visible(False)
            for spine in ("left", "bottom"):
                ax.spines[spine].set_color(LINE)
            ax.grid(axis="y", color=LINE, linewidth=0.6, alpha=0.6)
            ax.set_facecolor(PAPER)
        fig.patch.set_facecolor(PAPER)
        fig.tight_layout()

        buf = io.BytesIO()
        fig.savefig(buf, format="png", facecolor=fig.get_facecolor())
        plt.close(fig)
        buf.seek(0)
        display(widgets.Image(value=buf.getvalue(), format="png", layout=widgets.Layout(width="720px")))
        if flagged:
            display(widgets.HTML(
                f"<p style='color:{AMBER};font-size:11px;max-width:720px;margin-top:4px'>&#9888; "
                f"Gap(s) at {', '.join(flagged)}: the SEC data for that quarter had no usable transactions "
                "on one side of the buy/sell split after filtering -- flagged as a likely parsing artifact "
                "and excluded rather than shown as a false 0%/100%.</p>"
            ))

        # Raw underlying numbers for every quarter, always shown (not just
        # flagged ones) -- lets you see exactly why a quarter reads the way
        # it does (e.g. a genuine but extreme 97% vs. a suspicious exact
        # 100.0%) instead of trusting the headline % alone.
        rows_html = "".join(
            f"<tr><td style='padding:3px 10px 3px 0'>{d['quarter_label']}</td>"
            f"<td style='padding:3px 10px;text-align:right'>${d['total_buy_usd']/1e6:,.1f}M</td>"
            f"<td style='padding:3px 10px;text-align:right'>${d['total_sell_usd']/1e6:,.1f}M</td>"
            f"<td style='padding:3px 10px;text-align:right'>{d['n_buy_transactions']:,}</td>"
            f"<td style='padding:3px 10px;text-align:right'>{d['n_sell_transactions']:,}</td>"
            f"<td style='padding:3px 0 3px 10px'>{d.get('data_quality_flag') or ''}</td></tr>"
            for d in results
        )
        display(widgets.HTML(
            f"<table style='font-size:11px;color:{MUTED};margin-top:8px;border-collapse:collapse'>"
            f"<tr style='font-weight:600;color:{INK}'><td style='padding:3px 10px 3px 0'>Quarter</td>"
            "<td style='padding:3px 10px;text-align:right'>$ Bought</td>"
            "<td style='padding:3px 10px;text-align:right'>$ Sold</td>"
            "<td style='padding:3px 10px;text-align:right'>Buy txns</td>"
            "<td style='padding:3px 10px;text-align:right'>Sell txns</td>"
            "<td style='padding:3px 0 3px 10px'>Flag</td></tr>"
            f"{rows_html}</table>"
        ))

def on_insider_trend_click(b):
    def progress(msg):
        insider_trend_status.value = f"<span style='color:{MUTED};font-size:12px'>{msg}</span>"
    results = fetch_insider_history(n_quarters=5, progress_cb=progress)
    if results:
        insider_trend_status.value = (
            f"<span style='color:{GREEN};font-size:12px'>Loaded {len(results)} quarters "
            f"({results[0]['quarter_label']} &ndash; {results[-1]['quarter_label']}).</span>"
        )
    else:
        insider_trend_status.value = f"<span style='color:{RED};font-size:12px'>Failed to load any quarters.</span>"
    _render_insider_trend(results)

insider_trend_btn.on_click(on_insider_trend_click)


def render_cash_liquidity():
    gauge_rows = []
    for name, (series_id, scale) in CASH_GAUGE_FRED.items():
        zs_days, zs_min = (365 * 5, 6) if series_id == "MMMFFAQ027S" else (365, 20)
        d = fetch_fred_level(series_id, zscore_lookback_days=zs_days, zscore_min_points=zs_min)
        pct_html = (f"<span style='font-size:11px;color:{MUTED}'>YoY </span>" + fmt_pct(d.get("m")))
        extra = fmt_period_changes(d, "1mo") + fmt_signal_line(d)
        gauge_rows.append({
            "name": name, "ticker": series_id,
            "price": fmt_big_dollars(d["value"], scale),
            "pct": pct_html, "extra": extra, "zscore": d.get("zscore"),
        })
    gauge_grid = render_grid(gauge_rows)

    ratio_rows = []
    for name, (num_id, den_id, num_scale, den_scale) in CASH_GAUGE_RATIOS.items():
        zs_days, zs_min = (365 * 5, 6) if num_id == "MMMFFAQ027S" else (365, 20)
        d = fetch_fred_ratio(num_id, den_id, num_scale, den_scale,
                              zscore_lookback_days=zs_days, zscore_min_points=zs_min)
        extra = ""
        if d["ok"]:
            bits = []
            for key, label in (("s", "1mo"), ("l", "5yr")):
                v = d.get(key)
                if v is not None:
                    bits.append(f"{label} {fmt_pp_change(v)}")
            if bits:
                extra += f"<div style='font-size:11px;color:{MUTED};margin-top:4px'>{' &middot; '.join(bits)}</div>"
            extra += fmt_signal_line(d)
        ratio_rows.append({
            "name": name, "ticker": f"{num_id} / {den_id}",
            "price": f"{d['value']:.3f}%" if d["value"] is not None else fmt_big_dollars(None),
            "pct": (f"<span style='font-size:11px;color:{MUTED}'>YoY </span>" + fmt_pp_change(d.get("m"))),
            "extra": extra, "zscore": d.get("zscore"),
        })
    ratio_grid = render_grid(ratio_rows)
    ratio_note = widgets.HTML(
        f"<p style='color:{MUTED};font-size:12px;max-width:820px;line-height:1.5'>The dollar figures "
        "above are nominal, so they drift upward over time just from inflation and monetary-base "
        "growth (M2 expansion) even with zero change in investor behavior -- a raw \"+12% YoY\" on a "
        "cash aggregate can't tell you whether people are actually getting more cautious or the money "
        "supply just grew. Dividing by M2 (FRED's <code>M2SL</code>, confirmed live) nets most of that "
        "out: if MMF assets are growing <i>faster</i> than M2 itself, that's real evidence of a "
        "relative shift toward cash, not just nominal drift. <b>Retail MMF Share of M2</b> is a clean "
        "read since retail MMF balances are technically already a component of M2's own definition "
        "(so this is a compositional share, not two independent series). <b>Total MMF Industry vs M2</b> "
        "adds institutional money too, which isn't part of M2 itself, so treat it as a broader "
        "cross-check rather than an exact share. Changes shown are in percentage points (pp), not %, "
        "to avoid confusion on a metric that's already a percentage. Also worth remembering separately: "
        "MMF asset growth is also driven by MMF yields vs. bank deposit yields (yield-chasing flows), "
        "which isn't a risk-sentiment signal at all -- this ratio doesn't isolate that effect.</p>"
    )

    gauge_note = widgets.HTML(
        f"<p style='color:{MUTED};font-size:12px;max-width:820px;line-height:1.5'>Aggregate "
        "cash-on-the-sidelines proxies, all free/live via FRED. <b>Retail MMF Assets</b> (weekly) and "
        "<b>Total MMF Industry Assets</b> (quarterly, retail+institutional combined via the Fed's Z.1 "
        "Flow of Funds) both track how much investor cash is parked in money market funds -- rising = "
        "more dry powder sitting out. <b>Fed O/N Reverse Repo Usage</b> (daily) tracks cash parked "
        "directly at the Fed overnight -- it has fallen dramatically from its 2022&ndash;2023 peak to "
        "near-zero recently, itself a real regime change worth noting, not a data glitch. The Fed "
        "discontinued its <i>institutional-only</i> money fund series in 2021 and never replaced it, "
        "so there's no free live \"institutional cash alone\" figure -- only the combined quarterly "
        "total above. This tab ignores the horizon toggle (hidden here) since it mixes daily, weekly, "
        "and quarterly series -- each card shows its own YoY headline plus 1mo/5yr change instead.</p>"
    )

    company_rows = []
    for name, cik in COMPANY_CASH_CIKS.items():
        d = fetch_company_cash(cik)
        if d["ok"]:
            pct_html = (f"<span style='font-size:11px;color:{MUTED}'>YoY </span>" + fmt_pct(d.get("m")))
            extra = fmt_period_changes(d, "QoQ") + fmt_signal_line(d)
        else:
            pct_html = fmt_pct(None)
            extra = (f"<div style='font-size:11px;color:{MUTED};margin-top:5px'>unavailable this "
                     f"refresh ({d.get('error', 'unknown error')})</div>")
        company_rows.append({
            "name": name, "ticker": "CIK " + cik.lstrip("0"),
            "price": fmt_big_dollars(d["value"], 1e-9),
            "pct": pct_html, "extra": extra,
        })
    company_grid = render_grid(company_rows)
    company_note = widgets.HTML(
        f"<p style='color:{MUTED};font-size:12px;max-width:820px;line-height:1.5'>Cash &amp; "
        "equivalents from Berkshire's own SEC filings (free XBRL API, no key) -- quarterly/annual, "
        "lags roughly 40 days after each quarter-end. This figure will run lower than headline news "
        "numbers, which usually also include its short-term Treasury bill holdings on top of cash. "
        "<b>Unlike every other source in this notebook, this integration could not be live-tested</b> "
        "from the environment that built it (SEC EDGAR requests wouldn't complete there) -- it follows "
        "SEC's documented, stable API format and degrades cleanly to \"unavailable\" if something "
        "doesn't match, but treat your first real run as this feature's actual verification, and flag "
        "back if the number looks obviously wrong. Also: edit <code>SEC_USER_AGENT</code> near the top "
        "of this notebook with your own name/email -- SEC asks for a real identifying contact, not a "
        "placeholder.</p>"
    )

    liq_d = fetch_liquidity_stress()
    if liq_d["ok"]:
        liq_price_html = f"{liq_d['value']:.3e}"
        liq_extra = fmt_signal_line(liq_d)
        liq_extra += (f"<div style='font-size:10px;color:{MUTED};margin-top:3px'>median across "
                       f"{liq_d.get('n_stocks', '?')} stocks &middot; as of {liq_d.get('as_of', 'n/a')}</div>")
    else:
        liq_price_html = fmt_big_dollars(None)
        liq_extra = (f"<div style='font-size:11px;color:{MUTED};margin-top:5px'>unavailable this "
                      f"refresh ({liq_d.get('error', 'unknown error')})</div>")
    liq_row = [{
        "name": "Liquidity Stress (Amihud)", "ticker": f"median |r|/$Vol x1e6, {len(ALL_STOCKS)}-stock universe",
        "price": liq_price_html, "pct": fmt_pct(liq_d.get("s")), "extra": liq_extra,
        "zscore": liq_d.get("zscore"),
    }]
    liq_grid = render_grid(liq_row)
    liq_note = widgets.HTML(
        f"<p style='color:{MUTED};font-size:12px;max-width:820px;line-height:1.5'>"
        "<b>Added 2026-09-22</b>, after discussing whether the academic Pastor-Stambaugh liquidity "
        "factor could power a signal here -- it can't, for free: PS's own measure needs intraday, "
        "buyer/seller-signed trade data, and the user's own research separately found even the real "
        "factor is trailing/coincident with a recession or boom's liquidity conditions, not leading. "
        "This is Amihud's (2002) illiquidity ratio instead -- the well-known, much simpler cousin of PS "
        "(same underlying idea: price impact of trading), needing only daily close and volume already "
        "cached for every tracked stock, no new network calls. <b>|daily return| &divide; dollar volume</b>, "
        "&times;10<sup>6</sup> for readability (Amihud's own convention), computed per stock then "
        f"aggregated as the <b>median</b> across the {len(ALL_STOCKS)}-stock universe each day (median, "
        "not mean, so a few thin small/mid-caps can't swamp the reading) into one daily market-wide "
        "series -- % change, z-score, and 52wk range are all computed on that aggregate's own history, "
        "same treatment as every other derived signal in this notebook. <b>Rising = liquidity thinning</b> "
        "(the market is paying more in price impact to trade); falling = liquidity healthy. "
        "<b>Deliberately trailing/coincident, not predictive</b> -- same honest limitation as the real PS "
        "factor -- so treat this as confirmation/sizing of a liquidity-stress regime already suspected "
        "from faster gauges (VIX, HY OAS on the Signals tab), not an early warning of one.</p>"
    )

    smart_money_section = widgets.VBox([
        widgets.HTML(f"<h3 style='margin:18px 0 4px;color:{INK}'>Smart Money Cash Position (Fund Industry)</h3>"),
        widgets.HTML(
            f"<p style='color:{MUTED};font-size:12px;max-width:820px;line-height:1.5'>Aggregate "
            "cash-as-%-of-assets across SEC-registered N-PORT filers (mutual funds &amp; ETFs, money "
            "market funds excluded) -- a real, regulator-sourced proxy for the classic \"mutual fund "
            "cash ratio\" sentiment gauge: low = funds are fully invested with little dry powder "
            "(historically nearer market tops), high = more cash cushion sitting out (historically "
            "nearer troughs). Shows two figures: <b>Loose Cash</b> (cash not already itemized as a "
            "holding) and <b>Near-Cash</b>, which adds holdings tagged as short-term investment "
            "vehicles (money market funds, liquidity pools) or repurchase agreements -- large funds "
            "mostly hold their liquidity this way rather than as loose cash, so Near-Cash is the more "
            "complete read. The category codes used for that match aren't published anywhere, so this "
            "detects them from the real data each run and shows exactly what it matched -- if it can't "
            "confidently identify them, Near-Cash is simply left unavailable rather than guessed. "
            "<b>Heavy and opt-in only</b> -- built from SEC's free bulk N-PORT dataset, ~400-480MB per "
            "quarter (Near-Cash specifically also reads the largest table in that file, so expect "
            "noticeably more memory/time than Loose Cash alone) -- not part of the normal Refresh "
            "cycle. Covers <i>all</i> N-PORT filers, not just equity-only funds. Only the latest "
            "available quarter loads; no automatic historical trend.</p>"
        ),
        widgets.HBox([smart_money_btn]),
        smart_money_status,
        smart_money_output,
    ])

    insider_section = widgets.VBox([
        widgets.HTML(f"<h3 style='margin:18px 0 4px;color:{INK}'>Insider Buying vs Selling (Aggregate Form 4)</h3>"),
        widgets.HTML(
            f"<p style='color:{MUTED};font-size:12px;max-width:820px;line-height:1.5'>Aggregate "
            "open-market buying vs. selling by company officers, directors, and 10%+ owners, from SEC's "
            "free bulk Insider Transactions (Form 3/4/5) data set. Restricted to genuine open-market "
            "purchases/sales only (transaction codes P/S) -- option exercises, stock grants, gifts, and "
            "other non-discretionary codes are excluded since they don't reflect a real conviction call. "
            "<b>Insider Buying Share of $ Volume</b> is total $ bought over total $ bought+sold this "
            "quarter -- insiders structurally sell far more than they buy (routine diversification and "
            "compensation-driven sales), so this is normally well under 50%; watch the trend and extremes, "
            "not the absolute level. <b>Companies: Net Buying vs Net Selling</b> is breadth -- how many "
            "distinct companies had more $ bought than sold this quarter vs. the reverse -- a broad shift "
            "across many companies is a more robust signal than one or two large trades. Doesn't exclude "
            "10b5-1 pre-scheduled sale plans (not reliably flagged in the raw data) -- a real but rougher "
            "proxy. Much lighter than Smart Money Cash (~10-15MB vs ~450MB) but still opt-in to keep the "
            "normal Refresh cycle fast. The second button below plots both figures across the last ~5 "
            "quarters instead of just the latest snapshot -- heavier (~50-75MB total) since it downloads "
            "multiple quarters, but each quarter is cached individually, so re-plotting or extending the "
            "window later never re-fetches a quarter already loaded.</p>"
        ),
        widgets.HBox([insider_btn]),
        insider_status,
        insider_output,
        widgets.HBox([insider_trend_btn]),
        insider_trend_status,
        insider_trend_output,
    ])

    return widgets.VBox([
        widgets.HTML(f"<h3 style='margin:0 0 4px;color:{INK}'>Market-Wide Cash Gauge</h3>"),
        gauge_note, gauge_grid,
        widgets.HTML(f"<h4 style='margin:14px 0 4px;color:{INK}'>Normalized (as a share of M2)</h4>"),
        ratio_note, ratio_grid,
        widgets.HTML(f"<h3 style='margin:18px 0 4px;color:{INK}'>Trading Liquidity Stress</h3>"),
        liq_note, liq_grid,
        smart_money_section,
        insider_section,
        widgets.HTML(f"<h3 style='margin:18px 0 4px;color:{INK}'>Berkshire Hathaway Cash (Buffett's Cash Pile)</h3>"),
        company_note, company_grid,
    ])


# ---------------------------------------------------------------------------
# Chart tab -- search any tracked ticker/yield and plot its real history.
# Widgets are created once here (module scope) so typed text / last plotted
# chart survive being re-parented into a fresh VBox on every re-render.
# ---------------------------------------------------------------------------

chart_picker = widgets.Combobox(
    placeholder="Type to search a ticker (e.g. NVIDIA, Gold, EUR/USD, Germany 10Y)…",
    options=list(MASTER_TICKERS.keys()),
    description="Ticker:",
    ensure_option=True,
    layout=widgets.Layout(width="480px"),
    style={"description_width": "50px"},
)
chart_period = widgets.ToggleButtons(
    options=[("1 Month", "1mo"), ("6 Months", "6mo"), ("1 Year", "1y"), ("5 Years", "5y"), ("10 Years", "10y")],
    value="6mo",
)
chart_plot_btn = widgets.Button(description="Plot", button_style="")
chart_status = widgets.HTML(value="")
chart_output = widgets.Output()

def _fred_period_days(period):
    return {"1mo": 45, "6mo": 200, "1y": 400, "5y": 365 * 5 + 30, "10y": 365 * 10 + 60}.get(period, 400)

def _fetch_chart_series(kind, sym, scale, period):
    """Shared data-fetch dispatch for the Chart tab -- used by both the
    single-ticker Plot button and the multi-ticker Compare overlay (added
    2026-09-16) so the two share the exact same per-`kind` fetch logic
    rather than risking them drifting apart over time. Returns
    (series, ylabel, error) -- on success `error` is None; on failure
    `series`/`ylabel` are None and `error` is a ready-to-display message
    (matches the exact wording plot_ticker() always printed before this
    was factored out, so this is a pure refactor, not a behavior change)."""
    try:
        if kind == "price":
            hist = yf.Ticker(sym).history(period=period, auto_adjust=True)
            if hist.empty:
                return None, None, "No data returned for this ticker/period -- try Refresh or a different period."
            series = hist["Close"] / scale
            ylabel = "Price"
        elif kind == "price_inverted":
            # USD/EUR, USD/GBP: Yahoo's own EURUSD=X/GBPUSD=X history is
            # USD per 1 EUR/GBP -- inverted here so the chart matches the
            # Currencies tab's USD-first cards (see CURRENCIES_INVERTED).
            hist = yf.Ticker(sym).history(period=period, auto_adjust=True)
            if hist.empty:
                return None, None, "No data returned for this ticker/period -- try Refresh or a different period."
            series = 1.0 / hist["Close"].replace(0, pd.NA).dropna()
            ylabel = "Price"
        elif kind == "yield_yf":
            hist = yf.Ticker(sym).history(period=period, auto_adjust=True)
            if hist.empty:
                return None, None, "No data returned for this ticker/period -- try Refresh or a different period."
            series = hist["Close"]  # already a direct % -- no /10 scaling
            ylabel = "Yield (%)"
        elif kind == "diesel_crack":
            ho_hist = yf.Ticker("HO=F").history(period=period, auto_adjust=True)
            cl_hist = yf.Ticker("CL=F").history(period=period, auto_adjust=True)
            if ho_hist.empty or cl_hist.empty:
                return None, None, "No data returned for this ticker/period -- try Refresh or a different period."
            ho_c, cl_c = ho_hist["Close"].dropna(), cl_hist["Close"].dropna()
            cl_aligned = cl_c.reindex(ho_c.index.union(cl_c.index)).sort_index().ffill().reindex(ho_c.index)
            series = (ho_c * 42.0 - cl_aligned).dropna()
            ylabel = "Spread ($/bbl)"
        elif kind in ("wti_brent_spread", "gold_silver_ratio", "copper_gold_ratio"):
            leg_syms = {"wti_brent_spread": ("CL=F", "BZ=F"), "gold_silver_ratio": ("GC=F", "SI=F"),
                        "copper_gold_ratio": ("HG=F", "GC=F")}[kind]
            h1 = yf.Ticker(leg_syms[0]).history(period=period, auto_adjust=True)
            h2 = yf.Ticker(leg_syms[1]).history(period=period, auto_adjust=True)
            if h1.empty or h2.empty:
                return None, None, "No data returned for this ticker/period -- try Refresh or a different period."
            c1, c2 = h1["Close"].dropna(), h2["Close"].dropna()
            c2_aligned = c2.reindex(c1.index.union(c2.index)).sort_index().ffill().reindex(c1.index)
            if kind == "wti_brent_spread":
                series = (c1 - c2_aligned).dropna()
                ylabel = "Spread ($/bbl)"
            elif kind == "gold_silver_ratio":
                series = (c1 / c2_aligned).dropna()
                ylabel = "Ratio"
            else:
                series = (c1 / c2_aligned * 1000.0).dropna()
                ylabel = "Ratio x1000"
        else:  # yield_fred -- monthly data, so short periods will show few points
            end = datetime.now()
            start = end - pd.Timedelta(days=_fred_period_days(period))
            df = pdr.DataReader(sym, "fred", start, end)
            series = df[sym].dropna()
            ylabel = "Yield (%)"

        if series.empty:
            return None, None, "No data points in this range."
        return series, ylabel, None
    except Exception as e:
        return None, None, f"Chart failed: {type(e).__name__}: {e}"

def plot_ticker(_btn=None):
    label = chart_picker.value
    if label not in MASTER_TICKERS:
        chart_status.value = f"<span style='color:{RED}'>Pick a valid ticker from the suggestions list.</span>"
        return
    chart_status.value = ""
    kind, sym, scale = MASTER_TICKERS[label]
    period = chart_period.value
    with chart_output:
        clear_output(wait=True)
        try:
            series, ylabel, error = _fetch_chart_series(kind, sym, scale, period)
            if error:
                print(error)
                return

            r, g, b = int(RED[1:3], 16), int(RED[3:5], 16), int(RED[5:7], 16)
            fig = go.Figure(go.Scatter(
                x=series.index, y=series.values,
                mode="lines",
                line=dict(color=RED, width=1.8),
                fill="tozeroy",
                fillcolor=f"rgba({r},{g},{b},0.08)",
                hovertemplate="%{x|%b %d, %Y}<br><b>" + ylabel + ": %{y:,.2f}</b><extra></extra>",
                name=label,
            ))
            fig.update_layout(
                title=dict(text=label, font=dict(size=13, color=INK), x=0, xanchor="left"),
                yaxis_title=ylabel,
                hovermode="x unified",
                plot_bgcolor=PAPER,
                paper_bgcolor=PAPER,
                font=dict(color=MUTED, size=11),
                margin=dict(l=55, r=20, t=40, b=30),
                height=380,
                width=760,
                showlegend=False,
                xaxis=dict(showgrid=False, showline=True, linecolor=LINE, showspikes=True,
                           spikemode="across", spikesnap="cursor", spikecolor=MUTED, spikethickness=1),
                yaxis=dict(showgrid=True, gridcolor=LINE, showline=True, linecolor=LINE),
            )
            # Hover along the curve shows the exact date + price/yield at that point --
            # a static image can't do this, so the Chart tab renders an interactive
            # Plotly figure instead (every other tab's cards stay static PNG via matplotlib).
            display(fig)
        except Exception as e:
            print(f"Chart failed: {type(e).__name__}: {e}")

chart_plot_btn.on_click(plot_ticker)

# ---------------------------------------------------------------- Compare overlay
# Added 2026-09-16 at the user's request: overlay 2-5 tickers on one chart.
# Uses the same MASTER_TICKERS lookup and chart_period toggle as the
# single-ticker Plot above (one shared period control for the whole tab)
# and the same _fetch_chart_series() dispatch, so it can mix any instrument
# type (stock, FX pair, bond yield, ratio/spread, etc.) in one comparison.
#
# Each series is normalized to % change from its OWN first available point
# within the fetched period, not a shared calendar start date -- so a stock,
# a 6,900-point index, and a 4% bond yield can sit on one axis. If one
# picked ticker has less history than the others (e.g. a newer listing), its
# line simply starts later on the shared x-axis rather than being forced to
# align to the others' start date.
compare_picker = widgets.TagsInput(
    value=[],
    allowed_tags=list(MASTER_TICKERS.keys()),
    allow_duplicates=False,
    layout=widgets.Layout(width="700px"),
)
compare_btn = widgets.Button(description="Compare", button_style="")
compare_status = widgets.HTML(value="")
compare_output = widgets.Output()

def plot_compare(_btn=None):
    labels = list(compare_picker.value)
    invalid = [l for l in labels if l not in MASTER_TICKERS]
    if invalid:
        compare_status.value = (f"<span style='color:{RED}'>Not a recognized ticker: "
                                 f"{', '.join(invalid)} -- remove it and pick from the suggestions.</span>")
        return
    if len(labels) < 2:
        compare_status.value = f"<span style='color:{MUTED}'>Add at least 2 tickers to compare (up to 5).</span>"
        return
    truncated = len(labels) > 5
    if truncated:
        labels = labels[:5]
    compare_status.value = (f"<span style='color:{AMBER}'>Only the first 5 tags are plotted -- "
                             f"remove one to compare a different set.</span>" if truncated else "")
    period = chart_period.value
    with compare_output:
        clear_output(wait=True)
        try:
            fig = go.Figure()
            any_plotted = False
            for i, label in enumerate(labels):
                kind, sym, scale = MASTER_TICKERS[label]
                series, ylabel, error = _fetch_chart_series(kind, sym, scale, period)
                if error:
                    print(f"{label}: {error}")
                    continue
                base = float(series.iloc[0])
                if base == 0:
                    print(f"{label}: can't normalize to % change -- starts at 0.")
                    continue
                norm = (series / base - 1.0) * 100.0
                color = COMPARE_COLORS[i % len(COMPARE_COLORS)]
                fig.add_trace(go.Scatter(
                    x=norm.index, y=norm.values, mode="lines",
                    line=dict(color=color, width=1.8),
                    hovertemplate="%{x|%b %d, %Y}<br><b>" + label + ": %{y:+.1f}%</b><extra></extra>",
                    name=label,
                ))
                any_plotted = True
            if not any_plotted:
                print("Nothing plottable -- see the message(s) above.")
                return
            fig.update_layout(
                title=dict(text="Compare -- % change from start of period", font=dict(size=13, color=INK),
                           x=0, xanchor="left"),
                yaxis_title="% change from period start",
                hovermode="x unified",
                plot_bgcolor=PAPER,
                paper_bgcolor=PAPER,
                font=dict(color=MUTED, size=11),
                margin=dict(l=55, r=20, t=50, b=30),
                height=420,
                width=760,
                showlegend=True,
                legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0, font=dict(size=10)),
                xaxis=dict(showgrid=False, showline=True, linecolor=LINE, showspikes=True,
                           spikemode="across", spikesnap="cursor", spikecolor=MUTED, spikethickness=1),
                yaxis=dict(showgrid=True, gridcolor=LINE, showline=True, linecolor=LINE,
                           zeroline=True, zerolinecolor=LINE, zerolinewidth=1),
            )
            display(fig)
        except Exception as e:
            print(f"Compare failed: {type(e).__name__}: {e}")

compare_btn.on_click(plot_compare)

def render_chart_tab():
    return widgets.VBox([
        widgets.HTML(f"<p style='color:{MUTED};font-size:12px;max-width:700px'>Search any tracked stock, "
                     "sector ETF, metal, energy or agriculture commodity, FX pair, index, or bond yield, "
                     "then click Plot to see its real price/yield history.</p>"),
        widgets.HBox([chart_picker, chart_plot_btn]),
        chart_period,
        chart_status,
        chart_output,
        widgets.HTML(f"<hr style='border:none;border-top:1px solid {LINE};margin:20px 0 14px 0'>"),
        widgets.HTML(f"<p style='color:{MUTED};font-size:12px;max-width:700px'><b>Compare</b> -- pick 2-5 "
                     "tickers (type to search, same list as above) to overlay on one chart, each normalized "
                     "to % change from the start of the period selected above, so any mix of instrument "
                     "types can share one axis.</p>"),
        widgets.HBox([compare_picker, compare_btn]),
        compare_status,
        compare_output,
    ])

RENDERERS = {
    "Metals": render_metals, "Energy": render_energy, "Agriculture": render_agri,
    "Currencies": render_currencies, "Crypto": render_crypto, "Indices": render_indices,
    "Signals": render_signals,
    "Top Performers": render_top_performers, "Industries": render_industries,
    "Bonds": render_bonds, "Cash & Liquidity": render_cash_liquidity, "Chart": render_chart_tab,
}

def refresh_view(name):
    with outputs[name]:
        clear_output(wait=True)
        display(RENDERERS[name]())

def refresh_all_views(*args):
    for name in TAB_NAMES:
        refresh_view(name)

def on_horizon_change(change):
    if change["name"] == "value":
        refresh_all_views()

def on_refresh_click(b):
    status_label.value = _status_html("Refreshing… this can take 20-60s for ~80 tickers/series", spinning=True)
    def progress(i, n):
        status_label.value = _status_html(f"Refreshing… {i}/{n}", spinning=True)
    refresh_all(progress_cb=progress)
    status_label.value = _status_html(f"Last refreshed {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    refresh_all_views()

horizon.observe(on_horizon_change, names="value")
refresh_btn.on_click(on_refresh_click)

app = widgets.VBox([
    widgets.HTML(f"<h2 style='margin:0 0 10px 0;color:{INK}'>Markets Dashboard</h2>"),
    widgets.HBox([horizon, refresh_btn]),
    status_label,
    tabs,
], layout=widgets.Layout(padding="18px 20px", border=f"1px solid {LINE}"))
app.add_class("md-dashboard-root")

STYLE = widgets.HTML(f"""
<style>
.md-dashboard-root {{
  background: {PAPER} !important;
  border-radius: 8px;
}}
.md-dashboard-root .widget-tab, .md-dashboard-root .p-TabPanel {{
  background: transparent !important;
}}
.md-dashboard-root .p-TabBar-tab.p-mod-current {{
  background: {INK} !important;
}}
.md-dashboard-root .p-TabBar-tab.p-mod-current .p-TabBar-tabLabel {{
  color: {PAPER} !important;
}}
.md-spinner {{
  display: inline-block;
  width: 11px;
  height: 11px;
  border: 2px solid {LINE};
  border-top-color: {INK};
  border-radius: 50%;
  vertical-align: -1px;
  animation: md-spin 0.7s linear infinite;
}}
@keyframes md-spin {{
  to {{ transform: rotate(360deg); }}
}}
</style>
""")
display(STYLE)
display(app)


HTML(value='\n<style>\n.md-dashboard-root {\n  background: #F5F4EF !important;\n  border-radius: 8px;\n}\n.md-…

In [24]:
# Auto-load live data when the notebook (or Voila app) starts.
# Comment this out if you'd rather load on-demand via the Refresh button only.
on_refresh_click(None)
